# CS-781 Phase 7 (Prior Case Retrieval) — Architecture 2: encoder-decoder v2 (90M, 6+6 layers, d640), new BPE

**Task.** For every query case, rank the candidate pool of the same split and return the **7 most relevant prior cases**
(Micro-F1@7; Micro-P@7 and Micro-R@7 secondary). Submission = `prediction.json` = `{"query_id": [7 unique candidate ids, best first]}`
for every test query, at the root of a zip.

**Data facts** (checked in cell 4): train 827 queries / 4,320 candidates, dev 118 / 1,023, test 237 / 1,727. **Every query is also in
its own pool**, so it is always removed from its own ranking. 4–5 relevant cases per query (median; mean 6–8), so a perfect ranking
scores only ~0.62–0.64. Documents are long (median ~4.5k words) and citations / names / organisations are masked (`<CITATION>`, `<NAME>`, `<ORG>`).

**System.**
* **Lexical retrieval** (TF-IDF / BM25 grid tuned on dev). It mines **hard negatives** for training and is **fused** with the model score.
* **Bi-encoder from my Phase-1 model**: exact Phase-1 tokenisation of the whole document → 350-token chunks (up to 20 kept:
  first + last + the chunks with the most `<CITATION>` masks + evenly spread ones) → encoder-decoder (the encoder reads the chunk bidirectionally, the decoder reads [<s>] + chunk with causal self-attention + cross-attention exactly like Phase-1 pretraining; decoder states are pooled) + an adaptation method →
  mean-pooled chunk vectors → centred + projected to 256-d → mean over chunks → unit vector. Score = cosine
  (chunk-level **MaxSim** is also tried at evaluation time).
* **Training**: InfoNCE. Each query is paired with one of its relevant cases; every other document in the batch is a negative (other queries,
  their positives, 2 BM25 hard negatives per query). The query's other relevant cases are masked. **GradCache** makes 16-query / 64-document
  batches fit a T4.
* **Ranking**: z-normalised model score fused with the lexical score, `w·model + (1−w)·lexical` with w ≥ 0.1, over the whole pool or the
  lexical top-100. Tuned on dev, with an honest 2-fold cross-fitted estimate. An optional **temporal constraint** is decided on dev in cell 4:
  a candidate that mentions a year after the query's judgment date cannot be a *prior* case.

**Experiments** (one Codabench zip each, plus a `_model_only` zip; Phase-7 technique list):

| job | technique | note |
|---|---|---|
| `lorma` | LoRMA on the attention projections ["q", "k", "v"] (s = 1) | **compulsory**; KD teacher candidate |
| `full` | full fine-tuning | reference + KD teacher candidate |
| `kd` | knowledge distillation: teacher similarity distributions → 6-layer student | **compulsory** |
| `kd_small` | knowledge distillation → smaller student | student-size comparison |
| `partial` | partial / layer-wise: top 4 blocks trained | |
| `qlora` | QLoRA: frozen weights in 4-bit NF4 + LoRA | |
| `ia3` | IA³: rescale keys, values, FFN activation | |
| `prompt` | prompt tuning (20 soft tokens per chunk) | |
| `prefix` | prefix tuning (16 prefix vectors at every layer) | |
| `frozen` | frozen Phase-1 encoder + trained head (probe) | also runs the ablations: zero-shot transfer, pooling, similarity, objectives, hard negatives, chunks |


In [1]:
# ============================================================ 1. SETUP
import os, sys, json, time, shutil, subprocess, threading, pickle
from pathlib import Path
T_START = time.time()

QUICK_TEST = False        # True: tiny end-to-end check (~20-30 min).  False: the real run (commit it).
GROUP = "ALL"            # "ALL": all 10 jobs here (~10.5 h) | "A" + "B": the same jobs on two accounts at the same time (longer jobs)
SESSION_H = 12.0         # Kaggle session limit; jobs are shortened automatically so the notebook always finishes
ARCH, NORM = "encdec", "nfkc"
PHASE1_HINT = "archit-2-new-json"   # substring of the attached Phase-1 notebook's input path

WORK = Path("/kaggle/working/p7"); WORK.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
OUT  = WORK / ("runs_quick" if QUICK_TEST else "runs")

def find(name, must_contain="", anywhere=False):
    hits = sorted(p for p in Path("/kaggle/input").rglob(name) if (anywhere or PHASE1_HINT in str(p).lower()) and must_contain in str(p))
    return str(hits[0]) if hits else None

PHASE1_CKPT = find("best.pt", "phase1_encdec_v2") or find("best.pt", "ed2_run") or find("best.pt")
MODEL_PY    = find("edmodel.py", "phase1_encdec_v2") or find("edmodel.py")
TOK_PY      = find("a1_tokenization.py", "phase1_encdec_v2") or find("a1_tokenization.py") or find("a1_tokenization.py", anywhere=True)
TOK_PATH    = str(Path(TOK_PY).parent) if TOK_PY and (Path(TOK_PY).parent / "vocab.json").exists() else None
if TOK_PY and TOK_PATH is None:
    for v in sorted(Path("/kaggle/input").rglob("vocab.json")):
        if (v.parent / "merges.txt").exists(): TOK_PATH = str(v.parent); break
print("checkpoint :", PHASE1_CKPT); print("model code :", MODEL_PY); print("tokenizer  :", TOK_PY, "|", TOK_PATH)
assert PHASE1_CKPT and MODEL_PY, f"attach the Phase-1 notebook whose input path contains '{PHASE1_HINT}' (Add Input -> Your Work -> Notebooks), or fix PHASE1_HINT"
assert TOK_PY and TOK_PATH, "tokenizer files not found - attach the tokenizer dataset"

# Phase-7 data: a local copy if one is attached, otherwise download it from Hugging Face with my HF secret
# (to the HF cache outside /kaggle/working, so the private data never ends up in this notebook's output).
hits = sorted(Path("/kaggle/input").rglob("Phase_7_PCR/Train/train.json"))
if hits:
    DATA = hits[0].parent.parent; print("using local copy:", DATA)
else:
    HF_TOKEN = None
    try:
        from kaggle_secrets import UserSecretsClient
        for k in ["HF", "HF_TOKEN"]:
            try: HF_TOKEN = UserSecretsClient().get_secret(k); break
            except Exception: pass
    except ImportError:
        HF_TOKEN = os.environ.get("HF_TOKEN")
    assert HF_TOKEN, "add your Hugging Face token as a Kaggle secret named HF (Add-ons -> Secrets) and tick it for this notebook"
    from huggingface_hub import snapshot_download
    DATA = Path(snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=HF_TOKEN,
                                  allow_patterns=["Phase_7_PCR/*"])) / "Phase_7_PCR"
print("data:", DATA, sorted(str(p.relative_to(DATA)) for p in DATA.rglob("*.json")))


checkpoint : /kaggle/input/notebooks/amex786/archit-2-new-json/phase1_encdec_v2/best.pt
model code : /kaggle/input/notebooks/amex786/archit-2-new-json/phase1_encdec_v2/edmodel.py
tokenizer  : /kaggle/input/notebooks/amex786/archit-2-new-json/phase1_encdec_v2/a1_tokenization.py | /kaggle/input/notebooks/amex786/archit-2-new-json/phase1_encdec_v2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

data: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR ['Dev/dev.json', 'Dev/dev_candidates.json', 'Test/test.json', 'Test/test_candidates.json', 'Train/train.json', 'Train/train_candidates.json']


In [2]:
%%writefile /kaggle/working/p7/p2_ner.py
"""CS-781 Phase 2 (L-NER) on top of my Phase-1 models: Transformer-XL (arch=txl), decoder-only LegalLM (arch=dec),
the encoder of the encoder-decoder (arch=encdec), or RetNet (arch=ret).

One file, used two ways:
  * imported by the notebook  -> data prep, alignment checks, model-equivalence test
  * run as a script            -> `python p2_ner.py job.json` trains/evaluates/predicts ONE experiment

Pipeline
  text --(ASCII-fold + whitespace-collapse, exactly like Phase-1 pretraining, but keeping a char map)-->
  BPE tokens with ORIGINAL character offsets --> BIO tags (25) --> 512-token sliding windows -->
  Transformer-XL encoder (+ PEFT) + token head --> overlap-weighted log-prob merge per document -->
  BIO-constrained Viterbi --> character spans (whitespace-stripped) --> prediction.json

Adaptation methods (METHOD):
  full     full fine-tuning
  partial  top-N blocks + head (layer-wise / partial fine-tuning)
  lora     LoRA:  y = W x + s*B(A x)
  lorma    LoRMA_+ (Bihany, Patel & Modi 2025, multiplicative):  y = (I + s*B A) W x   (B = 0 -> identity at init)
  adapter  Houlsby-style bottleneck adapters after attention and FFN sub-layers
  bitfit   biases (+ norms; TXL u/v biases). Bias-free LegalLM: zero-init biases added to every projection
  kd       knowledge distillation: a fine-tuned 14-layer teacher -> a shallower student built from Phase-1 layers
"""
import json
import math
import os
import pickle
import re
import sys
import time
import unicodedata
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

# ============================================================== text normalisation with char map
_MAP = str.maketrans({
    "‘": "'", "’": "'", "‚": "'", "‛": "'", "“": '"', "”": '"',
    "„": '"', "«": '"', "»": '"', "–": "-", "—": "-", "−": "-",
    "‐": "-", "‑": "-", "‒": "-", "―": "-", " ": " ", "•": "*",
    "·": "*", "…": "...", "₹": "Rs. ",
})
_CTRL = re.compile(r"[\x00-\x08\x0e-\x1f\x7f]")
# Phase-1 pre-tokenizer WITH the underscore fix that the Phase-1 notebook applied
PRE_RE = re.compile(r" ?[^\W\d_]+| ?\d+| ?(?:[^\s\w]|_)+|\s+")


def normalize_with_map(text):
    """ASCII mode (Transformer-XL runs): same output as the TXL Phase-1 `normalize_text` (ASCII fold,
    straight quotes, single spaces, no newlines), plus omap[i] = ORIGINAL index of normalised char i."""
    norm, st, _ = normalize_ascii_map(text)
    return norm, st


def normalize_ascii_map(text):
    out, ost, oen = [], [], []
    pend = -1                                   # orig index of the first whitespace in a pending run
    for i, ch in enumerate(text):
        s = ch
        if not ch.isascii():
            s = ch.translate(_MAP)
            if not s.isascii():
                s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
        if s and _CTRL.search(s):
            s = _CTRL.sub("", s)
        for c in s:
            if c.isspace():
                if pend < 0:
                    pend = i
            else:
                if pend >= 0 and out:
                    out.append(" "); ost.append(pend); oen.append(pend + 1)
                pend = -1
                out.append(c); ost.append(i); oen.append(i + 1)
    return "".join(out), ost, oen


def normalize_nfkc_map(text, delete_table):
    """NFKC mode (decoder-only run): same output as a1_tokenization.normalize_text
    (delete Cc/Cf/Cs -> NFKC -> collapse whitespace -> strip). NFKC is applied per *cluster*
    (a char + its combining marks) so compositions match the whole-string NFKC.
    ost/oen[i] = ORIGINAL [start, end) of the cluster that produced normalised char i."""
    out, ost, oen = [], [], []
    pend = -1
    n, i = len(text), 0
    while i < n:
        j = i + 1
        while j < n and unicodedata.combining(text[j]):
            j += 1
        s = unicodedata.normalize("NFKC", text[i:j].translate(delete_table))
        for c in s:
            if c.isspace():
                if pend < 0:
                    pend = i
            else:
                if pend >= 0 and out:
                    out.append(" "); ost.append(pend); oen.append(pend + 1)
                pend = -1
                out.append(c); ost.append(i); oen.append(j)
        i = j
    return "".join(out), ost, oen


def align_map(orig, norm):
    """Character map norm -> orig from a diff (used only when the fast cluster map disagrees)."""
    import difflib
    st, en = [0] * len(norm), [0] * len(norm)
    sm = difflib.SequenceMatcher(None, orig, norm, autojunk=False)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        for j in range(j1, j2):
            if tag == "equal":
                st[j], en[j] = i1 + (j - j1), i1 + (j - j1) + 1
            else:                                  # replaced / inserted: map to the whole changed region
                st[j], en[j] = i1, max(i2, i1 + 1) if i2 > i1 else min(i1 + 1, len(orig))
    return st, en


# RetNet Phase-1 preprocessing: ret_data.clean_text = remove "\r", collapse 3+ newlines to 2, strip (unicode kept)
_NL3 = re.compile(r"\n{3,}")
# the ORIGINAL a1_tokenizer pre-tokenizer (no underscore fix: "_" is never matched, i.e. dropped) - used by the RetNet run
PRE_RE_ORIG = re.compile(r" ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+")


def clean_text_ret(t):
    return _NL3.sub("\n\n", t.replace("\r", "")).strip()


def normalize_clean_map(text):
    idx = [i for i, c in enumerate(text) if c != "\r"]
    keep, run = [], 0
    for i in idx:                                              # 3+ consecutive newlines -> keep the first two
        if text[i] == "\n":
            run += 1
            if run > 2:
                continue
        else:
            run = 0
        keep.append(i)
    a, b = 0, len(keep)
    while a < b and text[keep[a]].isspace():
        a += 1
    while b > a and text[keep[b - 1]].isspace():
        b -= 1
    keep = keep[a:b]
    return "".join(text[i] for i in keep), keep, [i + 1 for i in keep]


# Archit-2-v2 preprocessing: " ".join(text.split())  (whitespace runs -> one space, ends stripped, unicode kept)
def normalize_ws_map(text):
    out, ost, oen = [], [], []
    pend = -1
    for i, ch in enumerate(text):
        if ch.isspace():
            if pend < 0:
                pend = i
        else:
            if pend >= 0 and out:
                out.append(" "); ost.append(pend); oen.append(pend + 1)
            pend = -1
            out.append(ch); ost.append(i); oen.append(i + 1)
    return "".join(out), ost, oen


class HFBPE:
    """Archit-2-v2 tokenizer: HuggingFace `tokenizers` ByteLevelBPETokenizer(vocab.json, merges.txt)
    (<pad>0 <s>1 </s>2 <unk>3), text = " ".join(text.split()). Offsets come from the tokenizer itself."""

    def __init__(self, tok_path, norm="ws"):
        from tokenizers import ByteLevelBPETokenizer
        p = Path(tok_path)
        p = p.parent if p.is_file() else p
        self.tok = ByteLevelBPETokenizer(str(p / "vocab.json"), str(p / "merges.txt"))
        self.kind, self.norm = "hf", norm
        self.vocab_size = self.tok.get_vocab_size()
        assert [self.tok.token_to_id(t) for t in ("<pad>", "<s>", "</s>", "<unk>")] == [0, 1, 2, 3]

    def normalize(self, text):
        n, st, en = normalize_ws_map(text)
        assert n == " ".join(text.split())
        return n, st, en

    def reference_encode(self, raw_text):
        return self.tok.encode(" ".join(raw_text.split())).ids


# ============================================================== tokenizer with offsets
class BPE:
    """Uniform wrapper around my two from-scratch Phase-1 tokenizers.
      kind='old' : a1_tokenizer.py  + bpe_tokenizer.json   (<pad>0 <unk>1 <s>2 </s>3; underscore-fixed regex)
      kind='new' : a1_tokenization.py + vocab.json/merges.txt (<pad>0 <s>1 </s>2 <unk>3; GPT-2 style regex)
    norm='ascii' : TXL pretraining preprocessing (ASCII fold)   |   norm='nfkc' : a1_tokenization.normalize_text
    norm='clean' : RetNet preprocessing (\r removal, newline collapse; ORIGINAL old regex, underscores dropped)
    Exposes: pre_re, encode_piece(piece) -> ids, nbytes[id], vocab_size, normalize(text), reference_encode(text)."""

    def __init__(self, tokenizer_py, tok_path, norm="ascii"):
        import importlib.util
        name = Path(tokenizer_py).stem
        spec = importlib.util.spec_from_file_location(name + "_p2", tokenizer_py)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        self.mod, self.norm = mod, norm
        if hasattr(mod, "BPETokenizer"):                              # old tokenizer
            self.kind = "old"
            p = Path(tok_path)
            self.tok = mod.BPETokenizer.load(str(p if p.is_file() else p / "bpe_tokenizer.json"))
            self.pre_re = PRE_RE_ORIG if norm == "clean" else PRE_RE      # RetNet run used the unpatched regex
            self.encode_piece = self.tok._encode_pretoken
            self.nbytes = [len(b) for b in self.tok.id_to_bytes]
            self.vocab_size = self.tok.get_vocab_size()
        else:                                                         # new tokenizer
            self.kind = "new"
            p = Path(tok_path)
            self.tok = mod.Tokenizer.load(str(p.parent if p.is_file() else p))
            self.pre_re = mod.PRETOKEN_RE
            self.encode_piece = self.tok._encode_word
            k = mod.N_SPECIAL
            self.nbytes = [0] * k + [len(t) for t in self.tok.id_to_token[k:]]   # 1 char = 1 byte (GPT-2 byte map)
            self.vocab_size = self.tok.vocab_size
        assert (norm == "ascii") or (norm == "nfkc" and self.kind == "new") or (norm == "clean" and self.kind == "old")

    def normalize(self, text):
        if self.norm == "nfkc":
            n, st, en = normalize_nfkc_map(text, self.mod._DELETE_TABLE)
            ref = self.mod.normalize_text(text)
            if n != ref:                       # rare: NFKC interaction across clusters -> exact diff alignment
                self.n_fallback = getattr(self, "n_fallback", 0) + 1
                st, en = align_map(text, ref)
                n = ref
            return n, st, en
        if self.norm == "clean":
            n, st, en = normalize_clean_map(text)
            if n != clean_text_ret(text):
                self.n_fallback = getattr(self, "n_fallback", 0) + 1
                n = clean_text_ret(text)
                st, en = align_map(text, n)
            return n, st, en
        return normalize_ascii_map(text)

    def encode_full(self, norm_text):
        out = []
        for piece in self.pre_re.findall(norm_text):
            out.extend(self.encode_piece(piece))
        return out

    def reference_encode(self, raw_text):
        """Exactly what Phase-1 pretraining fed the model for this raw text (used only for checks)."""
        if self.norm == "nfkc":
            return self.tok.encode(raw_text)                         # normalize_text + encode
        if self.norm == "clean":                                     # ret_data.clean_text + original a1_tokenizer.encode
            t = clean_text_ret(raw_text)
            return [i for piece in PRE_RE_ORIG.findall(t) for i in self.tok._encode_pretoken(piece)]
        return self.encode_full(self.normalize(raw_text)[0])


def load_bpe(tokenizer_py, tok_path, norm="ascii"):
    if norm == "ws":
        return HFBPE(tok_path, norm)
    return BPE(tokenizer_py, tok_path, norm)


def encode_doc(tok, text):
    """-> dict(ids int32[n], cs int32[n], ce int32[n]).  For token t, the ORIGINAL-text span is [cs, ce),
    where cs skips the single leading space a BPE token may carry. Handles tokens that split a
    multi-byte UTF-8 character (byte-level BPE) by mapping byte offsets to characters."""
    norm, ost, oen = tok.normalize(text)
    ids, cs, ce = [], [], []
    if getattr(tok, "kind", "") == "hf":                         # offsets straight from the HF tokenizer
        if not norm:
            return dict(ids=np.zeros(0, np.int32), cs=np.zeros(0, np.int32), ce=np.zeros(0, np.int32))
        enc = tok.tok.encode(norm)
        for tid, (a, z) in zip(enc.ids, enc.offsets):
            z = max(z, a + 1) - 1                                # last normalised char touched by this token
            a = min(a, len(norm) - 1); z = min(max(z, a), len(norm) - 1)
            a2 = a + 1 if (norm[a] == " " and z > a) else a
            ids.append(tid); cs.append(ost[a2]); ce.append(oen[z])
        return dict(ids=np.asarray(ids, np.int32), cs=np.asarray(cs, np.int32), ce=np.asarray(ce, np.int32))
    for m in tok.pre_re.finditer(norm):
        piece, p0 = m.group(), m.start()
        if piece.isascii():
            b2c = None
            nb = len(piece)
        else:
            b2c = [ci for ci, ch in enumerate(piece) for _ in range(len(ch.encode("utf-8")))]
            nb = len(b2c)
        bp = 0
        for tid in tok.encode_piece(piece):
            L = tok.nbytes[tid]
            a = p0 + (bp if b2c is None else b2c[bp])
            z = p0 + ((bp + L - 1) if b2c is None else b2c[bp + L - 1])     # last char touched by this token
            bp += L
            a2 = a + 1 if (norm[a] == " " and z > a) else a
            ids.append(tid)
            cs.append(ost[a2])
            ce.append(oen[z])
        assert bp == nb, "token byte lengths do not add up"
    return dict(ids=np.asarray(ids, np.int32), cs=np.asarray(cs, np.int32), ce=np.asarray(ce, np.int32))


# ============================================================== labels / BIO
def make_tags(labels):
    tags = ["O"] + [f"B-{l}" for l in labels] + [f"I-{l}" for l in labels]
    return tags, {t: i for i, t in enumerate(tags)}


def strip_span(text, s, e):
    while s < e and text[s].isspace():
        s += 1
    while e > s and text[e - 1].isspace():
        e -= 1
    return s, e


def bio_from_gold(text, enc, ents, tag2id):
    """Token BIO ids from character-level gold entities (entities are whitespace-stripped first)."""
    owner = np.full(len(text) + 1, -1, np.int32)
    lab_of = []
    for k, e in enumerate(ents):
        s, en = strip_span(text, int(e[0]), int(e[1]))
        owner[s:en] = k
        lab_of.append(e[2])
    cs = enc["cs"]
    y = np.zeros(len(cs), np.int64)
    prev = -1
    for t in range(len(cs)):
        k = owner[cs[t]] if cs[t] < len(text) else -1
        if k >= 0 and text[cs[t]].isspace() and k != prev:     # a whitespace-only token continues an entity, never starts one
            k = -1
        if k >= 0:
            y[t] = tag2id[("I-" if k == prev else "B-") + lab_of[k]]
        prev = k
    return y


def spans_from_tags(text, enc, tag_ids, tags):
    out, cur = [], None                     # cur = [first_tok, last_tok, label]
    for t, ti in enumerate(tag_ids):
        tg = tags[ti]
        if tg == "O":
            if cur: out.append(cur); cur = None
            continue
        bi, lab = tg.split("-", 1)
        if bi == "B" or cur is None or cur[2] != lab:
            if cur: out.append(cur)
            cur = [t, t, lab]
        else:
            cur[1] = t
    if cur: out.append(cur)
    res = []
    for a, b, lab in out:
        s, e = strip_span(text, int(enc["cs"][a]), int(enc["ce"][b]))
        if e > s:
            res.append([s, e, lab])
    return res


def bio_transitions(tags):
    """allowed[i, j] for tag i -> tag j; start_ok[j]."""
    T = len(tags)
    allowed = np.ones((T, T), bool)
    start_ok = np.ones(T, bool)
    for j, tj in enumerate(tags):
        if tj.startswith("I-"):
            lab = tj[2:]
            start_ok[j] = False
            for i, ti in enumerate(tags):
                allowed[i, j] = ti in (f"B-{lab}", f"I-{lab}")
    return allowed, start_ok


def viterbi(logp, allowed, start_ok):
    """logp [n, T] log-probs -> best BIO-valid tag sequence."""
    n, T = logp.shape
    if n == 0:
        return np.zeros(0, np.int64)
    trans = np.where(allowed, 0.0, -1e9)
    score = np.where(start_ok, logp[0], -1e9)
    back = np.zeros((n, T), np.int32)
    for t in range(1, n):
        cand = score[:, None] + trans            # [from, to]
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    best = np.zeros(n, np.int64)
    best[-1] = int(score.argmax())
    for t in range(n - 1, 0, -1):
        best[t - 1] = back[t, best[t]]
    return best


# ============================================================== metric (strict span + label)
def strict_scores(gold, pred, labels):
    tp, fp, fn = Counter(), Counter(), Counter()
    for did in gold:
        g = {(int(s), int(e), l) for s, e, l, *_ in gold[did]}
        p = {(int(s), int(e), l) for s, e, l, *_ in pred.get(did, [])}
        for x in g & p: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    per = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        per[l] = dict(p=P, r=R, f1=F, tp=tp[l], fp=fp[l], fn=fn[l])
    mac = lambda k: sum(per[l][k] for l in labels) / len(labels)
    return dict(macro_f1=mac("f1"), macro_p=mac("p"), macro_r=mac("r"), per_label=per)


def error_breakdown(gold, pred):
    """Classify every false positive: label error (same span, other label), boundary error
    (overlaps a gold span of the same label), or spurious; plus the label-confusion counts."""
    kinds, conf = Counter(), Counter()
    for did in gold:
        g = [(int(s), int(e), l) for s, e, l, *_ in gold[did]]
        gs = set(g)
        for s, e, l in [(int(s), int(e), l) for s, e, l, *_ in pred.get(did, [])]:
            if (s, e, l) in gs:
                continue
            same = [gl for (a, b, gl) in g if a == s and b == e]
            if same:
                kinds["label_error"] += 1; conf[f"{same[0]}->{l}"] += 1
            elif any(a < e and s < b and gl == l for a, b, gl in g):
                kinds["boundary_error"] += 1
            elif any(a < e and s < b for a, b, gl in g):
                kinds["overlap_other_label"] += 1
            else:
                kinds["spurious"] += 1
    return dict(fp_kinds=dict(kinds), top_confusions=conf.most_common(15))


# ============================================================== data prep (called from the notebook)
def prepare(data_dir, tok, labels, cache_path):
    data_dir = Path(data_dir)
    tags, tag2id = make_tags(labels)
    out = {}
    for fold in ["fold1", "fold2", "fold3"]:
        sub = "Test" if fold == "fold3" else "Train"
        texts = json.load(open(data_dir / sub / f"{fold}-text.json", encoding="utf-8"))
        gold = json.load(open(data_dir / sub / f"{fold}-gold.json", encoding="utf-8")) if fold != "fold3" else None
        docs = {}
        for did, text in texts.items():
            enc = encode_doc(tok, text)
            d = dict(text=text, **enc)
            if gold is not None:
                d["gold"] = gold.get(did, [])
                d["y"] = bio_from_gold(text, enc, d["gold"], tag2id)
            docs[did] = d
        out[fold] = docs
    pickle.dump(dict(labels=labels, tags=tags, folds=out), open(cache_path, "wb"))
    return out


def alignment_report(folds, labels):
    """Upper bound of strict macro-F1 if the model predicted the gold BIO tags perfectly."""
    tags, _ = make_tags(labels)
    rep = {}
    for fold in ["fold1", "fold2"]:
        gold = {d: v["gold"] for d, v in folds[fold].items()}
        pred = {d: spans_from_tags(v["text"], v, v["y"], tags) for d, v in folds[fold].items()}
        sc = strict_scores(gold, pred, labels)
        miss = []
        for d, v in folds[fold].items():
            p = {tuple(x) for x in pred[d]}
            for s, e, l, *_ in v["gold"]:
                if (s, e, l) not in p:
                    miss.append((d, s, e, l, v["text"][max(0, s - 15):e + 15]))
        rep[fold] = dict(upper_bound_macro_f1=sc["macro_f1"], n_gold=sum(len(v["gold"]) for v in folds[fold].values()),
                         n_unrecoverable=len(miss), examples=miss[:12])
    ntok = {f: int(sum(len(v["ids"]) for v in folds[f].values())) for f in folds}
    rep["tokens_per_fold"] = ntok
    return rep


# ============================================================== torch parts
import torch                                   # noqa: E402
import torch.nn as nn                          # noqa: E402
import torch.nn.functional as F                # noqa: E402


def _import(path, name):
    import importlib.util
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


def load_phase1(txl_model_py, ckpt_path):
    """Transformer-XL Phase-1 checkpoint: {'model': state_dict, 'config': dict}."""
    mod = _import(txl_model_py, "txl_model_p2")
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = mod.TXLConfig(**ck["config"])
    m = mod.TransformerXL(cfg)
    m.load_state_dict(ck["model"])
    return m, mod


def load_decoder(legal_lm_py, weights_path):
    """Decoder-only LegalLM Phase-1 weights (run1/final/model.pt + model_config.json)."""
    mod = _import(legal_lm_py, "legal_lm_p2")
    m = mod.load_legal_lm(weights_path, device="cpu", dtype=torch.float32)
    return m.train(), mod


def load_encdec(edmodel_py, ckpt_path):
    """Encoder-decoder Phase-1 checkpoint (phase1_encdec_v2/best.pt: {'model', 'cfg', ...})."""
    mod = _import(edmodel_py, "edmodel_p2")
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m = mod.EncDecLM(mod.EDConfig(**ck["cfg"]))
    m.load_state_dict(ck["model"])
    return m, mod


def load_retnet(ret_model_py, ckpt_path):
    """RetNet Phase-1 checkpoint (retnet/ckpt/best.pt: {'model', 'config': {'model': ...}, 'vocab_size', ...})."""
    mod = _import(ret_model_py, "ret_model_p2")
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m = mod.RetNet(ck["vocab_size"], **ck["config"]["model"])
    sd = ck["model"]
    if any(k.startswith("module.") for k in sd):
        sd = {k[7:]: v for k, v in sd.items()}
    m.load_state_dict(sd)
    return m, mod


# ---- Archit-2-v2 encoder-decoder (same module/parameter names as the Phase-1 notebook, generation code removed)
class _A2Emb(nn.Module):
    def __init__(self, vocab_size, d_model, max_length, pad_id, dropout):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, ids, offset=0):
        T = ids.size(1)
        pos = torch.arange(offset, offset + T, device=ids.device).unsqueeze(0)
        return self.dropout(self.token_embedding(ids) * math.sqrt(self.d_model) + self.position_embedding(pos))


class _A2MHA(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        self.d_model, self.h, self.dh, self.p = d_model, num_heads, d_model // num_heads, dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def _split(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.h, self.dh).transpose(1, 2)

    def forward(self, query, key_value=None, key_padding_mask=None, causal=False):
        kv = query if key_value is None else key_value
        q, k, v = self._split(self.q_proj(query)), self._split(self.k_proj(kv)), self._split(self.v_proj(kv))
        mask = None
        if key_padding_mask is not None:
            mask = key_padding_mask[:, None, None, :].bool()
            if causal:
                cm = torch.tril(torch.ones(query.size(1), kv.size(1), dtype=torch.bool, device=query.device))[None, None]
                mask = mask & cm
                causal = False
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=self.p if self.training else 0.0, is_causal=causal)
        B, _, T, _ = o.shape
        return self.out_proj(o.transpose(1, 2).contiguous().view(B, T, self.d_model))


class _A2FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(self.dropout(self.activation(self.fc1(x))))


class _A2EncLayer(nn.Module):
    def __init__(self, d, h, d_ff, p):
        super().__init__()
        self.norm1, self.self_attn, self.dropout1 = nn.LayerNorm(d), _A2MHA(d, h, p), nn.Dropout(p)
        self.norm2, self.ffn, self.dropout2 = nn.LayerNorm(d), _A2FFN(d, d_ff, p), nn.Dropout(p)

    def forward(self, x, src_mask):
        x = x + self.dropout1(self.self_attn(self.norm1(x), key_padding_mask=src_mask))
        return x + self.dropout2(self.ffn(self.norm2(x)))


class _A2DecLayer(nn.Module):
    def __init__(self, d, h, d_ff, p):
        super().__init__()
        self.norm1, self.self_attn, self.dropout1 = nn.LayerNorm(d), _A2MHA(d, h, p), nn.Dropout(p)
        self.norm2, self.cross_attn, self.dropout2 = nn.LayerNorm(d), _A2MHA(d, h, p), nn.Dropout(p)
        self.norm3, self.ffn, self.dropout3 = nn.LayerNorm(d), _A2FFN(d, d_ff, p), nn.Dropout(p)

    def forward(self, x, enc_out, src_mask, self_mask=None):
        """self_mask=None -> causal (= Phase-1). Otherwise a key-padding mask -> bidirectional decoder."""
        if self_mask is None:
            x = x + self.dropout1(self.self_attn(self.norm1(x), causal=True))
        else:
            x = x + self.dropout1(self.self_attn(self.norm1(x), key_padding_mask=self_mask))
        x = x + self.dropout2(self.cross_attn(self.norm2(x), key_value=enc_out, key_padding_mask=src_mask))
        return x + self.dropout3(self.ffn(self.norm3(x)))


class _A2Stack(nn.Module):
    def __init__(self, emb, layers, d):
        super().__init__()
        self.embedding, self.layers, self.final_norm = emb, nn.ModuleList(layers), nn.LayerNorm(d)


class A2V2EncDec(nn.Module):
    def __init__(self, vocab_size=32000, d_model=768, heads=12, enc_layers=6, dec_layers=8, d_ff=3072,
                 dropout=0.05, enc_max=1024, dec_max=256, pad=0):
        super().__init__()
        self.cfg = dict(vocab_size=vocab_size, d_model=d_model, heads=heads, enc_layers=enc_layers,
                        dec_layers=dec_layers, d_ff=d_ff, enc_max=enc_max, dec_max=dec_max)
        ee = _A2Emb(vocab_size, d_model, enc_max, pad, dropout)
        de = _A2Emb(vocab_size, d_model, dec_max, pad, dropout)
        de.token_embedding.weight = ee.token_embedding.weight
        self.encoder = _A2Stack(ee, [_A2EncLayer(d_model, heads, d_ff, dropout) for _ in range(enc_layers)], d_model)
        self.decoder = _A2Stack(de, [_A2DecLayer(d_model, heads, d_ff, dropout) for _ in range(dec_layers)], d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = de.token_embedding.weight

    def encode(self, ids, src_mask):
        x = self.encoder.embedding(ids)
        for l in self.encoder.layers:
            x = l(x, src_mask)
        return self.encoder.final_norm(x)

    def decode_hidden(self, ids, enc, src_mask):
        x = self.decoder.embedding(ids)
        for l in self.decoder.layers:
            x = l(x, enc, src_mask)
        return self.decoder.final_norm(x)


def load_encdec2(ckpt_path):
    """Archit-2-v2 best_model.pt (a plain state_dict). Shapes are read from the weights (v2: 768/6/8/3072; v1: 512/6/6/2048)."""
    sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    if isinstance(sd, dict) and "model" in sd and not any(k.startswith("encoder.") for k in sd):
        sd = sd["model"]
    if any(k.startswith("module.") for k in sd):
        sd = {k[7:]: v for k, v in sd.items()}
    V, d = sd["encoder.embedding.token_embedding.weight"].shape
    ne = len({k.split(".")[2] for k in sd if k.startswith("encoder.layers.")})
    nd = len({k.split(".")[2] for k in sd if k.startswith("decoder.layers.")})
    m = A2V2EncDec(vocab_size=V, d_model=d, heads=d // 64, enc_layers=ne, dec_layers=nd,
                   d_ff=sd["encoder.layers.0.ffn.fc1.weight"].shape[0],
                   enc_max=sd["encoder.embedding.position_embedding.weight"].shape[0],
                   dec_max=sd["decoder.embedding.position_embedding.weight"].shape[0])
    m.load_state_dict(sd, strict=True)
    return m


def load_backbone(J):
    if J.get("arch", "txl") == "encdec2":
        return load_encdec2(J["phase1_ckpt"])
    if J.get("arch", "txl") == "ret":
        return load_retnet(J["model_py"], J["phase1_ckpt"])[0]
    if J.get("arch", "txl") == "encdec":
        return load_encdec(J["model_py"], J["phase1_ckpt"])[0]
    if J.get("arch", "txl") == "dec":
        return load_decoder(J["model_py"], J["phase1_ckpt"])[0]
    return load_phase1(J.get("model_py", J.get("txl_model_py")), J["phase1_ckpt"])[0]


class LowRankLinear(nn.Module):
    """mode='lora'  : y = W x + b + s * B(A x)
       mode='lorma' : y = (I + s * B A) (W x) + b       (LoRMA_+, multiplicative, identity at init)"""

    def __init__(self, base: nn.Linear, r, alpha, dropout, mode):
        super().__init__()
        self.base, self.mode, self.s = base, mode, alpha / r
        for p in base.parameters():
            p.requires_grad_(False)
        din = base.in_features if mode == "lora" else base.out_features
        self.A = nn.Parameter(torch.empty(r, din))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        if self.mode == "lora":
            return self.base(x) + self.s * F.linear(F.linear(self.drop(x), self.A), self.B)
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.s * F.linear(F.linear(self.drop(y0), self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias


class Adapter(nn.Module):
    """Houlsby-style bottleneck adapter applied to the output of a sub-layer projection."""

    def __init__(self, base: nn.Linear, bottleneck):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad_(False)
        d = base.out_features
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, 0, 0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))


class BiasLinear(nn.Module):
    """BitFit for bias-free projections: frozen W plus a trainable, zero-initialised bias."""

    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad_(False)
        self.bias = nn.Parameter(torch.zeros(base.out_features))

    def forward(self, x):
        return self.base(x) + self.bias


class TaggerHead(nn.Module):
    def _init_head(self, d, n_tags, dropout, head):
        self.drop = nn.Dropout(dropout)
        self.head_type = head
        if head == "bilstm":
            self.lstm = nn.LSTM(d, 256, batch_first=True, bidirectional=True)
            self.head = nn.Linear(512, n_tags)
        else:
            self.head = nn.Linear(d, n_tags)

    def forward(self, ids, attn_mask):
        h = self.hidden(ids, attn_mask)
        if self.head_type == "bilstm":
            h, _ = self.lstm(self.drop(h))
        return self.head(self.drop(h))


class TXLTagger(TaggerHead):
    """Transformer-XL (Phase-1 weights) used as a token encoder.
    attn='bidir' : every token sees left AND right context; relative distances are extended to negative
                   values (the same sinusoid / r_net / v-bias parametrisation, just evaluated at d<0).
    attn='causal': exactly the Phase-1 computation (verified equal in the notebook test)."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear"):
        super().__init__()
        self.attn_mode = attn
        self.emb, self.pos_emb, self.ln_f = base.emb, base.pos_emb, base.ln_f
        keep = list(range(len(base.blocks))) if layers is None else list(layers)
        self.layer_ids = keep
        self.blocks = nn.ModuleList([base.blocks[i] for i in keep])
        self.d = base.cfg.d_model
        for m in self.blocks.modules():
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self.final_norm = self.ln_f
        self._init_head(self.d, n_tags, dropout, head)

    def _block(self, blk, h, r, gidx, mask):
        a = blk.attn
        B, L, _ = h.shape
        H, dh = a.H, a.dh
        x = blk.ln1(h)
        q = a.q_net(x).view(B, L, H, dh).transpose(1, 2)
        kv = a.kv_net(x).view(B, L, 2, H, dh)
        k, v = kv[:, :, 0].transpose(1, 2), kv[:, :, 1].transpose(1, 2)
        rk = a.r_net(r).view(-1, H, dh).transpose(0, 1).unsqueeze(0)          # [1,H,2L-1,dh]
        qu = (q + a.u[None, :, None, :]) * a.scale
        qv = (q + a.v[None, :, None, :]) * a.scale
        AC = torch.matmul(qu, k.transpose(-1, -2))                              # [B,H,L,L]
        BDf = torch.matmul(qv, rk.transpose(-1, -2))                            # [B,H,L,2L-1]
        BD = BDf.gather(-1, gidx.expand(B, H, L, L))
        score = (AC + BD).float().masked_fill(mask, -1e30)
        prob = a.dropatt(F.softmax(score, dim=-1)).to(v.dtype)
        out = torch.matmul(prob, v).transpose(1, 2).reshape(B, L, H * dh)
        h = h + a.drop(a.o_net(out))
        h = h + blk.drop(blk.ff2(blk.drop(F.gelu(blk.ff1(blk.ln2(h))))))
        return h

    def hidden(self, ids, attn_mask):
        """ids [B,L] long, attn_mask [B,L] bool (True = real token) -> final hidden [B,L,D]."""
        B, L = ids.shape
        dev = ids.device
        h = self.emb(ids)
        pos = torch.arange(L - 1, -L, -1, device=dev, dtype=torch.float32)       # distance d = L-1-c
        r = self.pos_emb(pos)
        i = torch.arange(L, device=dev)
        gidx = ((L - 1) - i[:, None] + i[None, :])[None, None]                   # c for distance i-j
        mask = (~attn_mask)[:, None, None, :]                                    # [B,1,1,L]
        if self.attn_mode == "causal":
            mask = mask | torch.ones(L, L, dtype=torch.bool, device=dev).triu(1)[None, None]
        for blk in self.blocks:
            h = self._block(blk, h, r, gidx, mask)
        return self.ln_f(h)


class DecTagger(TaggerHead):
    """Decoder-only LegalLM (RoPE, RMSNorm, SwiGLU; Phase-1 weights) used as a token encoder.
    attn='bidir' : causal mask removed (RoPE is relative, so left and right distances are both defined)
    attn='causal': exactly the Phase-1 computation (verified equal in the notebook test)."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear"):
        super().__init__()
        self.attn_mode = attn
        self.cfg = base.cfg
        self.tok_emb, self.norm = base.tok_emb, base.norm
        keep = list(range(len(base.blocks))) if layers is None else list(layers)
        self.layer_ids = keep
        self.blocks = nn.ModuleList([base.blocks[i] for i in keep])
        self.d = base.cfg.d_model
        self.rope_mod = sys.modules[type(base).__module__]
        for blk in self.blocks:
            blk.drop.p = dropout
        self.final_norm = self.norm
        self._init_head(self.d, n_tags, dropout, head)

    def hidden(self, ids, attn_mask):
        B, T = ids.shape
        dev = ids.device
        hd = self.cfg.d_model // self.cfg.n_head
        pos = torch.arange(T, device=dev)[None]
        cos, sin = self.rope_mod.rope_cos_sin(pos, hd, self.cfg.rope_theta)
        allow = attn_mask[:, None, None, :]                                   # [B,1,1,T] True = may attend
        if self.attn_mode == "causal":
            allow = allow & torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None, None]
        allow = allow | torch.eye(T, dtype=torch.bool, device=dev)[None, None]  # pad rows never fully masked
        x = self.tok_emb(ids)
        for blk in self.blocks:
            a = blk.attn
            hx = blk.attn_norm(x)
            q = a.q_proj(hx).view(B, T, a.h, a.hd).transpose(1, 2)
            k = a.k_proj(hx).view(B, T, a.h, a.hd).transpose(1, 2)
            v = a.v_proj(hx).view(B, T, a.h, a.hd).transpose(1, 2)
            q, k = self.rope_mod.apply_rope(q, cos, sin), self.rope_mod.apply_rope(k, cos, sin)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=allow)
            x = x + blk.drop(a.o_proj(y.transpose(1, 2).reshape(B, T, -1)))
            x = x + blk.drop(blk.mlp(blk.mlp_norm(x)))
        return self.norm(x)


class EncTagger(TaggerHead):
    """Phase-1 encoder-decoder as a tagger.
    use_decoder=False : encoder only (natively bidirectional).
    use_decoder=True  : BART-style. The window goes through the encoder AND, as decoder input, through the decoder
                        (self-attention + cross-attention to the encoder); tags are read from the DECODER states.
                        attn='bidir' removes the decoder's causal mask, attn='causal' keeps it (= Phase-1 computation).
    KD student layers (if given) are applied to both stacks."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear", use_decoder=False):
        super().__init__()
        self.attn_mode, self.use_decoder = attn, use_decoder
        self.cfg = base.c
        self.tok, self.enc_pos, self.enc_norm = base.tok, base.enc_pos, base.enc_norm
        ke = list(range(len(base.enc_layers))) if layers is None else [i for i in layers if i < len(base.enc_layers)]
        self.enc_blocks = nn.ModuleList([base.enc_layers[i] for i in ke])
        if use_decoder:
            self.dec_pos, self.dec_norm = base.dec_pos, base.dec_norm
            kd = list(range(len(base.dec_layers))) if layers is None else [i for i in layers if i < len(base.dec_layers)]
            self.dec_blocks = nn.ModuleList([base.dec_layers[i] for i in kd])
            self.blocks = nn.ModuleList(list(self.enc_blocks) + list(self.dec_blocks))   # for PEFT / partial
            self.final_norm = self.dec_norm
        else:
            self.blocks = self.enc_blocks
            self.final_norm = self.enc_norm
        self.layer_ids = layers
        self.d = base.c.d_model
        for m in self.blocks.modules():
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self.emb_drop = nn.Dropout(dropout)
        self._init_head(self.d, n_tags, dropout, head)

    def encode(self, ids, attn_mask):
        T = ids.size(1)
        dev = ids.device
        x = self.emb_drop(self.tok(ids) + self.enc_pos(torch.arange(T, device=dev))[None])
        mask = attn_mask[:, None, None, :]
        if self.attn_mode == "causal" and not self.use_decoder:
            mask = mask & torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None, None]
        mask = mask | torch.eye(T, dtype=torch.bool, device=dev)[None, None]
        for l in self.enc_blocks:
            x = l(x, mask)
        return self.enc_norm(x)

    def hidden(self, ids, attn_mask):
        mem = self.encode(ids, attn_mask)
        if not self.use_decoder:
            return mem
        T = ids.size(1)
        dev = ids.device
        x = self.emb_drop(self.tok(ids) + self.dec_pos(torch.arange(T, device=dev))[None])
        cmask = attn_mask[:, None, None, :]                                     # cross-attn: real encoder tokens
        smask = attn_mask[:, None, None, :]
        if self.attn_mode == "causal":
            smask = smask & torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None, None]
        smask = smask | torch.eye(T, dtype=torch.bool, device=dev)[None, None]
        for l in self.dec_blocks:
            x = x + l.drop(l.self_attn(l.n1(x), mask=smask))
            x = x + l.drop(l.cross(l.n2(x), mem=mem, mask=cmask))
            x = x + l.drop(l.ffn(l.n3(x)))
        return self.dec_norm(x)


class ED2Tagger(TaggerHead):
    """Archit-2-v2 encoder-decoder as a tagger.
    use_decoder=False : tags from the ENCODER (natively bidirectional, 1024 positions). attn must be 'bidir'.
    use_decoder=True  : BART-style. Window -> encoder, same window -> decoder (cross-attends to the encoder);
                        tags from the decoder states. attn='causal' = Phase-1 computation. Decoder has 256 positions.
    KD student layers (if given) are applied to both stacks."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear", use_decoder=False):
        super().__init__()
        self.attn_mode, self.use_decoder = attn, use_decoder
        enc = base.encoder
        self.enc_emb, self.enc_norm = enc.embedding, enc.final_norm
        ke = list(range(len(enc.layers))) if layers is None else [i for i in layers if i < len(enc.layers)]
        self.enc_blocks = nn.ModuleList([enc.layers[i] for i in ke])
        if use_decoder:
            dec = base.decoder
            self.dec_emb, self.dec_norm = dec.embedding, dec.final_norm
            kd = list(range(len(dec.layers))) if layers is None else [i for i in layers if i < len(dec.layers)]
            self.dec_blocks = nn.ModuleList([dec.layers[i] for i in kd])
            self.blocks = nn.ModuleList(list(self.enc_blocks) + list(self.dec_blocks))
            self.final_norm = self.dec_norm
        else:
            self.blocks = self.enc_blocks
            self.final_norm = self.enc_norm
        self.layer_ids = layers
        self.d = enc.embedding.d_model
        for m in list(self.blocks.modules()) + [self.enc_emb.dropout] + ([self.dec_emb.dropout] if use_decoder else []):
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self._init_head(self.d, n_tags, dropout, head)

    def hidden(self, ids, attn_mask):
        x = self.enc_emb(ids)
        for l in self.enc_blocks:
            x = l(x, attn_mask)
        mem = self.enc_norm(x)
        if not self.use_decoder:
            return mem
        y = self.dec_emb(ids)
        sm = None if self.attn_mode == "causal" else attn_mask
        for l in self.dec_blocks:
            y = l(y, mem, attn_mask, self_mask=sm)
        return self.dec_norm(y)


class RetTagger(TaggerHead):
    """RetNet (Phase-1 weights) as a token encoder, parallel form of multi-scale retention over a window.
    attn='causal': D_nm = gamma^(n-m) for n>=m  (exactly the Phase-1 computation; verified in the notebook)
    attn='bidir' : D_nm = gamma^|n-m|           (symmetric decay, so every token also retains its right context)
    Padding uses document id -1, so the same-document mask of RetNet also removes pads."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear"):
        super().__init__()
        self.attn_mode = attn
        self.emb, self.final_ln = base.emb, base.final_ln
        self.register_buffer("inv_freq", base.inv_freq.clone(), persistent=False)
        keep = list(range(len(base.layers))) if layers is None else list(layers)
        self.layer_ids = keep
        self.blocks = nn.ModuleList([base.layers[i] for i in keep])
        self.d = base.d_model
        self.rot = sys.modules[type(base).__module__].rotate
        self.emb_drop = nn.Dropout(dropout)
        for blk in self.blocks:
            blk.drop.p = dropout
        self.final_norm = self.final_ln
        self._init_head(self.d, n_tags, dropout, head)

    def _msr(self, m, x, cos, sin, doc):
        B, L, _ = x.shape
        H, dk, dv = m.H, m.dk, m.dv
        q = m.wq(x).view(B, L, H, dk)
        k = m.wk(x).view(B, L, H, dk)
        v = m.wv(x).view(B, L, H, dv)
        g = m.wg(x)
        with torch.autocast(device_type=x.device.type, enabled=False):
            q = (self.rot(q.float(), cos, sin) * dk ** -0.5).transpose(1, 2)
            k = self.rot(k.float(), cos, sin).transpose(1, 2)
            v = v.float().transpose(1, 2)
            i = torch.arange(L, device=x.device, dtype=torch.float32)
            diff = i.view(L, 1) - i.view(1, L)
            lg = m.log_gamma.view(H, 1, 1)
            if self.attn_mode == "causal":
                D = torch.exp(lg * diff.clamp(min=0)) * (diff >= 0)
            else:
                D = torch.exp(lg * diff.abs())
            same = (doc.unsqueeze(2) == doc.unsqueeze(1)).unsqueeze(1)
            o = torch.matmul(torch.matmul(q, k.transpose(-1, -2)) * D.unsqueeze(0) * same, v)
            o = o.transpose(1, 2).reshape(B * L, H * dv)
            o = F.group_norm(o, H, m.gn_w.float(), m.gn_b.float(), 1e-5)
        o = o.view(B, L, H * dv)
        return m.wo(F.silu(g) * o.to(g.dtype))

    def hidden(self, ids, attn_mask):
        B, L = ids.shape
        doc = torch.where(attn_mask, torch.zeros_like(ids), torch.full_like(ids, -1))
        pos = torch.arange(L, device=ids.device)[None].expand(B, L)
        ang = pos.float().unsqueeze(-1) * self.inv_freq
        cos, sin = ang.cos().unsqueeze(2), ang.sin().unsqueeze(2)
        h = self.emb_drop(self.emb(ids))
        for blk in self.blocks:
            h = h + blk.drop(self._msr(blk.msr, blk.ln1(h), cos, sin, doc))
            y = blk.ln2(h)
            h = h + blk.drop(blk.down(F.silu(blk.gate(y)) * blk.up(y)))
        return self.final_ln(h)


def _linear_slots(blk, names):
    """(holder_module, attribute) for every nn.Linear attribute in `names` inside one block."""
    out = []
    for mod in blk.modules():
        for nm in names:
            if isinstance(getattr(mod, nm, None), nn.Linear) and not isinstance(mod, (LowRankLinear, Adapter, BiasLinear)):
                out.append((mod, nm))
    return out


def apply_method(model, method, J):
    """Freeze / wrap according to the adaptation method. The head is always trainable."""
    for p in model.parameters():
        p.requires_grad_(method in ("full", "kd"))
    if method == "partial":
        for blk in list(model.blocks)[-J["partial_top_layers"]:]:
            for p in blk.parameters():
                p.requires_grad_(True)
        for p in model.final_norm.parameters():
            p.requires_grad_(True)
    elif method in ("lora", "lorma"):
        for blk in model.blocks:
            for holder, nm in _linear_slots(blk, J["lowrank_targets"]):
                setattr(holder, nm, LowRankLinear(getattr(holder, nm), J["rank"], J["alpha"], J["lowrank_dropout"], method))
    elif method == "adapter":
        for blk in model.blocks:
            for holder, nm in _linear_slots(blk, J.get("adapter_targets", ["o_net", "ff2"])):
                setattr(holder, nm, Adapter(getattr(holder, nm), J["adapter_dim"]))
    elif method == "bitfit":
        if J.get("arch", "txl") in ("dec", "ret"):            # bias-free model: add zero-init biases + norms
            for blk in model.blocks:
                for holder, nm in _linear_slots(blk, J["lowrank_targets"]):
                    setattr(holder, nm, BiasLinear(getattr(holder, nm)))
        for n, p in model.named_parameters():
            if n.endswith("bias") or n.endswith(".u") or n.endswith(".v") or n.endswith(".gn_w") or n.endswith(".gn_b"):
                p.requires_grad_(True)
        for m in model.modules():                              # every LayerNorm / RMSNorm
            if "Norm" in type(m).__name__:
                for p in m.parameters():
                    p.requires_grad_(True)
    for p in model.head.parameters():
        p.requires_grad_(True)
    if getattr(model, "lstm", None) is not None:
        for p in model.lstm.parameters():
            p.requires_grad_(True)
    return model


def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr


# ============================================================== windows / batching
def windows_for(n, L, stride):
    if n <= L:
        return [0]
    st = list(range(0, n - L + 1, stride))
    if st[-1] + L < n:
        st.append(n - L)
    return st


def make_train_windows(docs, L, stride):
    W = []
    for d in docs.values():
        n = len(d["ids"])
        for s in windows_for(n, L, stride):
            W.append((d["ids"][s:s + L], d["y"][s:s + L]))
    return W


def collate(items, with_y=True):
    L = max(len(x[0]) for x in items)
    ids = torch.zeros(len(items), L, dtype=torch.long)
    am = torch.zeros(len(items), L, dtype=torch.bool)
    y = torch.full((len(items), L), -100, dtype=torch.long)
    for b, it in enumerate(items):
        n = len(it[0])
        ids[b, :n] = torch.as_tensor(it[0], dtype=torch.long)
        am[b, :n] = True
        if with_y:
            y[b, :n] = torch.as_tensor(it[1], dtype=torch.long)
    return ids, am, y


@torch.no_grad()
def doc_logprobs(model, docs, L, stride, bs, device, amp=True):
    """Merge per-window log-probs into one [n, T] array per doc (triangular weights: tokens near a window
    edge get little weight, so every token is mostly judged with context on both sides)."""
    model.eval()
    jobs = []
    for did, d in docs.items():
        n = len(d["ids"])
        if n:
            for s in windows_for(n, L, stride):
                jobs.append((did, s, min(L, n - s)))
    T = model.head.out_features
    acc = {did: np.zeros((len(d["ids"]), T), np.float32) for did, d in docs.items()}
    wsum = {did: np.zeros((len(d["ids"]), 1), np.float32) for did, d in docs.items()}
    jobs.sort(key=lambda j: -j[2])
    for k in range(0, len(jobs), bs):
        chunk = jobs[k:k + bs]
        items = [(docs[did]["ids"][s:s + n], None) for did, s, n in chunk]
        ids, am, _ = collate(items, with_y=False)
        ids, am = ids.to(device), am.to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp and device.type == "cuda"):
            lg = model(ids, am)
        lp = F.log_softmax(lg.float(), -1).cpu().numpy()
        for b, (did, s, n) in enumerate(chunk):
            pos = np.arange(n)
            w = np.minimum(pos + 1, n - pos).astype(np.float32)[:, None]
            if s == 0:
                w[: n // 2] = np.maximum(w[: n // 2], n / 2)        # doc start: no left context exists anyway
            if s + n == len(docs[did]["ids"]):
                w[n // 2:] = np.maximum(w[n // 2:], n / 2)          # doc end
            acc[did][s:s + n] += lp[b, :n] * w
            wsum[did][s:s + n] += w
    return {did: acc[did] / np.maximum(wsum[did], 1e-6) for did in docs}


def decode_docs(docs, lps, tags, use_viterbi=True):
    allowed, start_ok = bio_transitions(tags)
    pred = {}
    for did, d in docs.items():
        lp = lps[did]
        tg = viterbi(lp, allowed, start_ok) if use_viterbi else lp.argmax(-1)
        pred[did] = spans_from_tags(d["text"], d, tg, tags)
    return pred


# ============================================================== training
def class_weights(windows, T, mode):
    if mode == "none":
        return None
    cnt = np.bincount(np.concatenate([w[1] for w in windows]), minlength=T).astype(np.float64)
    w = np.ones(T)
    ent = cnt[1:]
    mean = ent[ent > 0].mean()
    w[1:] = np.clip(np.sqrt(mean / np.maximum(ent, 1)), 0.5, 4.0)
    return torch.tensor(w, dtype=torch.float32)


def build_model(J, base_factory, n_tags, method):
    base = base_factory()
    layers = J["kd_student_layers"] if method == "kd" else None
    cls = {"dec": DecTagger, "encdec": EncTagger, "encdec2": ED2Tagger, "ret": RetTagger}.get(J.get("arch", "txl"), TXLTagger)
    kw = dict(use_decoder=J.get("encdec_use_decoder", False)) if cls in (EncTagger, ED2Tagger) else {}
    if cls is ED2Tagger and not kw["use_decoder"]:
        assert J["attn"] == "bidir", "encoder-only tagging is always bidirectional"
    m = cls(base, n_tags, attn=J["attn"], layers=layers, dropout=J["dropout"], head=J["head"], **kw)
    return apply_method(m, method, J)


def train_one(J, model, train_docs, val_docs, tags, labels, device, epochs, log, teacher=None):
    torch.manual_seed(J["seed"]); np.random.seed(J["seed"])
    model.to(device)
    W = make_train_windows(train_docs, J["max_len"], J["train_stride"])
    cw = class_weights(W, len(tags), J["class_weight"])
    cw = cw.to(device) if cw is not None else None
    head_names = {n for n, _ in model.named_parameters() if n.startswith("head") or n.startswith("lstm")}
    new_ids = set()
    for mm in model.modules():
        if isinstance(mm, LowRankLinear):
            new_ids |= {id(mm.A), id(mm.B)}
        elif isinstance(mm, Adapter):
            new_ids |= {id(p) for p in list(mm.down.parameters()) + list(mm.up.parameters())}
    new_names = {n for n, p in model.named_parameters() if id(p) in new_ids}
    g_head = [p for n, p in model.named_parameters() if p.requires_grad and (n in head_names or n in new_names)]
    g_body = [p for n, p in model.named_parameters() if p.requires_grad and n not in head_names and n not in new_names]
    groups = [dict(params=g_head, lr=J["head_lr"])]
    if g_body:
        groups.append(dict(params=g_body, lr=J["lr"], is_body=True))
    opt = torch.optim.AdamW(groups, weight_decay=J["weight_decay"])
    for g in opt.param_groups:
        g["base_lr"] = g["lr"]
    steps_per_epoch = math.ceil(len(W) / J["batch_size"])
    total = steps_per_epoch * epochs
    warm = max(1, int(0.06 * total))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", init_scale=2.0 ** 10, enabled=use_amp)   # low start: fewer skipped fp16 steps
    n_skip = 0
    best = dict(f1=-1, epoch=0, state=None, report=None, pred=None)
    hist, step, bad = [], 0, 0
    t0 = time.time()
    if use_amp:
        torch.cuda.reset_peak_memory_stats(device)
    for ep in range(1, epochs + 1):
        model.train()
        perm = np.random.permutation(len(W))
        tl, tn, gsum = 0.0, 0, [0.0]
        for k in range(0, len(W), J["batch_size"]):
            ids, am, y = collate([W[i] for i in perm[k:k + J["batch_size"]]])
            ids, am, y = ids.to(device), am.to(device), y.to(device)
            f = (step + 1) / warm if step < warm else max(0.0, (total - step) / max(1, total - warm))   # warmup + linear decay
            # LP-FT: optionally keep the pretrained body frozen for the first epoch(s) so the random head
            # does not distort the Phase-1 features, then re-warm the body learning rate.
            nf = J.get("freeze_body_epochs", 0)
            if ep <= nf:
                fb = 0.0
            else:
                bs = step - nf * steps_per_epoch
                fb = min(1.0, (bs + 1) / warm) * (1.0 if step < warm else max(0.0, (total - step) / max(1, total - warm)))
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * (fb if (g.get("is_body") and nf) else f)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                lg = model(ids, am)
            lg = lg.float()
            loss = F.cross_entropy(lg.view(-1, lg.size(-1)), y.view(-1), weight=cw, ignore_index=-100)
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                    tl_ = teacher(ids, am).float()
                Tt = J["kd_temperature"]
                kl = F.kl_div(F.log_softmax(lg / Tt, -1), F.log_softmax(tl_ / Tt, -1), log_target=True, reduction="none").sum(-1)
                kl = (kl * am).sum() / am.sum()
                loss = J["kd_alpha"] * loss + (1 - J["kd_alpha"]) * (Tt ** 2) * kl
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            # clip each parameter group separately: a large body gradient must never shrink the head's update
            gn = [float(torch.nn.utils.clip_grad_norm_(g["params"], 1.0)) for g in opt.param_groups]
            gsum = [a + b for a, b in zip(gsum, gn)] if tn else gn
            sc0 = scaler.get_scale() if use_amp else 1.0
            scaler.step(opt); scaler.update()
            if use_amp and scaler.get_scale() < sc0:
                n_skip += 1                                     # overflow -> this optimizer step was skipped
            tl += loss.item(); tn += 1; step += 1
        rec = dict(epoch=ep, train_loss=tl / max(tn, 1), secs=time.time() - t0, skipped_steps=n_skip,
                   grad_norm_head=round(gsum[0] / max(tn, 1), 3),
                   grad_norm_body=round(gsum[1] / max(tn, 1), 3) if len(gsum) > 1 else None)
        if val_docs:
            lps = doc_logprobs(model, val_docs, J["max_len"], J["eval_stride"], J["eval_batch_size"], device)
            pred = decode_docs(val_docs, lps, tags)
            sc = strict_scores({d: v["gold"] for d, v in val_docs.items()}, pred, labels)
            rec.update(val_f1=sc["macro_f1"], val_p=sc["macro_p"], val_r=sc["macro_r"])
            if sc["macro_f1"] > best["f1"]:
                best.update(f1=sc["macro_f1"], epoch=ep, report=sc, pred=pred,
                            state={n: p.detach().cpu().clone() for n, p in model.state_dict().items()})
                bad = 0
            else:
                bad += 1
        hist.append(rec)
        log(json.dumps(rec))
        if val_docs and bad >= J["patience"]:
            log(f"early stop at epoch {ep} (best epoch {best['epoch']}, val F1 {best['f1']:.4f})")
            break
    info = dict(train_secs=time.time() - t0, history=hist, n_windows=len(W), steps=step,
                peak_mem_gb=(torch.cuda.max_memory_allocated(device) / 2**30) if use_amp else None)
    return best, info


# ============================================================== one experiment (script entry point)
def run_job(job_path):
    J = json.load(open(job_path))
    out = Path(J["out_dir"]); out.mkdir(parents=True, exist_ok=True)
    logf = open(out / "log.txt", "a")

    def log(msg):
        line = f"[{time.strftime('%H:%M:%S')}][{J['name']}] {msg}"
        print(line, flush=True); logf.write(line + "\n"); logf.flush()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    D = pickle.load(open(J["cache"], "rb"))
    labels, tags, folds = D["labels"], D["tags"], D["folds"]
    if J.get("quick"):
        folds = {f: dict(list(v.items())[: J["quick_docs"]]) for f, v in folds.items()}
    method = J["method"]
    base_factory = lambda: load_backbone(J)
    log(f"arch={J.get('arch', 'txl')} method={method} attn={J['attn']} head={J['head']} device={device}")

    def get_teacher(which):
        tp = Path(J["kd_teacher_dir"]) / f"{which}_model.pt"
        assert tp.exists(), f"KD needs the teacher checkpoint {tp} (run the teacher job first)"
        tj = json.load(open(Path(J["kd_teacher_dir"]) / "job.json"))
        t = build_model(tj, base_factory, len(tags), tj["method"])
        t.load_state_dict(torch.load(tp, map_location="cpu"))
        t.to(device).eval()
        for p in t.parameters():
            p.requires_grad_(False)
        return t

    results = dict(name=J["name"], method=method, attn=J["attn"], head=J["head"], config=J)
    # ---- (1) model selection: train fold1 -> validate fold2
    model = build_model(J, base_factory, len(tags), method)
    results["params_total"], results["params_trainable"] = count_params(model)
    log(f"params total={results['params_total']/1e6:.2f}M trainable={results['params_trainable']/1e6:.3f}M")
    teacher = get_teacher("val") if method == "kd" else None
    if teacher is not None:
        results["teacher_params"] = count_params(teacher)[0]
    best, info = train_one(J, model, folds["fold1"], folds["fold2"], tags, labels, device, J["epochs"], log, teacher)
    results.update(val_macro_f1=best["f1"], best_epoch=best["epoch"], val_report=best["report"], val_train=info,
                   val_errors=error_breakdown({d: v["gold"] for d, v in folds["fold2"].items()}, best["pred"]))
    json.dump(best["pred"], open(out / "val_pred_fold2.json", "w"))
    log(f"VALIDATION fold1->fold2 strict macro-F1 = {best['f1']:.4f} (epoch {best['epoch']})")
    if J["save_model"]:
        torch.save(best["state"], out / "val_model.pt")
    del teacher
    # ---- (2) final model
    if J["final_fit"] == "all":
        epochs = max(1, best["epoch"] + J.get("final_extra_epochs", 0))
        model = build_model(J, base_factory, len(tags), method)
        teacher = get_teacher("final") if method == "kd" else None
        train_docs = {**folds["fold1"], **folds["fold2"]}
        _, finfo = train_one(J, model, train_docs, None, tags, labels, device, epochs, log, teacher)
        results["final_train"] = finfo
        del teacher
    else:
        model.load_state_dict(best["state"])
    model.to(device)
    if J["save_model"]:
        torch.save({n: p.detach().cpu() for n, p in model.state_dict().items()}, out / "final_model.pt")
    # ---- (3) test prediction
    t0 = time.time()
    test = folds["fold3"]
    lps = doc_logprobs(model, test, J["max_len"], J["eval_stride"], J["eval_batch_size"], device)
    pred = decode_docs(test, lps, tags)
    results["test_infer_secs"] = time.time() - t0
    all_ids = list(D["folds"]["fold3"].keys())
    final = {did: [[int(s), int(e), l] for s, e, l in pred.get(did, [])] for did in all_ids}
    assert set(final) == set(all_ids)
    for did, spans in final.items():
        n = len(D["folds"]["fold3"][did]["text"])
        assert all(0 <= s < e <= n and l in labels for s, e, l in spans), did
    pj = out / "prediction.json"
    json.dump(final, open(pj, "w"))
    zp = out / f"submission_{J['name']}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    results["test_entities"] = sum(len(v) for v in final.values())
    results["test_label_counts"] = dict(Counter(l for v in final.values() for _, _, l in v))
    json.dump(results, open(out / "results.json", "w"), indent=1, default=str)
    log(f"DONE -> {zp} | test entities={results['test_entities']} | {json.dumps(results['test_label_counts'])}")


if __name__ == "__main__":
    run_job(sys.argv[1])


Writing /kaggle/working/p7/p2_ner.py


In [3]:
%%writefile /kaggle/working/p7/p5_lsi.py
"""CS-781 Phase 5 - Legal Statute Identification (LSI: multi-label, 100 statute ids, Macro-F1) on my Phase-1 models.

One file, used two ways (same as Phases 2-4):
  * imported by the notebook  -> data cleaning, tokenisation checks, chunk cache
  * run as a script            -> `python p5_lsi.py job.json` trains / evaluates / predicts ONE experiment

Data facts that shaped the design (printed again by the notebook):
  * 42,750 train / 10,181 dev / 13,019 test documents, exactly 100 statute ids (0-99), 3.7 statutes per document
    (max 31), every document has >= 1. Rarest statute: 272 train / 68 dev documents -> per-statute thresholds
    can be tuned reliably on dev for all 100 statutes.
  * Statute mentions are MASKED in the text (<SECTION>, <ACT>, <ENTITY>): the model must infer the statutes from the
    facts, but the masks show WHERE statutes were cited -> used for evidence-based chunk selection.
  * Median ~500 words, p90 ~2.8k words, max 135k words.

Model (hierarchical: chunk encoder + document pooling + label-wise attention)
  document --(EXACT Phase-1 preprocessing + tokenizer, encoded in whitespace-aligned pieces = identical ids)-->
    consecutive chunks of `chunk_len` tokens; up to `max_store` chunks are kept per document (all of them for ~95%
    of the documents; longer ones keep first + last + the chunks with the most <SECTION>/<ACT> markers)
    --> every chunk: Phase-1 backbone (bidirectional, + adaptation method) --> mean-pooled chunk vector
        + relative-position embedding (16 buckets) + first/last flag
    --> BiGRU over the chunks of the document
    --> label-wise attention (one attention query per statute over the chunks, LWAN) + a max-pool term
    --> 100 sigmoid logits.
  Training samples `train_chunks` random stored chunks per document; prediction uses all stored chunks.
  Loss: BCE with clipped pos_weight (default) | BCE | ASL | focal.  Thresholds: global vs per-statute, compared by
  2-fold cross-fitting on dev (honest estimate); every document gets >= 1 statute (top-1 fallback).

Data hygiene: load_lsi() drops every dev/train row whose id or exact text occurs in test (and train rows that occur in
dev), de-duplicates by text, de-duplicates label lists, and NEVER reads test labels.

Adaptation methods (method=...), Phase-5 list: full, partial / gradual (layer-wise), QLoRA, LoRMA, AdaLoRA, adapters,
prefix tuning, prompt tuning, KD.  Plus `frozen` (frozen Phase-1 encoder, document head only: a probe; its chunk
vectors are cached once, so it also runs a cheap ablation suite: losses, heads, long-document strategies).
  lorma    LoRMA_+ (Bihany, Patel & Modi 2025): y = (I + s*B A) W x                       (compulsory; KD teacher)
  kd       knowledge distillation: best fine-tuned teacher -> shallower student (subset of Phase-1 layers),
           loss = a * task_loss + (1-a) * T^2 * BCE(sigmoid(student/T), sigmoid(teacher/T))   (compulsory)
  qlora    QLoRA: frozen weights stored as 4-bit NormalFloat (NF4, block 64, double-quantised scales),
           de-quantised on the fly; LoRA adapters (16-bit) on every projection
  adalora  AdaLoRA via the `peft` library
  adapter  Houlsby bottleneck adapters on the attention-output and FFN-output projections
  prompt   prompt tuning (Lester et al.): `n_prompt` trainable soft tokens prepended to every chunk
  prefix   prefix tuning (deep prompts, P-Tuning-v2 form): trainable prefix vectors at the input of EVERY layer's
           attention, i.e. learned per-layer keys/values the real tokens attend to
  partial  top `partial_top_layers` blocks fine-tuned;  gradual: head first, then the top 2/4/6 blocks unfrozen
  full     full fine-tuning
"""
import hashlib
import json
import math
import multiprocessing as mp
import os
import pickle
import re
import sys
import time
import unicodedata
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, str(Path(__file__).resolve().parent))
import p2_ner as P2                            # noqa: E402  (Phase-2 loaders / encoders / LoRA / LoRMA / adapters)


# ============================================================== data: leak removal + de-duplication
def _md5(t):
    return hashlib.md5(t.encode("utf-8", "ignore")).hexdigest()


def _read(path):
    d = json.load(open(path, encoding="utf-8"))
    items = d if isinstance(d, list) else [dict(id=k, **v) if isinstance(v, dict) else dict(id=k, text=v) for k, v in d.items()]
    out, n_dup_lab = [], 0
    for it in items:
        t = it.get("text", "")
        t = " ".join(map(str, t)) if isinstance(t, list) else str(t)
        lab = it.get("labels", it.get("label"))
        if lab is not None:
            lab = lab if isinstance(lab, (list, tuple)) else [lab]
            n_dup_lab += int(len(set(lab)) != len(lab))
            lab = sorted({int(x) for x in lab})
        out.append(dict(id=it["id"], text=t, labels=lab))
    return out, n_dup_lab


def load_lsi(data_dir, log=print):
    """-> dict(train, dev, test, labels, stats). Every item: dict(id, text, labels) (labels None for test - never read)."""
    data_dir = Path(data_dir)
    test_raw, _ = _read(data_dir / "Test" / "test.json")
    test = [dict(id=x["id"], text=x["text"], labels=None) for x in test_raw]
    del test_raw
    test_ids = [str(x["id"]) for x in test]
    assert len(set(test_ids)) == len(test_ids), "duplicate ids in test.json"
    test_ids, test_h = set(test_ids), {_md5(x["text"]) for x in test}

    def clean(items, banned_ids, banned_h):
        seen, out, st = {}, [], Counter()
        for x in items:
            h = _md5(x["text"])
            if str(x["id"]) in banned_ids or h in banned_h:
                st["dropped_same_id_or_text_as_test_or_dev"] += 1
                continue
            if h in seen:
                st["dropped_duplicate_text"] += 1
                st["duplicates_with_different_labels"] += int(seen[h] != x["labels"])
                continue
            if not x["labels"]:
                st["dropped_no_labels"] += 1
                continue
            seen[h] = x["labels"]
            out.append(x)
        return out, dict(st)

    dev_raw, dup_dev = _read(data_dir / "Train" / "dev.json")
    n_dev_raw = len(dev_raw)
    dev, st_dev = clean(dev_raw, test_ids, test_h)
    del dev_raw
    ids = [str(x["id"]) for x in dev]
    assert len(set(ids)) == len(ids), "dev still has duplicate ids after de-duplication"
    train_raw, dup_tr = _read(data_dir / "Train" / "train.json")
    n_train_raw = len(train_raw)
    train, st_tr = clean(train_raw, test_ids | set(ids), test_h | {_md5(x["text"]) for x in dev})
    del train_raw
    labels = sorted({l for x in train for l in x["labels"]})
    dev_only = sorted({l for x in dev for l in x["labels"]} - set(labels))
    st_tr["label_lists_with_repeats_fixed"], st_dev["label_lists_with_repeats_fixed"] = dup_tr, dup_dev
    stats = dict(train_rows=n_train_raw, train_kept=len(train), train=st_tr, dev_rows=n_dev_raw, dev_kept=len(dev), dev=st_dev,
                 test=len(test), n_labels=len(labels), label_min=min(labels), label_max=max(labels), labels_only_in_dev=dev_only)
    log(json.dumps(stats))
    return dict(train=train, dev=dev, test=test, labels=labels, stats=stats)


def multi_hot(docs, labels):
    idx = {l: i for i, l in enumerate(labels)}
    Y = np.zeros((len(docs), len(labels)), bool)
    for r, d in enumerate(docs):
        for l in d["y"]:
            if l in idx:
                Y[r, idx[l]] = True
    return Y


def label_report(train, dev, labels, top=15):
    """Label frequencies, labels per document and co-occurrence (for the report)."""
    idx = {l: i for i, l in enumerate(labels)}
    Y = np.zeros((len(train), len(labels)), np.float32)
    for r, d in enumerate(train):
        for l in d["labels"]:
            Y[r, idx[l]] = 1
    f = Y.sum(0)
    C = Y.T @ Y
    np.fill_diagonal(C, 0)
    pairs = [(int(C[a, b]), labels[a], labels[b]) for a in range(len(labels)) for b in range(a + 1, len(labels)) if C[a, b] > 0]
    pairs.sort(reverse=True)
    cond = [(round(float(C[a, b] / f[a]), 3), labels[a], labels[b], int(f[a])) for a in range(len(labels)) for b in range(len(labels))
            if a != b and f[a] >= 100 and C[a, b] / max(f[a], 1) >= 0.5]
    cond.sort(reverse=True)
    lpd = Y.sum(1)
    return dict(train_docs_per_label={str(l): int(v) for l, v in zip(labels, f)},
                labels_per_doc=dict(mean=round(float(lpd.mean()), 3), p50=int(np.median(lpd)), p90=int(np.percentile(lpd, 90)),
                                    max=int(lpd.max()), hist={str(k): int(v) for k, v in sorted(Counter(lpd.astype(int).tolist()).items())}),
                top_pairs=[dict(a=a, b=b, docs=c) for c, a, b in pairs[:top]],
                strong_implications=[dict(if_label=a, then_label=b, p=p, support=s) for p, a, b, s in cond[:top]],
                dev_docs=len(dev))


# ============================================================== tokenisation (exact Phase-1 pipeline, whole document)
_MAP = str.maketrans({
    "‘": "'", "’": "'", "‚": "'", "‛": "'", "“": '"', "”": '"',
    "„": '"', "«": '"', "»": '"', "–": "-", "—": "-", "−": "-",
    "‐": "-", "‑": "-", "‒": "-", "―": "-", " ": " ", "•": "*",
    "·": "*", "…": "...", "₹": "Rs. ",
})
_CTRL = re.compile(r"[\x00-\x08\x0e-\x1f\x7f]")


def txl_normalize(t):
    """Exactly prep_data.normalize_text of the Phase-1 Transformer-XL runs."""
    if not t:
        return ""
    if not t.isascii():
        t = t.translate(_MAP)
        t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode("ascii")
    t = _CTRL.sub("", t)
    return " ".join(t.split())


def text_fns(tok, arch):
    """(normalize(raw) -> the text Phase-1 pretraining tokenised,  encode(normalized text) -> ids)."""
    if tok.kind == "hf":                         # Archit-2-v2: " ".join(text.split()) -> ByteLevelBPETokenizer
        return (lambda t: " ".join(t.split())), (lambda s: tok.tok.encode(s).ids)
    if arch == "txl" and tok.kind == "old":      # TXL old BPE: Phase 1 PATCHED a1_tokenizer so "_" is kept
        enc = tok.tok._encode_pretoken
        return txl_normalize, (lambda s: [i for pc in P2.PRE_RE.findall(s) for i in enc(pc)])
    if arch == "txl":                            # TXL new BPE: prep_data.normalize_text -> tokenizer.encode(text). Its own
        # NFKC/whitespace normalisation is a no-op on ASCII-folded text, so encode(normalize=False) gives the same ids -
        # and it does not strip the leading space of a piece (needed for the piece-wise encoding; checked in the notebook)
        return txl_normalize, (lambda s: tok.tok.encode(s, normalize=False))
    if tok.kind == "new":                        # new BPE: a1.normalize_text -> encode(normalize=False)
        return tok.mod.normalize_text, (lambda s: tok.tok.encode(s, normalize=False))
    if tok.norm == "clean":                      # RetNet: ret_data.clean_text -> ORIGINAL a1_tokenizer regex
        enc = tok.tok._encode_pretoken
        return P2.clean_text_ret, (lambda s: [i for pc in P2.PRE_RE_ORIG.findall(s) for i in enc(pc)])
    raise ValueError(f"no Phase-1 text pipeline for arch={arch} tokenizer={tok.kind}/{tok.norm}")


_WORD = re.compile(r"\S+")
MARK_RE = re.compile(r"<(?:SECTION|ACT)>")       # where a statute was cited (the statute itself is masked)


def encode_pieces(raw, norm_fn, enc_fn, piece_words=128):
    """Whole document -> (ids int64, [(token_offset, n_markers)], n_words).
    The normalised text is cut at the END of every `piece_words`-th word (= the start of a whitespace run) and each
    piece is encoded on its own. The Phase-1 pre-tokenisers never let a token cross such a position, so the
    concatenation is exactly the full-document encoding (asserted in the notebook) - but we also learn where the
    <SECTION>/<ACT> masks are in token space."""
    t = norm_fn(raw)
    ends = [m.end() for m in _WORD.finditer(t)]
    cuts = [0] + ends[piece_words - 1::piece_words]
    if cuts[-1] != len(t):
        cuts.append(len(t))
    ids, marks = [], []
    for a, b in zip(cuts, cuts[1:]):
        seg = t[a:b]
        e = enc_fn(seg)
        for m in MARK_RE.finditer(seg):
            marks.append(len(ids) + int(len(e) * m.start() / max(1, len(seg))))
        ids.extend(e)
    return np.asarray(ids, np.int64), marks, len(ends)


def select_chunks(n, mk, budget, mode):
    """Indices of the chunks to keep when a document has n > budget chunks."""
    if n <= budget:
        return list(range(n))
    if mode == "head":
        return list(range(budget))
    if mode == "tail":
        return list(range(n - budget, n))
    if mode == "headtail":
        return [0] + list(range(n - budget + 1, n))
    if mode == "uniform":
        return sorted({int(round(x)) for x in np.linspace(0, n - 1, budget)})
    assert mode == "marker", mode               # first + last + the chunks citing the most statutes, rest spread out
    keep = {0, n - 1}
    for k in sorted(range(1, n - 1), key=lambda k: (-mk[k], k)):
        if len(keep) >= budget or mk[k] <= 0:
            break
        keep.add(k)
    rest = [k for k in range(n) if k not in keep]
    need = budget - len(keep)
    if need > 0:
        keep |= {rest[int(round(x))] for x in np.linspace(0, len(rest) - 1, need)}
    return sorted(keep)


def make_chunks(ids, marks, chunk_len, bos, max_store, select):
    c = chunk_len - (bos is not None)
    if len(ids) == 0:
        ids = np.asarray([P2_UNK_FALLBACK], np.int64)
    n = (len(ids) + c - 1) // c
    mk = np.zeros(n, np.int32)
    for off in marks:
        mk[min(off // c, n - 1)] += 1
    keep = select_chunks(n, mk, max_store, select)
    pieces = [ids[k * c:(k + 1) * c] for k in keep]
    if bos is not None:
        pieces = [np.concatenate([[bos], p]) for p in pieces]
    return [np.asarray(p, np.uint16) for p in pieces], np.asarray(keep, np.int32), n, mk[keep]


P2_UNK_FALLBACK = 1          # a document with no tokens at all becomes one token (never happens on real data)
_W = {}                      # per-worker state (fork)


def _init_worker(spec):
    tok = P2.load_bpe(spec["tok_py"], spec["tok_path"], norm=spec["norm"])
    _W["norm"], _W["enc"] = text_fns(tok, spec["arch"])
    _W["spec"] = spec


def _work(i):
    s = _W["spec"]
    ids, marks, nw = encode_pieces(_W["texts"][i], _W["norm"], _W["enc"], s.get("piece_words", 128))
    chunks, kidx, ntot, mk = make_chunks(ids, marks, s["chunk_len"], s["bos"], s["max_store"], s["select"])
    return i, chunks, kidx, ntot, mk, nw, len(ids), len(marks)


def prepare(data, spec, cache_path, n_proc=None, log=print):
    """Tokenise + chunk every split (multiprocessing, fork) and pickle {'spec', 'splits', 'labels', 'stats'}."""
    n_proc = n_proc or max(1, os.cpu_count() or 1)
    splits = {}
    for sp in ("train", "dev", "test"):
        docs = data[sp]
        _W["texts"] = [d["text"] for d in docs]
        out = [None] * len(docs)
        t0 = time.time()
        if n_proc > 1:
            with mp.get_context("fork").Pool(n_proc, initializer=_init_worker, initargs=(spec,)) as pool:
                for r in pool.imap_unordered(_work, range(len(docs)), chunksize=16):
                    out[r[0]] = r[1:]
        else:
            _init_worker(spec)
            for i in range(len(docs)):
                out[i] = _work(i)[1:]
        splits[sp] = [dict(id=d["id"], y=d["labels"], chunks=ch, kidx=ki, ntot=nt, mk=mk, n_words=nw, n_tok=ntk, n_marks=nm)
                      for d, (ch, ki, nt, mk, nw, ntk, nm) in zip(docs, out)]
        log(f"[prepare] {sp}: {len(docs)} documents tokenised in {time.time() - t0:.0f}s")
    _W.pop("texts", None)
    pickle.dump(dict(spec=spec, splits=splits, labels=data["labels"], stats=data.get("stats")), open(cache_path, "wb"), protocol=4)
    return splits


def chunk_stats(splits, spec):
    rows = {}
    for sp, docs in splits.items():
        nc = np.asarray([len(d["chunks"]) for d in docs])
        nt = np.asarray([d["ntot"] for d in docs])
        nw = np.asarray([d["n_words"] for d in docs])
        ntok = np.asarray([d["n_tok"] for d in docs])
        nm = np.asarray([d["n_marks"] for d in docs])
        rows[sp] = dict(docs=len(docs), words_p50=int(np.median(nw)), words_p90=int(np.percentile(nw, 90)), words_max=int(nw.max()),
                        tok_per_word=round(float(ntok.sum() / max(1, nw.sum())), 3),
                        chunks_total_mean=round(float(nt.mean()), 2), chunks_kept_mean=round(float(nc.mean()), 2),
                        docs_fully_covered_pct=round(100 * float(np.mean(nt <= spec["max_store"])), 1),
                        markers_per_doc=round(float(nm.mean()), 1), docs_without_markers_pct=round(100 * float(np.mean(nm == 0)), 1),
                        labels_per_doc=round(float(np.mean([len(d["y"]) for d in docs])), 2) if docs and docs[0]["y"] is not None else "-")
    return rows


# ============================================================== views (which stored chunks a prediction looks at)
VIEWS = {
    "all": lambda d: list(range(len(d["chunks"]))),
    "first1": lambda d: [0],
    "last1": lambda d: [len(d["chunks"]) - 1],
    "head4": lambda d: select_chunks(len(d["chunks"]), d["mk"], 4, "head"),
    "tail4": lambda d: select_chunks(len(d["chunks"]), d["mk"], 4, "tail"),
    "headtail4": lambda d: select_chunks(len(d["chunks"]), d["mk"], 4, "headtail"),
    "uniform4": lambda d: select_chunks(len(d["chunks"]), d["mk"], 4, "uniform"),
    "marker4": lambda d: select_chunks(len(d["chunks"]), d["mk"], 4, "marker"),
}


# ============================================================== metrics + thresholds (multi-label)
GRID = np.round(np.arange(0.02, 0.9001, 0.01), 2)


def decide(p, thr):
    """Binary predictions; every document gets at least one statute (the one most above its threshold)."""
    thr = np.asarray(thr, np.float32)
    pred = p >= thr
    empty = ~pred.any(1)
    if empty.any():
        te = thr[empty] if thr.ndim == 2 else thr
        j = (p[empty] / np.maximum(te, 1e-6)).argmax(1)
        pred[np.where(empty)[0], j] = True
    return pred


def _counts(y, pred):
    tp = (y & pred).sum(0).astype(np.float64)
    fp = (~y & pred).sum(0).astype(np.float64)
    fn = (y & ~pred).sum(0).astype(np.float64)
    return tp, fp, fn


def ml_metrics(y, pred):
    """Macro-F1 over the statutes present in the reference (like an evaluator that derives the label set from the
    reference), Micro-F1, subset accuracy, sample-F1, labels per document."""
    tp, fp, fn = _counts(y, pred)
    sup = y.sum(0) > 0
    den = 2 * tp + fp + fn
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    P = np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0)
    R = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    inter = (y & pred).sum(1)
    return dict(macro_f1=float(f1[sup].mean()) if sup.any() else 0.0, macro_p=float(P[sup].mean()) if sup.any() else 0.0,
                macro_r=float(R[sup].mean()) if sup.any() else 0.0,
                micro_f1=float(2 * tp.sum() / max(1.0, 2 * tp.sum() + fp.sum() + fn.sum())),
                micro_p=float(tp.sum() / max(1.0, tp.sum() + fp.sum())), micro_r=float(tp.sum() / max(1.0, tp.sum() + fn.sum())),
                subset_acc=float((y == pred).all(1).mean()),
                sample_f1=float(np.mean(2 * inter / np.maximum(1, y.sum(1) + pred.sum(1)))),
                labels_per_doc_pred=float(pred.sum(1).mean()), labels_per_doc_gold=float(y.sum(1).mean()))


def tune_global(y, p):
    best = None
    for t in GRID:
        f = ml_metrics(y, decide(p, t))["macro_f1"]
        key = (round(f, 6), -abs(t - 0.3))
        if best is None or key > best[0]:
            best = (key, float(t), f)
    return best[1], best[2]


def tune_per_label(y, p, t_glob, min_pos=30):
    """Per-statute threshold maximising that statute's F1 (ties -> closest to the global threshold);
    statutes with fewer than `min_pos` positives keep the global threshold."""
    Fm = np.zeros((len(GRID), y.shape[1]))
    for i, t in enumerate(GRID):
        tp, fp, fn = _counts(y, p >= t)
        den = 2 * tp + fp + fn
        Fm[i] = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    score = Fm - 1e-9 * np.abs(GRID - t_glob)[:, None]
    thr = GRID[score.argmax(0)].astype(np.float32)
    thr[y.sum(0) < min_pos] = t_glob
    return thr


def crossfit(y, p, min_pos=30, seed=0):
    """Honest dev estimate of both threshold strategies: tune on one half of dev, score on the other, swap, average."""
    idx = np.random.default_rng(seed).permutation(len(y))
    halves = [idx[: len(y) // 2], idx[len(y) // 2:]]
    g, l = [], []
    for a, b in [(0, 1), (1, 0)]:
        A, B = halves[a], halves[b]
        tg, _ = tune_global(y[A], p[A])
        tl = tune_per_label(y[A], p[A], tg, min_pos)
        g.append(ml_metrics(y[B], decide(p[B], tg))["macro_f1"])
        l.append(ml_metrics(y[B], decide(p[B], tl))["macro_f1"])
    return float(np.mean(g)), float(np.mean(l))


def choose_thresholds(y, p, min_pos=30):
    tg, fg = tune_global(y, p)
    tl = tune_per_label(y, p, tg, min_pos)
    cg, cl = crossfit(y, p, min_pos)
    mode = "per_label" if cl >= cg else "global"
    thr = tl if mode == "per_label" else np.full(y.shape[1], tg, np.float32)
    return dict(mode=mode, thr=thr, t_global=tg, crossfit_global=cg, crossfit_per_label=cl,
                insample_global=fg, insample_per_label=ml_metrics(y, decide(p, tl))["macro_f1"])


def analyze(y, pred, labels, train_docs_per_label, n_words, top=12):
    """Per-statute F1, rare-vs-frequent buckets, most missed / most spurious statutes, confused pairs, doc length."""
    tp, fp, fn = _counts(y, pred)
    den = 2 * tp + fp + fn
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    sup = y.sum(0)
    trf = np.asarray([train_docs_per_label[i] for i in range(len(labels))])
    per = {str(l): dict(f1=round(float(f1[i]), 4), support=int(sup[i]), train_docs=int(trf[i]), tp=int(tp[i]), fp=int(fp[i]), fn=int(fn[i]))
           for i, l in enumerate(labels)}
    buckets = {}
    for name, lo, hi in [("<500 train docs", 0, 500), ("500-1k", 500, 1000), ("1k-3k", 1000, 3000), (">=3k", 3000, 10 ** 9)]:
        m = (trf >= lo) & (trf < hi) & (sup > 0)
        if m.any():
            buckets[name] = dict(n_statutes=int(m.sum()), macro_f1=round(float(f1[m].mean()), 4))
    conf = Counter()
    for r in range(len(y)):
        a, b = np.where(y[r] & ~pred[r])[0], np.where(~y[r] & pred[r])[0]
        if len(a) and len(b):
            for i in a:
                for j in b:
                    conf[(labels[i], labels[j])] += 1
    order = np.argsort(f1)
    nw = np.asarray(n_words)
    by_len = {}
    for name, lo, hi in [("<300 words", 0, 300), ("300-1k", 300, 1000), ("1k-3k", 1000, 3000), (">=3k", 3000, 10 ** 9)]:
        m = (nw >= lo) & (nw < hi)
        if m.any():
            mm = ml_metrics(y[m], pred[m])
            by_len[name] = dict(docs=int(m.sum()), macro_f1=round(mm["macro_f1"], 4), micro_f1=round(mm["micro_f1"], 4))
    by_count = {}
    k = y.sum(1)
    for name, lo, hi in [("1 statute", 1, 2), ("2-3", 2, 4), ("4-6", 4, 7), (">=7", 7, 10 ** 9)]:
        m = (k >= lo) & (k < hi)
        if m.any():
            mm = ml_metrics(y[m], pred[m])
            by_count[name] = dict(docs=int(m.sum()), micro_f1=round(mm["micro_f1"], 4), pred_per_doc=round(mm["labels_per_doc_pred"], 2))
    return dict(per_label=per, by_train_freq=buckets,
                worst=[dict(label=labels[i], f1=round(float(f1[i]), 4), support=int(sup[i]), train_docs=int(trf[i])) for i in order[:top]],
                best=[dict(label=labels[i], f1=round(float(f1[i]), 4), support=int(sup[i])) for i in order[::-1][:5]],
                most_missed=[dict(label=labels[i], fn=int(fn[i]), support=int(sup[i])) for i in np.argsort(-fn)[:top]],
                most_spurious=[dict(label=labels[i], fp=int(fp[i])) for i in np.argsort(-fp)[:top]],
                confused_pairs=[dict(gold_missed=a, predicted_instead=b, docs=c) for (a, b), c in conf.most_common(top)],
                by_doc_length=by_len, by_n_gold_statutes=by_count,
                corr_f1_vs_log_train_freq=round(float(np.corrcoef(f1, np.log(np.maximum(trf, 1)))[0, 1]), 3) if len(labels) > 2 else None)


# ============================================================== backbone wrappers
class EncTaggerBOS(P2.EncTagger):
    """Encoder-decoder v2 used BART-style, fed like its Phase-1 pretraining: encoder <- chunk,
    decoder <- [<s>] + chunk; returns the decoder states of the chunk tokens (the <s> state is dropped).
    Prompt / prefix tuning: the soft tokens go into the ENCODER input only (the decoder has 352 positions)."""

    def __init__(self, *a, bos_id=1, **kw):
        super().__init__(*a, **kw)
        self.bos_id = bos_id
        assert self.use_decoder

    def hidden(self, ids, attn_mask):
        pt = getattr(self, "_pt", None)
        B, dev = ids.size(0), ids.device
        if pt:
            P = pt["P"]
            ids_e = torch.cat([torch.zeros(B, P, dtype=ids.dtype, device=dev), ids], 1)
            am_e = torch.cat([torch.ones(B, P, dtype=torch.bool, device=dev), attn_mask], 1)
            pt["emb_on"] = pt["on"] = True
            try:
                mem = self.encode(ids_e, am_e)
            finally:
                pt["emb_on"] = pt["on"] = False
        else:
            mem, am_e = self.encode(ids, attn_mask), attn_mask
        dids = torch.cat([torch.full((B, 1), self.bos_id, dtype=ids.dtype, device=dev), ids], 1)
        dam = torch.cat([torch.ones(B, 1, dtype=torch.bool, device=dev), attn_mask], 1)
        T = dids.size(1)
        x = self.emb_drop(self.tok(dids) + self.dec_pos(torch.arange(T, device=dev))[None])
        cmask = am_e[:, None, None, :]
        smask = dam[:, None, None, :]
        if self.attn_mode == "causal":
            smask = smask & torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None, None]
        smask = smask | torch.eye(T, dtype=torch.bool, device=dev)[None, None]
        for l in self.dec_blocks:
            x = x + l.drop(l.self_attn(l.n1(x), mask=smask))
            x = x + l.drop(l.cross(l.n2(x), mem=mem, mask=cmask))
            x = x + l.drop(l.ffn(l.n3(x)))
        return self.dec_norm(x)[:, 1:]


def build_encoder(J, layers=None):
    """Phase-1 backbone wrapped by the (already verified) Phase-2 encoder class; only .hidden() is used."""
    base = P2.load_backbone(J)
    arch = J["arch"]
    cls = {"dec": P2.DecTagger, "encdec": P2.EncTagger, "encdec2": P2.ED2Tagger, "ret": P2.RetTagger}.get(arch, P2.TXLTagger)
    kw = dict(use_decoder=J.get("encdec_use_decoder", False)) if cls in (P2.EncTagger, P2.ED2Tagger) else {}
    if cls is P2.EncTagger and J.get("dec_bos") and kw["use_decoder"]:
        cls, kw = EncTaggerBOS, dict(kw, bos_id=J.get("bos_id", 1))
    enc = cls(base, 2, attn=J["attn"], layers=layers, dropout=J["dropout"], head="linear", **kw)
    enc.head = nn.Identity()                   # the Phase-2 token head is not used
    return enc


# ---- prompt / prefix tuning (soft tokens inserted after <s> if the chunks start with one)
def _token_embedding(enc, arch):
    return {"txl": lambda: enc.emb, "dec": lambda: enc.tok_emb, "ret": lambda: enc.emb, "encdec": lambda: enc.tok,
            "encdec2": lambda: enc.enc_emb.token_embedding}[arch]()


def _first_norms(enc, arch):
    """The norm in front of every layer's (self-)attention: its output at the prefix positions = the prefix that
    the layer turns into keys/values."""
    if arch == "encdec":
        return [b.n1 for b in enc.enc_blocks]
    if arch == "encdec2":
        return [b.norm1 for b in enc.enc_blocks]
    if arch == "dec":
        return [b.attn_norm for b in enc.blocks]
    return [b.ln1 for b in enc.blocks]          # txl, ret


def add_prompts(enc, J, deep):
    arch, P = J["arch"], int(J["n_prompt"])
    s = 1 if J.get("_bos") is not None else 0
    emb = _token_embedding(enc, arch)
    g = torch.Generator().manual_seed(J["seed"])
    rows = torch.randint(300, min(20000, emb.num_embeddings), (P,), generator=g)
    enc.prompt = nn.Parameter(emb.weight.detach()[rows].clone().float())       # Lester et al.: init from real tokens
    enc._pt = dict(on=False, emb_on=False, s=s, P=P)
    new = [enc.prompt]

    def emb_hook(mod, inp, out):
        pt = enc._pt
        if not pt["emb_on"]:
            return out
        pt["emb_on"] = False                    # one-shot: the enc-dec decoder re-uses the same embedding
        B = out.size(0)
        return torch.cat([out[:, :s], enc.prompt.to(out.dtype)[None].expand(B, -1, -1), out[:, s + P:]], 1)

    emb.register_forward_hook(emb_hook)
    if deep:
        norms = _first_norms(enc, arch)
        enc.prefix = nn.ParameterList()
        for nm in norms:
            w = getattr(nm, "weight", None)
            scale = w.detach().abs().mean().item() if w is not None else 1.0
            enc.prefix.append(nn.Parameter(torch.randn(P, emb.embedding_dim, generator=g) * scale))

        def make_hook(i):
            def hook(mod, inp, out):
                if not enc._pt["on"] or out.dim() != 3:
                    return out
                B = out.size(0)
                return torch.cat([out[:, :s], enc.prefix[i].to(out.dtype)[None].expand(B, -1, -1), out[:, s + P:]], 1)
            return hook

        for i, nm in enumerate(norms):
            nm.register_forward_hook(make_hook(i))
        new += list(enc.prefix)
    if arch != "encdec":                        # (EncTaggerBOS handles the prompts itself)
        orig = enc.hidden

        def hidden(ids, am):
            B = ids.size(0)
            ph = torch.zeros(B, P, dtype=ids.dtype, device=ids.device)          # placeholders, replaced by the hooks
            ids2 = torch.cat([ids[:, :s], ph, ids[:, s:]], 1)
            am2 = torch.cat([am[:, :s], torch.ones(B, P, dtype=torch.bool, device=am.device), am[:, s:]], 1)
            enc._pt["emb_on"] = enc._pt["on"] = True
            try:
                h = orig(ids2, am2)
            finally:
                enc._pt["emb_on"] = enc._pt["on"] = False
            return torch.cat([h[:, :s], h[:, s + P:]], 1)

        enc.hidden = hidden
    return new


# ---- QLoRA: NF4 4-bit storage of the frozen weights
NF4_LEVELS = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
              -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
              0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
              0.7229568362236023, 1.0]


class NF4Linear(nn.Module):
    """Frozen nn.Linear stored as 4-bit NormalFloat (QLoRA, Dettmers et al. 2023): blocks of `block` weights share one
    absmax scale; the scales are themselves 8-bit quantised in groups of 256 (double quantisation).
    forward() de-quantises to the compute dtype on the fly."""

    def __init__(self, lin: nn.Linear, block=64, block2=256):
        super().__init__()
        W = lin.weight.detach().float().cpu()
        self.in_features, self.out_features, self.block = lin.in_features, lin.out_features, block
        self.n = W.numel()
        flat = torch.cat([W.reshape(-1), W.new_zeros((-self.n) % block)]).view(-1, block)
        amax = flat.abs().amax(1).clamp(min=1e-12)
        lv = torch.tensor(NF4_LEVELS)
        codes = torch.bucketize((flat / amax[:, None]).clamp(-1, 1), (lv[1:] + lv[:-1]) / 2).to(torch.uint8)
        self.register_buffer("qweight", (codes[:, 0::2] << 4) | codes[:, 1::2])
        self.nb = amax.numel()
        a = torch.cat([amax, amax.new_zeros((-self.nb) % block2)]).view(-1, block2)
        s2 = a.amax(1).clamp(min=1e-12) / 255.0
        self.register_buffer("absmax_q", torch.round(a / s2[:, None]).to(torch.uint8))
        self.register_buffer("absmax_s2", s2)
        self.register_buffer("levels", lv, persistent=False)
        self.bias = nn.Parameter(lin.bias.detach().clone(), requires_grad=False) if lin.bias is not None else None

    def dequant(self, dtype):
        q = self.qweight
        codes = torch.stack([(q >> 4), (q & 15)], -1).view(q.size(0), -1).long()
        amax = (self.absmax_q.float() * self.absmax_s2[:, None]).view(-1)[: self.nb]
        W = (self.levels.to(q.device)[codes] * amax[:, None]).view(-1)[: self.n]
        return W.view(self.out_features, self.in_features).to(dtype)

    def forward(self, x):
        dt = torch.float16 if (x.is_cuda and torch.is_autocast_enabled()) else x.dtype     # the notebook uses fp16 autocast
        b = None if self.bias is None else self.bias.to(dt)
        return F.linear(x.to(dt), self.dequant(dt), b)

    def nbytes(self):
        return self.qweight.numel() + self.absmax_q.numel() + 4 * self.absmax_s2.numel()


# ============================================================== document model
class LSIModel(nn.Module):
    def __init__(self, enc, J, n_labels, prior):
        super().__init__()
        self.enc, self.n_labels = enc, n_labels
        d = enc.d
        self.pos = nn.Embedding(16, d)           # relative position of the chunk in the document (16 buckets)
        self.flag = nn.Embedding(3, d)           # 1 = first chunk, 2 = last chunk, 0 = other
        nn.init.normal_(self.pos.weight, 0, 0.02); nn.init.normal_(self.flag.weight, 0, 0.02)
        self.drop = nn.Dropout(J["dropout"])
        self.head_type = J["head"]
        if J["doc_pool"] == "bigru":
            self.rnn = nn.GRU(d, J["rnn_hidden"], batch_first=True, bidirectional=True)
            dout = 2 * J["rnn_hidden"]
        else:
            self.rnn, dout = None, d
        prior = torch.as_tensor(prior, dtype=torch.float32).clamp(1e-4, 0.5)
        b0 = torch.log(prior / (1 - prior))
        if self.head_type == "lwan":             # label-wise attention: every statute looks for its own evidence chunk
            self.U = nn.Parameter(torch.randn(n_labels, dout) * 0.02)
            self.W = nn.Parameter(torch.randn(n_labels, dout) * 0.02)
            self.glob = nn.Linear(dout, n_labels)
            nn.init.zeros_(self.glob.weight)
            with torch.no_grad():
                self.glob.bias.copy_(b0)
        else:                                    # 'pool': one document vector (attention + max pooling) -> linear
            self.att = nn.Sequential(nn.Linear(dout, 128), nn.Tanh(), nn.Linear(128, 1))
            self.cls = nn.Linear(2 * dout, n_labels)
            with torch.no_grad():
                self.cls.bias.copy_(b0)
        self._amp, self._ckpt, self._ckpt_micro, self._n_overflow = True, bool(J.get("grad_ckpt")), J.get("ckpt_micro", 4), 0

    def chunk_vecs(self, ids, am):
        h = self.enc.hidden(ids, am)
        m = am.unsqueeze(-1).to(h.dtype)
        return (h * m).sum(1) / m.sum(1).clamp(min=1.0)

    def pool(self, vecs, pos, flag, lengths):
        """vecs [N, d] = the chunks of B documents, document after document -> logits [B, n_labels]."""
        with torch.autocast(device_type=vecs.device.type, enabled=False):
            x = self.drop(vecs.float() + self.pos(pos).float() + self.flag(flag).float())
            X = nn.utils.rnn.pad_sequence(list(torch.split(x, lengths)), batch_first=True)
            M = torch.zeros(X.shape[:2], dtype=torch.bool, device=x.device)
            for b, n in enumerate(lengths):
                M[b, :n] = True
            if self.rnn is not None:
                pk = nn.utils.rnn.pack_padded_sequence(X, torch.as_tensor(lengths, dtype=torch.long), batch_first=True,
                                                       enforce_sorted=False)
                H, _ = self.rnn(pk)
                H, _ = nn.utils.rnn.pad_packed_sequence(H, batch_first=True, total_length=X.size(1))
            else:
                H = X
            H = self.drop(H)
            mx = H.masked_fill(~M.unsqueeze(-1), -1e4).max(1).values
            if self.head_type == "lwan":
                S = torch.einsum("bnd,ld->bnl", H, self.U).masked_fill(~M.unsqueeze(-1), -1e4)
                A = torch.softmax(S, 1)
                ctx = torch.einsum("bnl,bnd->bld", A, H)
                return (ctx * self.W.unsqueeze(0)).sum(-1) + self.glob(mx)
            a = torch.softmax(self.att(H).squeeze(-1).masked_fill(~M, -1e4), -1)
            return self.cls(torch.cat([(a.unsqueeze(-1) * H).sum(1), mx], -1))


def encode_chunks(model, ids, am, log=None):
    """Chunk vectors [N, d] in fp16 autocast; if fp16 overflows (inf/NaN), recompute in fp32 (after 3 overflows the
    model stays in fp32). Optional gradient checkpointing per `ckpt_micro` chunks (enabled automatically on OOM)."""
    def fwd():
        if model._ckpt and torch.is_grad_enabled():
            m, outs = model._ckpt_micro, []
            for k in range(0, ids.size(0), m):
                outs.append(torch.utils.checkpoint.checkpoint(model.chunk_vecs, ids[k:k + m], am[k:k + m], use_reentrant=False))
            return torch.cat(outs)
        return model.chunk_vecs(ids, am)

    if ids.is_cuda and model._amp:
        with torch.autocast("cuda", dtype=torch.float16):
            v = fwd()
        if torch.isfinite(v).all():
            return v.float()
        model._n_overflow += 1
        if model._n_overflow >= 3:
            model._amp = False
            if log:
                log("fp16 overflow in the encoder 3x -> encoder runs in fp32 from now on")
    return fwd().float()


def collate_chunks(arrs):
    L = max(len(a) for a in arrs)
    ids = torch.zeros(len(arrs), L, dtype=torch.long)
    am = torch.zeros(len(arrs), L, dtype=torch.bool)
    for i, a in enumerate(arrs):
        ids[i, :len(a)] = torch.from_numpy(a.astype(np.int64))
        am[i, :len(a)] = True
    return ids, am


def pos_feats(docs, sels, device):
    pos, flag = [], []
    for d, sel in zip(docs, sels):
        n = max(1, int(d["ntot"]))
        for k in sel:
            ki = int(d["kidx"][k])
            pos.append(min(15, 16 * ki // n)); flag.append(1 if ki == 0 else (2 if ki == n - 1 else 0))
    t = lambda v: torch.as_tensor(v, dtype=torch.long, device=device)
    return t(pos), t(flag)


def forward_docs(model, docs, sels, device, log=None, feats=None, no_grad_enc=False):
    """-> (logits [B, L], real tokens encoded). feats = cached chunk vectors (frozen probe) instead of the encoder."""
    pos, flag = pos_feats(docs, sels, device)
    lengths = [len(s) for s in sels]
    if feats is not None:
        v = torch.cat([feats[d["_i"]][sel] for d, sel in zip(docs, sels)]).to(device).float()
        return model.pool(v, pos, flag, lengths), 0
    ids, am = collate_chunks([d["chunks"][k] for d, sel in zip(docs, sels) for k in sel])
    ids, am = ids.to(device), am.to(device)
    if no_grad_enc:
        with torch.no_grad():
            v = encode_chunks(model, ids, am, log)
    else:
        v = encode_chunks(model, ids, am, log)
    return model.pool(v, pos, flag, lengths), int(am.sum())


@torch.no_grad()
def encode_split(model, docs, device, tok_budget=32768, log=None):
    """Chunk vectors of every stored chunk (fp16, CPU), batched by length -> list of [n_chunks, d] per document."""
    model.eval()
    items = sorted(((i, j, len(c)) for i, d in enumerate(docs) for j, c in enumerate(d["chunks"])), key=lambda x: x[2])
    out = [torch.zeros(len(d["chunks"]), model.enc.d, dtype=torch.float16) for d in docs]
    k = 0
    while k < len(items):
        b = [items[k]]; k += 1
        while k < len(items) and (len(b) + 1) * items[k][2] <= tok_budget:
            b.append(items[k]); k += 1
        ids, am = collate_chunks([docs[i]["chunks"][j] for i, j, _ in b])
        v = encode_chunks(model, ids.to(device), am.to(device), log).half().cpu()
        for (i, j, _), vec in zip(b, v):
            out[i][j] = vec
    return out


@torch.no_grad()
def pool_probs(model, vecs, docs, device, view="all", bs=256):
    """P(statute) [N, L] from cached chunk vectors, looking only at the chunks of `view`."""
    model.eval()
    fn = VIEWS[view]
    out = np.zeros((len(docs), model.n_labels), np.float32)
    for s in range(0, len(docs), bs):
        idx = list(range(s, min(s + bs, len(docs))))
        sels = [fn(docs[i]) for i in idx]
        v = torch.cat([vecs[i][sel] for i, sel in zip(idx, sels)]).to(device).float()
        pos, flag = pos_feats([docs[i] for i in idx], sels, device)
        lg = model.pool(v, pos, flag, [len(x) for x in sels])
        out[s:s + len(idx)] = torch.sigmoid(lg.float()).cpu().numpy()
    return out


# ============================================================== adaptation methods
def _slots(enc, names):
    out = []
    for blk in enc.blocks:
        out += P2._linear_slots(blk, names)
    return out


def apply_method(model, method, J):
    """Freeze / wrap the encoder for the adaptation method. Returns (new PEFT parameters, body parameters that
    gradual unfreezing will switch on later)."""
    enc = model.enc
    for p in enc.parameters():
        p.requires_grad_(method in ("full", "kd"))
    new, later = [], []
    if method in ("lora", "lorma"):
        sl = _slots(enc, J["lowrank_targets"])
        assert sl, "no LoRA/LoRMA target found - check lowrank_targets"
        for holder, nm in sl:
            m = P2.LowRankLinear(getattr(holder, nm), J["rank"], J["alpha"], J["lowrank_dropout"], method)
            setattr(holder, nm, m)
            new += [m.A, m.B]
    elif method == "qlora":
        sl = _slots(enc, J.get("qlora_targets") or J["lowrank_targets"])
        assert sl, "no QLoRA target found"
        for holder, nm in sl:
            m = P2.LowRankLinear(NF4Linear(getattr(holder, nm), J.get("nf4_block", 64)), J["rank"], J["qlora_alpha"],
                                 J["lowrank_dropout"], "lora")
            setattr(holder, nm, m)
            new += [m.A, m.B]
    elif method in ("prompt", "prefix"):
        new = add_prompts(enc, J, deep=(method == "prefix"))
    elif method == "adapter":
        sl = _slots(enc, J["adapter_targets"])
        assert sl, "no adapter target found - check adapter_targets"
        for holder, nm in sl:
            m = P2.Adapter(getattr(holder, nm), J["adapter_dim"])
            setattr(holder, nm, m)
            new += list(m.down.parameters()) + list(m.up.parameters())
    elif method == "adalora":
        from peft import AdaLoraConfig
        from peft.tuners.adalora import AdaLoraModel
        total = J["_total_steps"]
        cfg = AdaLoraConfig(target_modules=J["lowrank_targets"], init_r=J["ada_init_r"], target_r=J["ada_target_r"],
                            lora_alpha=J["alpha"], lora_dropout=J["lowrank_dropout"], total_step=total,
                            tinit=int(0.1 * total), tfinal=int(0.2 * total), deltaT=max(1, int(0.02 * total)),
                            orth_reg_weight=J["ada_orth_reg"])
        ada = AdaLoraModel(enc, cfg, "default")           # injects the adapters into enc IN PLACE
        object.__setattr__(model, "_ada", ada)             # a handle (not a sub-module) for update_and_allocate
        for n, p in enc.named_parameters():
            if "lora_" in n:
                p.requires_grad_(True)
                new.append(p)
        assert new, "AdaLoRA found no target modules"
    elif method == "partial":
        for blk in list(enc.blocks)[-J["partial_top_layers"]:]:
            for p in blk.parameters():
                p.requires_grad_(True)
        for p in enc.final_norm.parameters():
            p.requires_grad_(True)
    elif method == "gradual":
        later = [p for p in enc.parameters()]             # all frozen now; unfrozen top-down during training
    elif method in ("frozen", "full", "kd"):
        pass
    else:
        raise ValueError(method)
    return new, later


def set_gradual_stage(model, n_top):
    blocks = list(model.enc.blocks)
    for i, blk in enumerate(blocks):
        on = i >= len(blocks) - n_top
        for p in blk.parameters():
            p.requires_grad_(on)
    for p in model.enc.final_norm.parameters():
        p.requires_grad_(n_top > 0)


def adalora_orth_reg(model):
    reg, n = 0.0, 0
    for name, p in model.enc.named_parameters():
        if ("lora_A" in name or "lora_B" in name) and p.requires_grad and p.dim() == 2:
            pc = p @ p.T if "lora_A" in name else p.T @ p
            reg = reg + torch.norm(pc - torch.eye(pc.size(0), device=p.device, dtype=pc.dtype), p="fro")
            n += 1
    return reg / max(n, 1)


def build_model(J, method, n_labels, prior):
    layers = J["kd_student_layers"] if method == "kd" else None
    enc = build_encoder(J, layers)
    p1_ids = {id(p) for p in enc.parameters()}            # Phase-1 weights (frozen ones are not saved per job)
    model = LSIModel(enc, J, n_labels, prior)
    new, later = apply_method(model, method, J)
    model._p1_ids = p1_ids
    return model, new, later


def count_params(model):
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================== training
def pick_chunks(n, k, rng):
    """k random stored chunks of a document (in document order); all of them if it has <= k."""
    if n <= k:
        return list(range(n))
    return sorted(rng.choice(n, size=k, replace=False).tolist())


def next_batch(order, docs, sel_fn, rng, tok_budget, max_docs, feats_mode):
    """Greedy batch: documents until the padded tokens (chunks x longest chunk) would exceed tok_budget."""
    batch, sels, n, mx = [], [], 0, 0
    while order:
        d = docs[order[-1]]
        sel = sel_fn(d, rng)
        ln = max(len(d["chunks"][k]) for k in sel)
        if batch and (len(batch) >= max_docs or (not feats_mode and (n + len(sel)) * max(mx, ln) > tok_budget)):
            break
        batch.append(order.pop()); sels.append(sel); n += len(sel); mx = max(mx, ln)
    return batch, sels


def multilabel_loss(lg, y, J, pw):
    kind = J["loss"]
    if kind == "bce":
        return F.binary_cross_entropy_with_logits(lg, y)
    if kind == "bce_pw":
        return F.binary_cross_entropy_with_logits(lg, y, pos_weight=pw)
    p = torch.sigmoid(lg)
    if kind == "focal":
        ce = F.binary_cross_entropy_with_logits(lg, y, reduction="none")
        pt = p * y + (1 - p) * (1 - y)
        return (ce * (1 - pt).pow(J["focal_gamma"])).mean()
    assert kind == "asl", kind                                   # Asymmetric Loss (Ridnik et al. 2021)
    pm = (p - J["asl_clip"]).clamp(min=0)
    lp = y * torch.log(p.clamp(min=1e-8)) * (1 - p).pow(J["asl_gamma_pos"])
    ln = (1 - y) * torch.log((1 - pm).clamp(min=1e-8)) * pm.pow(J["asl_gamma_neg"])
    return -(lp + ln).mean()


def train_one(J, model, new_params, later_params, train_docs, dev_docs, device, log, pw, labels, teacher=None,
              feats=None, dev_vecs=None, view=None):
    torch.manual_seed(J["seed"])
    rng = np.random.default_rng(J["seed"])
    model.to(device)
    pw = pw.to(device)
    Ytr = torch.as_tensor(multi_hot(train_docs, labels), dtype=torch.float32)
    Ydev = multi_hot(dev_docs, labels)
    for i, d in enumerate(train_docs):
        d["_i"] = i
    new_ids = {id(p) for p in new_params}
    head = [p for n, p in model.named_parameters() if not n.startswith("enc.")]
    peft_p = [p for p in model.enc.parameters() if id(p) in new_ids]
    body = [p for p in model.enc.parameters() if id(p) not in new_ids and (p.requires_grad or any(p is q for q in later_params))]
    groups = [dict(params=head, lr=J["head_lr"], weight_decay=J["weight_decay"])]
    if peft_p:
        groups.append(dict(params=peft_p, lr=J["peft_lr"], weight_decay=0.0 if J["method"] in ("prompt", "prefix") else J["weight_decay"]))
    if body:
        groups.append(dict(params=body, lr=J["lr"], weight_decay=J["weight_decay"]))
    opt = torch.optim.AdamW(groups)
    for g in opt.param_groups:
        g["base_lr"] = g["lr"]
    opt_ids = {id(p) for g in opt.param_groups for p in g["params"]}
    frozen_ids = {i for i in model._p1_ids if i not in opt_ids}          # Phase-1 weights this method never trains
    keep_keys = ({n for n, p in model.named_parameters(remove_duplicate=False) if id(p) not in frozen_ids}
                 | {n for n, _ in model.named_buffers()})                   # everything else (incl. AdaLoRA ranknum)
    model._keep_keys = keep_keys
    feats_mode = feats is not None
    enc_trainable = (not feats_mode) and (any(p.requires_grad for p in model.enc.parameters()) or bool(later_params))
    K = J["train_chunks"]
    sel_fn = (lambda d, r: pick_chunks(len(d["chunks"]), K, r)) if view is None else (lambda d, r: VIEWS[view](d))
    max_docs = J["probe_docs_per_step"] if feats_mode else J["max_docs_per_step"]
    sim, sizes = np.random.default_rng(0).permutation(len(train_docs)).tolist(), []
    while sim and len(sizes) < 300:                                         # expected documents per step
        sizes.append(len(next_batch(sim, train_docs, sel_fn, np.random.default_rng(len(sizes)), J["tok_budget"], max_docs, feats_mode)[0]))
    dps = float(np.mean(sizes))
    steps_per_epoch = math.ceil(len(train_docs) / dps)
    total = max(10, int(J["epochs"] * steps_per_epoch))
    planned = total
    warm = max(1, int(J["warmup"] * total))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", init_scale=2.0 ** 10, enabled=use_amp)
    ada = getattr(model, "_ada", None)
    if ada is not None:                                                     # AdaLoRA's rank schedule = this run's steps
        c = ada.peft_config["default"]
        c.total_step, c.tinit, c.tfinal, c.deltaT = total, int(0.1 * total), int(0.2 * total), max(1, int(0.02 * total))
    if use_amp:
        torch.cuda.reset_peak_memory_stats(device)
    best = dict(f1=-1.0, step=0, state=None, thr=0.3)
    hist, evals = [], []
    step, n_nan, n_skip, n_oom, tok_seen, docs_seen = 0, 0, 0, 0, 0, 0
    budget = J["time_budget_h"] * 3600
    t0 = time.time()
    t_cal, tok_cal = None, 0
    next_eval = [int(total * (i + 1) / J["n_evals"]) for i in range(J["n_evals"])]
    gstage = -1
    early_done, early_abort = False, False
    log(f"~{dps:.1f} documents per step -> {steps_per_epoch} steps per epoch | {J['epochs']} epochs = {total} steps planned")

    def lr_factor(s):
        return (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm))

    def do_eval(tag):
        te = time.time()
        vecs = dev_vecs if dev_vecs is not None else encode_split(model, dev_docs, device, J["infer_tok_budget"], log)
        probs = pool_probs(model, vecs, dev_docs, device, view or "all")
        thr, f1 = tune_global(Ydev, probs)
        m = ml_metrics(Ydev, decide(probs, thr))
        rec = dict(step=step, docs_seen=docs_seen, epoch=round(docs_seen / len(train_docs), 3), dev_macro_f1=round(f1, 4), thr=thr,
                   dev_micro_f1=round(m["micro_f1"], 4), pred_per_doc=round(m["labels_per_doc_pred"], 2),
                   finite=bool(np.isfinite(probs).all()), eval_secs=round(time.time() - te, 1), tag=tag)
        evals.append(rec)
        log("[EVAL] " + json.dumps(rec))
        if f1 > best["f1"] and np.isfinite(probs).all():
            best.update(f1=f1, step=step, thr=thr,
                        state={k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep_keys})
        model.train()

    order = []
    model.train()
    tl, tn = 0.0, 0
    while step < total:
        if not order:
            order = rng.permutation(len(train_docs)).tolist()
        idx, sels = next_batch(order, train_docs, sel_fn, rng, J["tok_budget"], max_docs, feats_mode)
        docs = [train_docs[i] for i in idx]
        if J["method"] == "gradual":                       # gradual unfreezing: (fraction of training, top blocks)
            st = max(i for i, (f, _) in enumerate(J["gradual_schedule"]) if step >= f * total)
            if st != gstage:
                gstage = st
                set_gradual_stage(model, J["gradual_schedule"][st][1])
                log(f"gradual unfreezing: step {step} -> top {J['gradual_schedule'][st][1]} blocks trainable")
        f = lr_factor(step)
        for g in opt.param_groups:
            g["lr"] = g["base_lr"] * f
        y = Ytr[idx].to(device)
        while True:
            try:
                lg, ntok = forward_docs(model, docs, sels, device, log, feats=feats, no_grad_enc=not enc_trainable)
                lg = lg.float()
                loss = multilabel_loss(lg, y, J, pw)
                if teacher is not None:
                    with torch.no_grad():
                        tlg, _ = forward_docs(teacher, docs, sels, device)
                    T = J["kd_temperature"]
                    kd = F.binary_cross_entropy_with_logits(lg / T, torch.sigmoid(tlg.float() / T)) * T * T
                    loss = J["kd_alpha"] * loss + (1 - J["kd_alpha"]) * kd
                if ada is not None and J["ada_orth_reg"] > 0:
                    loss = loss + J["ada_orth_reg"] * adalora_orth_reg(model)
                if not torch.isfinite(loss):
                    n_nan += 1
                    if n_nan <= 5 or n_nan % 100 == 0:
                        log(f"non-finite loss at step {step} (#{n_nan}) - batch skipped")
                    opt.zero_grad(set_to_none=True)
                    break
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                for g in opt.param_groups:
                    torch.nn.utils.clip_grad_norm_([p for p in g["params"] if p.grad is not None], 1.0)
                sc0 = scaler.get_scale() if use_amp else 1.0
                scaler.step(opt)
                scaler.update()
                if use_amp and scaler.get_scale() < sc0:
                    n_skip += 1
                if ada is not None:
                    ada.update_and_allocate(step)
                tl += loss.item(); tn += 1; tok_seen += ntok
                break
            except torch.cuda.OutOfMemoryError:
                n_oom += 1
                opt.zero_grad(set_to_none=True)
                lg = loss = None
                torch.cuda.empty_cache()
                if not model._ckpt:
                    model._ckpt = True
                    log(f"CUDA OOM at step {step} -> gradient checkpointing ON (every {model._ckpt_micro} chunks)")
                elif J["tok_budget"] > 2048:
                    J["tok_budget"] = int(J["tok_budget"] * 0.75)
                    log(f"CUDA OOM again -> token budget per step reduced to {J['tok_budget']}")
                    break
                else:
                    raise
        step += 1
        docs_seen += len(docs)
        # ---- time calibration: fit the number of steps into the time budget (the LR decay ends exactly there)
        if step == 10:
            t_cal, tok_cal = time.time(), tok_seen
        if step == J["calib_steps"] and t_cal is not None:
            sps = (time.time() - t_cal) / (step - 10)
            if feats_mode:
                eval_s, final_s = 3.0, 10.0
            else:
                tps = max(1.0, (tok_seen - tok_cal) / max(1e-6, time.time() - t_cal))
                speed = 2.5 if enc_trainable else 1.0               # forward-only inference is ~2.5-3x faster
                eval_s = J["_n_eval_tokens"] / (speed * tps)
                final_s = J["_n_final_tokens"] / (speed * tps)
            reserve = J["n_evals"] * eval_s + final_s + (15 if feats_mode else 180)   # cached-feature heads: no big final pass
            fit = step + int((budget - (time.time() - t0) - reserve) / sps)
            total = max(step + 20, min(planned, fit))
            warm = min(warm, max(1, int(J["warmup"] * total)))
            next_eval = [max(step + 1, int(total * (i + 1) / J["n_evals"])) for i in range(J["n_evals"])]
            if ada is not None:                                       # keep AdaLoRA's rank schedule consistent
                c = ada.peft_config["default"]
                c.total_step, c.tinit, c.tfinal, c.deltaT = total, int(0.1 * total), int(0.2 * total), max(1, int(0.02 * total))
            log(f"calibration: {sps:.3f} s/step (~{dps:.1f} docs) | ~{eval_s / 60:.1f} min per dev-subset eval, ~{final_s / 60:.1f} min "
                f"for full dev + test | steps {planned} planned -> {total} fit the {J['time_budget_h']:.2f} h budget "
                f"(= {total * dps / len(train_docs):.2f} epochs of {len(train_docs)} documents)")
        if step % J["log_every"] == 0:
            el = time.time() - t0
            log(json.dumps(dict(step=step, total=total, loss=round(tl / max(tn, 1), 5), lr_f=round(f, 4),
                                tok_s=int(tok_seen / max(el, 1e-6)), min=round(el / 60, 1), nan=n_nan, skip=n_skip,
                                fp32=not model._amp, ckpt=model._ckpt)))
            hist.append(dict(step=step, loss=tl / max(tn, 1), min=el / 60))
            tl, tn = 0.0, 0
        if J.get("early_check") and not early_done and step > J["calib_steps"] and step >= J["early_check"] * total:
            early_done = True                              # a run that learnt nothing by now will not recover:
            do_eval("early-check")                         # stop it and free the GPU instead of wasting hours
            if best["f1"] < J.get("early_min_f1", 0.08) or evals[-1]["pred_per_doc"] > J.get("early_max_pred_per_doc", 30):
                log(f"!!! dev macro-F1 {best['f1']:.4f} (min {J.get('early_min_f1', 0.08)}), {evals[-1]['pred_per_doc']} statutes/doc after {step} steps "
                    f"({100 * step / total:.0f}% of training) -> collapsed early, stopping this job")
                early_abort = True
                break
        if step in next_eval or step >= total:
            do_eval("scheduled")
            next_eval = [s for s in next_eval if s > step]
        elif time.time() - t0 > 1.08 * budget:
            log(f"time budget exceeded at step {step}/{total} -> final eval and stop")
            do_eval("time-stop")
            break
    if best["state"] is None:                              # nothing finite was ever evaluated
        best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep_keys}
    info = dict(train_secs=time.time() - t0, steps=step, planned_steps=planned, docs_seen=docs_seen,
                epochs=docs_seen / len(train_docs), tok_per_s=tok_seen / max(time.time() - t0, 1e-6), docs_per_step=dps,
                nan_batches=n_nan, skipped_steps=n_skip, ooms=n_oom, fp32=not model._amp, grad_ckpt=model._ckpt,
                early_abort=early_abort, history=hist, evals=evals,
                peak_mem_gb=(torch.cuda.max_memory_allocated(device) / 2 ** 30) if use_amp else None)
    return best, info


# ============================================================== one experiment (script entry point)
def write_zip(pred, path):
    path = Path(path)
    pj = path.with_suffix(".json")
    json.dump(pred, open(pj, "w", encoding="utf-8"))
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    os.remove(pj)
    return path


def check_prediction(pred, test_docs, labels):
    assert isinstance(pred, list) and len(pred) == len(test_docs), "one prediction per test document"
    assert [p["id"] for p in pred] == [d["id"] for d in test_docs], "ids must be exactly the test ids (same order)"
    valid = set(labels)
    for p in pred:
        v = p["pred"]
        assert isinstance(v, list) and v, f"{p['id']}: pred must be a non-empty list"
        assert all(type(x) is int for x in v), f"{p['id']}: statute ids must be Python ints"
        assert len(set(v)) == len(v), f"{p['id']}: duplicate statute ids"
        assert set(v) <= valid, f"{p['id']}: unknown statute ids {set(v) - valid}"


def to_prediction(test_docs, pred, labels):
    return [dict(id=d["id"], pred=[int(labels[j]) for j in np.where(row)[0]]) for d, row in zip(test_docs, pred)]


def subsample(docs, n, seed):
    if n is None or len(docs) <= n:
        return docs
    idx = np.random.default_rng(seed).choice(len(docs), n, replace=False)
    return [docs[i] for i in sorted(idx)]


PROBE_ABLATION = [                                # cheap on cached frozen features: one change at a time
    ("loss=bce (no pos_weight)", dict(loss="bce")), ("loss=asl", dict(loss="asl")), ("loss=focal", dict(loss="focal")),
    ("head=pool (no label-wise attention)", dict(head="pool")),
    ("doc=first1", dict(view="first1")), ("doc=head4", dict(view="head4")), ("doc=headtail4", dict(view="headtail4")),
    ("doc=uniform4", dict(view="uniform4")), ("doc=marker4", dict(view="marker4")),
]


def run_job(job_path):
    J = json.load(open(job_path))
    out = Path(J["out_dir"]); out.mkdir(parents=True, exist_ok=True)
    logf = open(out / "log.txt", "a")

    def log(msg):
        line = f"[{time.strftime('%H:%M:%S')}][{J['name']}] {msg}"
        print(line, flush=True); logf.write(line + "\n"); logf.flush()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    D = pickle.load(open(J["cache"], "rb"))
    S, labels = D["splits"], D["labels"]
    L = len(labels)
    J["_bos"] = D["spec"].get("bos")
    train, dev, test = S["train"], S["dev"], S["test"]
    if J.get("quick"):
        q = J["quick_docs"]
        train, dev, test = subsample(train, q, 1), subsample(dev, max(40, q // 2), 2), subsample(test, max(30, q // 3), 3)
    sub_idx = sorted(np.random.default_rng(5).choice(len(dev), min(len(dev), J["eval_dev_docs"]), replace=False).tolist())
    dev_sub = [dev[i] for i in sub_idx]
    ntok = lambda docs: sum(len(c) for d in docs for c in d["chunks"])
    J["_n_eval_tokens"], J["_n_final_tokens"] = ntok(dev_sub), ntok(dev) + ntok(test)
    ytr, ydev = multi_hot(train, labels), multi_hot(dev, labels)
    freq = ytr.mean(0)
    pw = torch.as_tensor(np.clip(np.sqrt((1 - freq) / np.maximum(freq, 1e-4)), 1.0, J["pw_max"]), dtype=torch.float32)
    prior = freq if J["loss"] != "bce_pw" else freq * pw.numpy() / (freq * pw.numpy() + 1 - freq)
    method = J["method"]
    log(f"arch={J['arch']} method={method} head={J['head']} loss={J['loss']} attn={J['attn']} device={device} | "
        f"train {len(train)} dev {len(dev)} (subset {len(dev_sub)} for evals during training) test {len(test)} | "
        f"{L} statutes | budget {J['time_budget_h']:.2f} h")
    J["_total_steps"] = max(10, int(J["epochs"] * len(train) / 4))
    R = dict(name=J["name"], method=method, config={k: v for k, v in J.items() if not k.startswith("_")})

    teacher = None
    if method == "kd":                             # teacher = the best HEALTHY fine-tuned full-depth model
        cands = J["kd_teacher_dir"] if isinstance(J["kd_teacher_dir"], list) else [J["kd_teacher_dir"]]
        ok = []
        for c in map(Path, cands):
            if (c / "val_model.pt").exists() and (c / "results.json").exists():
                rc = json.load(open(c / "results.json"))
                f1 = rc.get("dev", {}).get("macro_f1", 0.0)
                log(f"teacher candidate {c.name}: dev macro-F1 {f1:.4f}, collapsed={rc.get('collapsed')}")
                if (J.get("quick") or not rc.get("collapsed")) and f1 >= J.get("kd_min_teacher_f1", 0.15):
                    ok.append((f1, str(c)))
        assert ok, f"KD needs a healthy teacher checkpoint in one of {cands} (teacher job with save_model=True)"
        tdir = Path(max(ok)[1])
        tj = json.load(open(tdir / "job.json"))
        tj["_total_steps"], tj["_bos"] = 1000, J["_bos"]
        teacher, _, _ = build_model(tj, tj["method"], L, prior if tj["loss"] == J["loss"] else freq)
        missing, unexpected = teacher.load_state_dict(torch.load(tdir / "val_model.pt", map_location="cpu"), strict=False)
        assert not unexpected, f"teacher checkpoint has unexpected keys {unexpected[:5]}"
        teacher.to(device).eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
        tres = json.load(open(tdir / "results.json"))
        R["teacher"] = dict(name=tj["name"], method=tj["method"], params=count_params(teacher)[0], dev_f1=tres.get("dev", {}).get("macro_f1"),
                            ms_per_doc=tres.get("test_infer_ms_per_doc"))
        log(f"teacher {tj['name']} ({tj['method']}) {R['teacher']['params'] / 1e6:.2f}M params, dev macro-F1 {R['teacher']['dev_f1']}")

    model, new, later = build_model(J, method, L, prior)
    R["params_total"] = count_params(model)[0]
    R["params_trainable"] = sum(p.numel() for p in model.parameters() if p.requires_grad or any(p is q for q in later))
    R["params_encoder"] = sum(p.numel() for p in model.enc.parameters())
    if method == "qlora":
        nf = [m for m in model.modules() if isinstance(m, NF4Linear)]
        R["qlora"] = dict(quantised_layers=len(nf), weights=sum(m.n for m in nf), nf4_bytes=sum(m.nbytes() for m in nf),
                          fp32_bytes=4 * sum(m.n for m in nf), fp16_bytes=2 * sum(m.n for m in nf))
        log(f"QLoRA: {len(nf)} projections in NF4: {R['qlora']['nf4_bytes'] / 2 ** 20:.1f} MB instead of "
            f"{R['qlora']['fp16_bytes'] / 2 ** 20:.1f} MB (fp16) / {R['qlora']['fp32_bytes'] / 2 ** 20:.1f} MB (fp32)")
        R["params_total"] += R["qlora"]["weights"]; R["params_encoder"] += R["qlora"]["weights"]
    log(f"params total={R['params_total'] / 1e6:.2f}M (encoder {R['params_encoder'] / 1e6:.2f}M) "
        f"trainable={R['params_trainable'] / 1e6:.3f}M | PEFT tensors {len(new)}")

    if method == "frozen":                         # probe: encode every stored chunk ONCE, then train heads on the cache
        model.to(device)
        t0 = time.time()
        feats = {sp: encode_split(model, docs, device, J["infer_tok_budget"], log) for sp, docs in (("train", train), ("dev", dev), ("test", test))}
        R["feature_secs"] = time.time() - t0
        log(f"frozen features of {sum(len(v) for v in feats.values())} documents cached in {R['feature_secs'] / 60:.1f} min")
        sub_vecs = [feats["dev"][i] for i in sub_idx]
        PJ = dict(J, epochs=J["probe_epochs"], time_budget_h=max(0.1, 0.6 * (J["time_budget_h"] - R["feature_secs"] / 3600)))
        best, info = train_one(PJ, model, new, later, train, dev_sub, device, log, pw, labels, feats=feats["train"], dev_vecs=sub_vecs)
        vec_dev, vec_test = feats["dev"], feats["test"]
        test_enc_secs = R["feature_secs"] * ntok(test) / max(1, ntok(train) + ntok(dev) + ntok(test))
        if J.get("probe_ablation"):
            ab = {}
            for name, ov in PROBE_ABLATION:
                view = ov.get("view")
                J2 = dict(J, **{k: v for k, v in ov.items() if k != "view"}, epochs=J["probe_ablation_epochs"], n_evals=2, early_check=0,
                          time_budget_h=0.05)                    # <= 3 min each (normally ~1 min: heads on cached vectors)
                pr2 = freq if J2["loss"] != "bce_pw" else prior
                m2 = LSIModel(model.enc, J2, L, pr2).to(device)
                m2._p1_ids = model._p1_ids
                b2, i2 = train_one(J2, m2, [], [], train, dev_sub, device, lambda s: None, pw, labels, feats=feats["train"],
                                   dev_vecs=sub_vecs, view=view)
                m2.load_state_dict(b2["state"], strict=False)
                p2 = pool_probs(m2, vec_dev, dev, device, view or "all")
                ch = choose_thresholds(ydev, p2, J["thr_min_pos"])
                mm = ml_metrics(ydev, decide(p2, ch["thr"]))
                ab[name] = dict(dev_macro_f1=round(ch["insample_per_label" if ch["mode"] == "per_label" else "insample_global"], 4),
                                crossfit_global=round(ch["crossfit_global"], 4), crossfit_per_label=round(ch["crossfit_per_label"], 4),
                                micro_f1=round(mm["micro_f1"], 4), epochs=round(i2["epochs"], 2), secs=round(i2["train_secs"], 1))
                log(f"[probe ablation] {name}: {json.dumps(ab[name])}")
                del m2
            R["probe_ablation"] = ab
    else:
        best, info = train_one(J, model, new, later, train, dev_sub, device, log, pw, labels, teacher=teacher)
        del teacher
        model.load_state_dict(best["state"], strict=False)
        model.to(device).eval()
        t0 = time.time()
        vec_dev = encode_split(model, dev, device, J["infer_tok_budget"], log)
        t1 = time.time()
        vec_test = encode_split(model, test, device, J["infer_tok_budget"], log)
        test_enc_secs = time.time() - t1
        log(f"encoded full dev in {(t1 - t0) / 60:.1f} min, test in {test_enc_secs / 60:.1f} min")
    R["train"] = info
    if J["save_model"]:
        torch.save(best["state"], out / "val_model.pt")
    if method == "gradual":                        # report what was trainable at the end of gradual unfreezing
        R["params_trainable"] = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # ---- dev: which chunks to look at (views) and which threshold strategy (global vs per statute, cross-fitted)
    views = {}
    for v in VIEWS:
        pv = pool_probs(model, vec_dev, dev, device, v)
        tg, fg = tune_global(ydev, pv)
        views[v] = dict(probs=pv, macro_f1_global_thr=round(fg, 4), thr=tg)
    R["views"] = {v: dict(macro_f1_global_thr=x["macro_f1_global_thr"], thr=x["thr"]) for v, x in views.items()}
    log("VIEWS (dev macro-F1 at a tuned global threshold) " + json.dumps({v: x["macro_f1_global_thr"] for v, x in views.items()}))
    cand = {v: choose_thresholds(ydev, views[v]["probs"], J["thr_min_pos"]) for v in J["infer_views"]}
    pick = max(cand, key=lambda v: (round(max(cand[v]["crossfit_global"], cand[v]["crossfit_per_label"]), 6), v == "all"))
    ch, p = cand[pick], views[pick]["probs"]
    pred_dev = decide(p, ch["thr"])
    R["infer_view"] = pick
    R["thresholds"] = dict(mode=ch["mode"], t_global=ch["t_global"], per_label={str(l): float(t) for l, t in zip(labels, ch["thr"])},
                           crossfit_global=ch["crossfit_global"], crossfit_per_label=ch["crossfit_per_label"],
                           insample_global=ch["insample_global"], insample_per_label=ch["insample_per_label"],
                           by_view={v: dict(crossfit_global=round(c["crossfit_global"], 4), crossfit_per_label=round(c["crossfit_per_label"], 4))
                                    for v, c in cand.items()})
    R["dev"] = ml_metrics(ydev, pred_dev)
    R["dev"]["macro_f1_crossfit"] = max(ch["crossfit_global"], ch["crossfit_per_label"])
    R["dev_at_0.5"] = ml_metrics(ydev, decide(p, 0.5))
    R["dev_top3"] = ml_metrics(ydev, decide(p, np.sort(p, 1)[:, -3:-2]))           # always the 3 most likely statutes
    R["best_step"] = best["step"]
    trf = [int(x) for x in ytr.sum(0)]
    R["analysis"] = analyze(ydev, pred_dev, labels, trf, [d["n_words"] for d in dev])
    np.savez_compressed(out / "dev_probs.npz", ids=np.asarray([str(d["id"]) for d in dev]), probs=p.astype(np.float16),
                        y=ydev.astype(np.uint8), labels=np.asarray(labels), thr=ch["thr"])
    log(f"DEV macro-F1 {R['dev']['macro_f1']:.4f} (cross-fitted {R['dev']['macro_f1_crossfit']:.4f}; global {ch['crossfit_global']:.4f} "
        f"vs per-statute {ch['crossfit_per_label']:.4f} -> {ch['mode']}) | micro-F1 {R['dev']['micro_f1']:.4f} | subset acc "
        f"{R['dev']['subset_acc']:.4f} | {R['dev']['labels_per_doc_pred']:.2f} statutes/doc predicted (gold {R['dev']['labels_per_doc_gold']:.2f}) "
        f"| view '{pick}' | at 0.5: {R['dev_at_0.5']['macro_f1']:.4f} | best step {best['step']} | by train frequency "
        f"{json.dumps(R['analysis']['by_train_freq'])}")

    t0 = time.time()
    probs = pool_probs(model, vec_test, test, device, pick)
    R["test_infer_secs"] = test_enc_secs + time.time() - t0
    R["test_infer_ms_per_doc"] = 1000 * R["test_infer_secs"] / max(1, len(test))
    np.savez_compressed(out / "test_probs.npz", ids=np.asarray([str(d["id"]) for d in test]), probs=probs.astype(np.float16),
                        labels=np.asarray(labels), thr=ch["thr"])
    pt = decide(probs, ch["thr"])
    pred = to_prediction(test, pt, labels)
    check_prediction(pred, test, labels)
    write_zip(pred, out / f"submission_{J['name']}.zip")
    k = pt.sum(1)
    R["test_pred"] = dict(statutes_per_doc=round(float(k.mean()), 3), max_per_doc=int(k.max()), distinct_statutes=int(pt.any(0).sum()),
                          docs_top1_fallback=int(((probs >= ch["thr"]).sum(1) == 0).sum()))
    R["collapsed"] = bool(info.get("early_abort") or not np.isfinite(probs).all() or R["dev"]["macro_f1"] < J.get("collapse_f1", 0.10)
                          or R["test_pred"]["distinct_statutes"] < min(10, L) or R["test_pred"]["statutes_per_doc"] > 15)
    json.dump(R, open(out / "results.json", "w"), indent=1, default=str)
    log(f"DONE -> submission_{J['name']}.zip | test: {R['test_pred']}"
        + (" | !!! COLLAPSED - do NOT upload" if R["collapsed"] else ""))


if __name__ == "__main__":
    run_job(sys.argv[1])


Writing /kaggle/working/p7/p5_lsi.py


In [4]:
%%writefile /kaggle/working/p7/p7_pcr.py
"""CS-781 Phase 7 - Prior Case Retrieval (PCR): for every query case, rank its split's candidate pool and return the
7 most relevant prior cases (Micro-F1@7), starting from my Phase-1 models.

One file, used two ways (same as Phases 2-6):
  * imported by the notebook  -> data checks, lexical retrieval (TF-IDF / BM25), temporal check, chunk cache
  * run as a script            -> `python p7_pcr.py job.json` trains / evaluates / predicts ONE experiment

Data facts that shaped the design (printed again by the notebook):
  * train 827 queries / 4,320 candidates, dev 118 / 1,023, test 237 / 1,727. EVERY query is also inside its own
    candidate pool -> a query is always removed from its own ranking (as is any pool document with the same text).
  * 6.4 (train) / 8.3 (dev) relevant cases per query on average, median 4-5, max 92. Exactly 7 ids must be returned,
    so even a perfect ranking scores only ~0.62-0.64 Micro-F1@7.
  * documents are long (median ~4.5k words, p90 ~13-15k, max 410k) and citations / names / organisations are
    masked (<CITATION>, <NAME>, <ORG>): the model must match legal CONTENT. The <CITATION> masks show where a case
    discusses precedents -> long documents keep the chunks with the most <CITATION> masks (+ first, last, spread).
  * TF-IDF (words + bigrams) already reaches ~0.40 dev Micro-F1@7.

System
  lexical   TF-IDF / BM25 grid (unigrams | + bigrams, stop words on/off, k1, b, query-term weighting) tuned on dev.
            Used (a) to mine hard negatives for training and (b) as a second score fused with the model score.
  model     bi-encoder on the Phase-1 backbone (shared for queries and candidates): exact Phase-1 tokenisation of the
            whole document -> chunks -> backbone (bidirectional) + adaptation method -> mean-pooled chunk vectors ->
            centred + projected (256-d) -> mean over the chunks -> L2-normalised document embedding; score = cosine.
            Chunk-level late interaction (MaxSim over the chunk embeddings) is also scored at evaluation time.
  training  InfoNCE: each query vs ONE of its relevant cases; every other document of the batch (the other queries,
            their positives, 2 BM25 hard negatives per query) is a negative; the query itself and its other relevant
            cases are masked. GradCache (Gao et al. 2021) makes 16-query / 64-document batches fit a T4: embeddings are
            computed without gradients, the loss gives d loss / d embedding, then each micro-batch is re-encoded with
            the same dropout masks (RNG restored) and back-propagated with that gradient.
  ranking   z-normalised model score fused with the z-normalised lexical score, s = w*model + (1-w)*lexical with
            w >= 0.1 (so the model always counts), over the whole pool or the lexical top-100; tuned on dev, with an
            honest 2-fold cross-fitted estimate. Optional temporal constraint (decided on dev in the notebook): a
            candidate that mentions a year after the query's judgment date cannot be a PRIOR case.

Adaptation methods (method=...), Phase-7 list: full, partial (layer-wise), QLoRA, LoRMA, prefix, prompt, IA3, KD.
  lorma    LoRMA_+ (Bihany, Patel & Modi 2025): y = (I + s*B A) W x on the attention projections   (compulsory)
  kd       knowledge distillation (compulsory): the best fine-tuned full-depth model (LoRMA or full) is the teacher,
           the student is a subset of the Phase-1 layers; loss = a * InfoNCE + (1-a) * T^2 * KL(teacher || student)
           over the SAME in-batch candidate list (the teacher's similarity distribution per query)
  qlora    frozen weights stored as 4-bit NormalFloat (block 64, double-quantised scales) + LoRA on every projection
  ia3      IA3 (Liu et al. 2022): learned vectors rescale keys, values and the FFN inner activation
  prompt   prompt tuning: trainable soft tokens prepended to every chunk
  prefix   prefix tuning (deep prompts): trainable prefix vectors at the input of every layer's attention
  partial  only the top `partial_top_layers` blocks (+ final norm) are trained
  full     full fine-tuning
  frozen   frozen Phase-1 encoder (a probe): chunk vectors are cached once; only the projection head is trained.
           Also runs the cheap ablations: zero-shot Phase-1 transfer, pooling, similarity functions, objectives,
           hard negatives, which chunks of a long document matter.
Pooling / similarity / objectives / negatives are analysed, but they do not count as adaptation techniques.
"""
import hashlib
import json
import math
import multiprocessing as mp
import os
import pickle
import re
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

K = 7                                        # the metric looks at the top 7 candidates of every query


# ============================================================== data
def _md5(t):
    return hashlib.md5(t.encode("utf-8", "ignore")).hexdigest()


def _items(d):
    if isinstance(d, dict):
        return [dict(id=k, **v) if isinstance(v, dict) else dict(id=k, text=v) for k, v in d.items()]
    return list(d)


def _text(x):
    t = x.get("text", x.get("document", ""))
    return " ".join(map(str, t)) if isinstance(t, list) else str(t or "")


def make_split(name, qids, pool, qkey, ckey, rel):
    """Index structures of one split. pool = candidate ids (order of the file); every query is ranked against it."""
    pidx = {c: i for i, c in enumerate(pool)}
    by_key = defaultdict(list)
    for i, c in enumerate(pool):
        by_key[ckey[c]].append(i)
    excl, gold, missing = [], [], 0
    for q in qids:
        e = set(by_key.get(qkey[q], []))                      # the query itself + any pool copy of its text
        if q in pidx:
            e.add(pidx[q])
        excl.append(sorted(e))
        if rel is not None:
            g = [pidx[c] for c in rel.get(q, []) if c in pidx]
            missing += len(rel.get(q, [])) - len(g)
            gold.append(sorted(set(g)))
    return dict(name=name, qids=list(qids), pool=list(pool), qkey=dict(qkey), ckey=dict(ckey), pidx=pidx,
                excl=excl, gold=gold if rel is not None else None, gold_missing_from_pool=missing)


def load_pcr(data_dir, log=print):
    """-> dict(texts={md5: text}, splits={train, dev, test}, stats). Test relevance is never read."""
    data_dir = Path(data_dir)
    texts, splits, stats = {}, {}, {}
    for sp, folder in (("train", "Train"), ("dev", "Dev"), ("test", "Test")):
        qs = _items(json.load(open(data_dir / folder / f"{sp}.json", encoding="utf-8")))
        cs = _items(json.load(open(data_dir / folder / f"{sp}_candidates.json", encoding="utf-8")))
        ckey, pool, dup_c = {}, [], 0
        for c in cs:
            cid, t = str(c["id"]), _text(c)
            if cid in ckey:
                dup_c += 1
                continue
            k = _md5(t)
            texts.setdefault(k, t)
            ckey[cid] = k
            pool.append(cid)
        qkey, qids, rel, dup_q = {}, [], {}, 0
        for q in qs:
            qid, t = str(q["id"]), _text(q)
            if qid in qkey:
                dup_q += 1
                continue
            k = _md5(t)
            texts.setdefault(k, t)
            qkey[qid] = k
            qids.append(qid)
            if sp != "test":                                  # test labels are never read (not even if present)
                rel[qid] = list(dict.fromkeys(str(x) for x in (q.get("relevant_candidates") or [])))
        S = make_split(sp, qids, pool, qkey, ckey, rel if sp != "test" else None)
        st = dict(queries=len(qids), pool=len(pool), duplicate_query_ids=dup_q, duplicate_candidate_ids=dup_c,
                  queries_in_own_pool=sum(q in S["pidx"] for q in qids),
                  pool_texts_unique=len(set(ckey.values())),
                  queries_with_pool_copies_of_their_text=sum(len(e) > (1 if q in S["pidx"] else 0) for q, e in zip(qids, S["excl"])))
        if S["gold"] is not None:
            ng = np.asarray([len(g) for g in S["gold"]])
            gold_is_query_text = sum(ckey[pool[i]] == qkey[q] for q, g in zip(qids, S["gold"]) for i in g)
            st.update(relevant_mean=round(float(ng.mean()), 2), relevant_p50=int(np.median(ng)), relevant_max=int(ng.max()),
                      queries_without_relevant=int((ng == 0).sum()), relevant_missing_from_pool=S["gold_missing_from_pool"],
                      relevant_with_the_query_text=int(gold_is_query_text),
                      f1_at_7_upper_bound=round(2 * float(np.minimum(ng, K).sum()) / max(1, K * len(ng) + ng.sum()), 4))
        stats[sp] = st
        splits[sp] = S
        log(f"[data] {sp}: {json.dumps(st)}")
    return dict(texts=texts, splits=splits, stats=stats)


def split_keys(S):
    """Unique document keys a split needs (pool first, then queries that are not in the pool)."""
    out = list(dict.fromkeys([S["ckey"][c] for c in S["pool"]] + [S["qkey"][q] for q in S["qids"]]))
    return out


# ---- years (temporal constraint: a prior case must be decided before the query)
DATE_RE = re.compile(r"DATE\s+OF\s+JUDGMENT\s*[:\-]?\s*(\d{1,2})\s*[/.\-]\s*(\d{1,2})\s*[/.\-]\s*(\d{4})", re.I)
_MON = r"(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\.?"
YEAR_RE = re.compile(r"(?<!\d)(1[89]\d\d|20[0-2]\d)(?!\d)")
YEAR_CTX_RE = re.compile(r"(?:\d{1,2}\s*[./\-]\s*\d{1,2}\s*[./\-]\s*|" + _MON + r"\s+\d{1,2}(?:st|nd|rd|th)?\s*,?\s*|"
                         r"\d{1,2}(?:st|nd|rd|th)?\s+(?:day\s+of\s+)?" + _MON + r"\s*,?\s*|\b(?:of|in|year|dated|since|till|until)\s+|,\s*)$",
                         re.I)


def doc_year(t):
    """(judgment year from a 'DATE OF JUDGMENT' header or None, latest year mentioned in a date-like context or None).
    Date-like context = a date (12.3.1975 / March 12, 1975 / 12th March 1975) or 'of 1975', 'in 1975', ', 1975' -
    so section numbers or amounts that happen to look like years are not taken as years."""
    m = DATE_RE.search(t[:4000])
    hy = int(m.group(3)) if m and 1850 <= int(m.group(3)) <= 2026 else None
    ys = [int(y.group(1)) for y in YEAR_RE.finditer(t) if YEAR_CTX_RE.search(t[max(0, y.start() - 30):y.start()])]
    ys = [y for y in ys if 1850 <= y <= 2026]
    return hy, (max(ys) if ys else None)


def best_year(y):
    return y[0] if y[0] is not None else y[1]


def temporal_violations(S, years, slack):
    """bool [n_queries, n_pool]: candidate's year > query's year + slack (both known) -> cannot be a PRIOR case."""
    qy = np.asarray([best_year(years[S["qkey"][q]]) or 9999 for q in S["qids"]], np.int32)
    cy = np.asarray([best_year(years[S["ckey"][c]]) or 0 for c in S["pool"]], np.int32)
    return cy[None, :] > (qy[:, None] + slack)


def temporal_report(S, years):
    out = {}
    hdr = np.mean([years[S["qkey"][q]][0] is not None for q in S["qids"]])
    cyk = np.mean([best_year(years[S["ckey"][c]]) is not None for c in S["pool"]])
    out["queries_with_date_header"] = round(float(hdr), 3)
    out["candidates_with_a_year"] = round(float(cyk), 3)
    if S["gold"] is not None:
        for slack in (0, 1, 2):
            V = temporal_violations(S, years, slack)
            g = [V[i, j] for i, gl in enumerate(S["gold"]) for j in gl]
            out[f"slack{slack}"] = dict(gold_pairs_violating=round(float(np.mean(g)) if g else 0.0, 4),
                                        all_pairs_violating=round(float(V.mean()), 4))
    return out


# ============================================================== metrics + ranking
def excl_mask(S):
    E = np.zeros((len(S["qids"]), len(S["pool"])), bool)
    for i, e in enumerate(S["excl"]):
        E[i, e] = True
    return E


def zrows(S, E):
    """Per-query z-normalisation over the candidates that can be returned."""
    X = np.where(E, np.nan, S.astype(np.float64))
    mu = np.nanmean(X, 1, keepdims=True)
    sd = np.nanstd(X, 1, keepdims=True)
    return np.nan_to_num((X - mu) / np.maximum(sd, 1e-9), nan=0.0)


def final_scores(Sm, Sl, w, E, T=None, cand=None):
    """Fused ranking score. Sm/Sl raw model / lexical scores (either may be None). T = temporal violations (pushed
    below everything else), cand = allowed candidate set (outside it pushed further down)."""
    if Sm is None:
        F = zrows(Sl, E)
    elif Sl is None or w >= 1.0:
        F = zrows(Sm, E)
    else:
        F = w * zrows(Sm, E) + (1 - w) * zrows(Sl, E)
    if T is not None:
        F = F - 100.0 * T
    if cand is not None:
        F = F - 1000.0 * (~cand)
    return np.where(E, -np.inf, F)


def topk(F, k=K):
    idx = np.argpartition(-F, k - 1, axis=1)[:, :k]
    o = np.argsort(-np.take_along_axis(F, idx, 1), 1, kind="stable")
    return np.take_along_axis(idx, o, 1)


def prf(top, gold, rows=None):
    rows = range(len(gold)) if rows is None else rows
    tp = sum(len(set(top[i].tolist()) & set(gold[i])) for i in rows)
    npred = K * len(rows)
    ngold = sum(len(gold[i]) for i in rows)
    P, R = tp / max(1, npred), tp / max(1, ngold)
    return dict(f1=2 * P * R / max(1e-12, P + R), p=P, r=R, tp=tp)


def rank_metrics(F, gold, ks=(7, 20, 50, 100)):
    """Recall@k (micro), MRR of the first relevant, mean rank of the relevant cases."""
    order = np.argsort(-F, 1, kind="stable")
    pos = np.empty_like(order)
    np.put_along_axis(pos, order, np.arange(F.shape[1])[None].repeat(F.shape[0], 0), 1)
    out, ranks, rr = {}, [], []
    for i, g in enumerate(gold):
        if g:
            r = pos[i, g] + 1
            ranks += r.tolist()
            rr.append(1.0 / r.min())
    ng = max(1, len(ranks))
    for k in ks:
        out[f"recall@{k}"] = round(sum(r <= k for r in ranks) / ng, 4)
    out["mrr"] = round(float(np.mean(rr)) if rr else 0.0, 4)
    out["median_gold_rank"] = int(np.median(ranks)) if ranks else None
    return out


def lexical_cands(Sl, E, depth):
    F = np.where(E, -np.inf, Sl)
    C = np.zeros(F.shape, bool)
    d = min(depth, F.shape[1] - 1)
    np.put_along_axis(C, np.argpartition(-F, d - 1, axis=1)[:, :d], True, 1)
    return C


def choose_ranker(model_scores, Sl, S, E, T, w_grid=None, cand_depths=(None, 100), rows=None):
    """Grid over (model score function, candidate set, fusion weight) by dev Micro-F1@7.
    model_scores: {fn: raw score matrix}. Returns (best config, table of every config)."""
    w_grid = w_grid or [round(x, 2) for x in np.arange(0.1, 1.0001, 0.1)]
    cands = {d: (None if d is None else lexical_cands(Sl, E, d)) for d in cand_depths}
    res = []
    Z = {fn: zrows(M, E) for fn, M in model_scores.items()}
    Zl = zrows(Sl, E)
    for fn in model_scores:
        for d, C in cands.items():
            for w in w_grid:
                F = w * Z[fn] + (1 - w) * Zl if w < 1 else Z[fn].copy()
                if T is not None:
                    F = F - 100.0 * T
                if C is not None:
                    F = F - 1000.0 * (~C)
                F = np.where(E, -np.inf, F)
                m = prf(topk(F), S["gold"], rows)
                res.append(dict(fn=fn, cand=d, w=w, f1=m["f1"], p=m["p"], r=m["r"]))
    best = max(res, key=lambda x: (round(x["f1"], 6), x["cand"] is None, x["w"]))
    return best, res


def crossfit_ranker(model_scores, Sl, S, E, T, seed=0, **kw):
    """Honest estimate of the dev-tuned fusion: choose on one half of the queries, score on the other, swap."""
    idx = np.random.default_rng(seed).permutation(len(S["qids"]))
    halves = [sorted(idx[: len(idx) // 2].tolist()), sorted(idx[len(idx) // 2:].tolist())]
    tot = dict(tp=0, npred=0, ngold=0)
    for a, b in [(0, 1), (1, 0)]:
        best, _ = choose_ranker(model_scores, Sl, S, E, T, rows=halves[a], **kw)
        F = final_scores(model_scores[best["fn"]], Sl, best["w"], E, T,
                         None if best["cand"] is None else lexical_cands(Sl, E, best["cand"]))
        m = prf(topk(F), S["gold"], halves[b])
        tot["tp"] += m["tp"]; tot["npred"] += K * len(halves[b]); tot["ngold"] += sum(len(S["gold"][i]) for i in halves[b])
    P, R = tot["tp"] / max(1, tot["npred"]), tot["tp"] / max(1, tot["ngold"])
    return 2 * P * R / max(1e-12, P + R)


def to_prediction(top, S):
    return {q: [S["pool"][j] for j in top[i]] for i, q in enumerate(S["qids"])}


def check_prediction(pred, S):
    """The Codabench rules: every test query, exactly 7 unique ids from the test pool, ordered (we never return the query)."""
    assert isinstance(pred, dict), "prediction must be a dict {query_id: [7 candidate ids]}"
    assert set(pred) == set(S["qids"]) and len(pred) == len(S["qids"]), "every test query exactly once, no unknown ids"
    pool = S["pidx"]
    for q, lst in pred.items():
        assert isinstance(lst, list) and len(lst) == K, f"{q}: exactly {K} candidate ids"
        assert all(isinstance(c, str) for c in lst), f"{q}: ids must be strings"
        assert len(set(lst)) == K, f"{q}: duplicate ids"
        assert all(c in pool for c in lst), f"{q}: id not in the test candidate pool"
        assert q not in lst, f"{q}: the query itself was returned"


def write_zip(pred, path):
    path = Path(path)
    pj = path.with_suffix(".json")
    json.dump(pred, open(pj, "w", encoding="utf-8"), indent=1)
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    os.remove(pj)
    return path


# ============================================================== lexical retrieval (TF-IDF / BM25)
PH_RE = re.compile(r"<[a-z_]+>|\[[a-z_]+\]")


def lex_clean(t):
    return PH_RE.sub(" ", t.lower())                         # masks (<citation>, <name>, <org>, [fb]) carry no content


_LEX = {}


def _vectorize(args):
    name, ngram, stop = args
    from sklearn.feature_extraction.text import CountVectorizer
    texts = _LEX[name]
    kw = dict(preprocessor=lex_clean, token_pattern=r"(?u)\b\w\w+\b", ngram_range=(1, ngram), dtype=np.float32,
              stop_words="english" if stop else None)
    try:
        X = CountVectorizer(min_df=2, max_df=0.9, **kw).fit_transform(texts)
    except ValueError:                                       # tiny corpora (tests): keep every term
        X = CountVectorizer(**kw).fit_transform(texts)
    return (name, ngram, stop), X.tocsr()


def vectorize_many(jobs, texts_by_name, n_proc=None):
    """jobs: [(name, ngram, stop)] -> {(name, ngram, stop): csr count matrix}; documents in the order of texts_by_name[name]."""
    _LEX.clear()
    _LEX.update(texts_by_name)
    n_proc = min(len(jobs), n_proc or max(1, os.cpu_count() or 1))
    if n_proc > 1:
        with mp.get_context("fork").Pool(n_proc) as pool:
            out = dict(pool.map(_vectorize, jobs, chunksize=1))
    else:
        out = dict(map(_vectorize, jobs))
    _LEX.clear()
    return out


def bm25_scores(X, qrows, k1, b, qmode):
    X = X.tocsr()
    N = X.shape[0]
    df = np.bincount(X.indices, minlength=X.shape[1]).astype(np.float64)
    idf = np.log1p((N - df + 0.5) / (df + 0.5)).astype(np.float32)
    dl = np.asarray(X.sum(1)).ravel()
    avg = max(1e-9, dl.mean())
    W = X.copy()
    rows = np.repeat(np.arange(N), np.diff(X.indptr))
    tf = W.data
    W.data = (idf[W.indices] * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[rows] / avg))).astype(np.float32)
    Q = X[qrows].copy()
    Q.data = np.ones_like(Q.data) if qmode == "binary" else (1 + np.log(Q.data)).astype(np.float32)
    return (Q @ W.T).toarray().astype(np.float32)


def tfidf_scores(X, qrows):
    from sklearn.feature_extraction.text import TfidfTransformer
    T = TfidfTransformer(sublinear_tf=True).fit_transform(X).tocsr()
    return (T[qrows] @ T.T).toarray().astype(np.float32)


LEX_VEC = [(1, False), (1, True), (2, False), (2, True)]
LEX_SCORE = [("tfidf", None, None, None)] + [("bm25", k1, b, q) for k1 in (0.9, 1.5, 2.5) for b in (0.6, 0.9) for q in ("binary", "log")]


def lex_name(ngram, stop, kind, k1, b, q):
    s = f"{kind} {'uni' if ngram == 1 else 'uni+bi'}{' -stop' if stop else ''}"
    return s + (f" k1={k1} b={b} q={q}" if kind == "bm25" else "")


def lex_doc_texts(S, texts):
    """Texts in the row order used for a split: pool candidates first, then queries that are not in the pool."""
    keys = [S["ckey"][c] for c in S["pool"]]
    extra = [S["qkey"][q] for q in S["qids"] if q not in S["pidx"]]
    qrows = [S["pidx"][q] if q in S["pidx"] else len(keys) + extra.index(S["qkey"][q]) for q in S["qids"]]
    return [texts[k] for k in keys + extra], qrows


def lex_matrix(X, qrows, npool, cfg):
    kind, k1, b, q = cfg
    M = tfidf_scores(X, qrows) if kind == "tfidf" else bm25_scores(X, qrows, k1, b, q)
    return M[:, :npool]


def hard_negatives(Sl, S, E, depth=30):
    """Per query: the `depth` best lexical candidates that are neither the query nor relevant (pool indices)."""
    F = np.where(E, -np.inf, Sl).copy()
    out = []
    for i, g in enumerate(S["gold"]):
        F[i, g] = -np.inf
        d = min(depth, int(np.isfinite(F[i]).sum()))
        out.append(np.argsort(-F[i], kind="stable")[:d].tolist() if d > 0 else [])
    return out


# ============================================================== tokenisation + chunks (exact Phase-1 pipeline)
_WORD = re.compile(r"\S+")
CIT_RE = re.compile(r"<CITATION>")


def encode_pieces(raw, norm_fn, enc_fn, piece_words=128):
    """Whole document -> (ids, token offsets of the <CITATION> masks, n_words). Encoded in whitespace-aligned pieces:
    the Phase-1 pre-tokenisers never let a token cross such a cut, so the result equals the full encoding
    (asserted in the notebook)."""
    t = norm_fn(raw)
    ends = [m.end() for m in _WORD.finditer(t)]
    cuts = [0] + ends[piece_words - 1::piece_words]
    if cuts[-1] != len(t):
        cuts.append(len(t))
    ids, marks = [], []
    for a, b in zip(cuts, cuts[1:]):
        seg = t[a:b]
        e = enc_fn(seg)
        for m in CIT_RE.finditer(seg):
            marks.append(len(ids) + int(len(e) * m.start() / max(1, len(seg))))
        ids.extend(e)
    return np.asarray(ids, np.int64), marks, len(ends)


_W = {}


def _init_worker(spec):
    import p2_ner as P2
    import p5_lsi as P5
    tok = P2.load_bpe(spec["tok_py"], spec["tok_path"], norm=spec["norm"])
    _W["norm"], _W["enc"] = P5.text_fns(tok, spec["arch"])
    _W["spec"] = spec


def _work(i):
    import p5_lsi as P5
    s = _W["spec"]
    ids, marks, nw = encode_pieces(_W["texts"][i], _W["norm"], _W["enc"], s.get("piece_words", 128))
    chunks, kidx, ntot, mk = P5.make_chunks(ids, marks, s["chunk_len"], s["bos"], s["max_store"], s["select"])
    return i, chunks, kidx, ntot, mk, nw, len(ids), len(marks)


def prepare(texts, keys, spec, cache_path, extra=None, n_proc=None, log=print):
    """Tokenise + chunk every needed document once (multiprocessing, fork) and pickle {'spec', 'docs', ...extra}."""
    n_proc = n_proc or max(1, os.cpu_count() or 1)
    _W["texts"] = [texts[k] for k in keys]
    out = [None] * len(keys)
    t0 = time.time()
    if n_proc > 1:
        with mp.get_context("fork").Pool(n_proc, initializer=_init_worker, initargs=(spec,)) as pool:
            for r in pool.imap_unordered(_work, range(len(keys)), chunksize=8):
                out[r[0]] = r[1:]
    else:
        _init_worker(spec)
        for i in range(len(keys)):
            out[i] = _work(i)[1:]
    _W.pop("texts", None)
    docs = {k: dict(chunks=ch, kidx=ki, ntot=nt, mk=mk, n_words=nw, n_tok=ntk, n_marks=nm)
            for k, (ch, ki, nt, mk, nw, ntk, nm) in zip(keys, out)}
    log(f"[prepare] {len(keys)} unique documents tokenised + chunked in {time.time() - t0:.0f}s")
    pickle.dump(dict(spec=spec, docs=docs, **(extra or {})), open(cache_path, "wb"), protocol=4)
    return docs


def chunk_stats(docs, splits, spec):
    rows = {}
    for sp, S in splits.items():
        ks = split_keys(S)
        nc = np.asarray([len(docs[k]["chunks"]) for k in ks])
        nt = np.asarray([docs[k]["ntot"] for k in ks])
        nw = np.asarray([docs[k]["n_words"] for k in ks])
        ntok = np.asarray([docs[k]["n_tok"] for k in ks])
        nm = np.asarray([docs[k]["n_marks"] for k in ks])
        rows[sp] = dict(docs=len(ks), words_p50=int(np.median(nw)), words_p90=int(np.percentile(nw, 90)), words_max=int(nw.max()),
                        tok_per_word=round(float(ntok.sum() / max(1, nw.sum())), 3), chunks_total_mean=round(float(nt.mean()), 2),
                        chunks_kept_mean=round(float(nc.mean()), 2), docs_fully_covered_pct=round(100 * float(np.mean(nt <= spec["max_store"])), 1),
                        tokens_kept_pct=round(100 * float(sum(sum(len(c) for c in docs[k]["chunks"]) for k in ks) / max(1, ntok.sum())), 1),
                        citations_per_doc=round(float(nm.mean()), 1))
    return rows


# ============================================================== torch part (only the training jobs need it)
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError:                                          # the lexical / data part works without torch
    torch = None

if torch is not None:
    sys.path.insert(0, str(Path(__file__).resolve().parent))
    import p2_ner as P2                                       # noqa: E402  (Phase-2 loaders / encoders / LoRA / LoRMA)
    import p5_lsi as P5                                       # noqa: E402  (Phase-5 chunk encoder, prompts, NF4, views)

    VIEWS = {
        "all": lambda d: list(range(len(d["chunks"]))),
        "first1": lambda d: [0],
        "head4": lambda d: P5.select_chunks(len(d["chunks"]), d["mk"], 4, "head"),
        "uniform4": lambda d: P5.select_chunks(len(d["chunks"]), d["mk"], 4, "uniform"),
        "marker4": lambda d: P5.select_chunks(len(d["chunks"]), d["mk"], 4, "marker"),
        "eval6": lambda d: P5.select_chunks(len(d["chunks"]), d["mk"], 6, "marker"),
        "eval8": lambda d: P5.select_chunks(len(d["chunks"]), d["mk"], 8, "marker"),
    }

    class IA3Linear(nn.Module):
        """IA3: y = l * (W x)  (where='out': keys / values)   or   y = W (l * x)  (where='in': FFN inner activation)."""

        def __init__(self, base: nn.Linear, where):
            super().__init__()
            self.base, self.where = base, where
            for p in base.parameters():
                p.requires_grad_(False)
            self.l = nn.Parameter(torch.ones(base.out_features if where == "out" else base.in_features))

        def forward(self, x):
            if self.where == "out":
                return self.base(x) * self.l.to(x.dtype)
            return self.base(x * self.l.to(x.dtype))

    class PCRModel(nn.Module):
        """Bi-encoder: backbone -> mean-pooled chunk vectors -> (v - centre) @ P -> pooled over chunks -> L2 norm."""

        def __init__(self, enc, J):
            super().__init__()
            self.enc = enc
            d, e = enc.d, J["emb_dim"]
            self.register_buffer("center", torch.zeros(d))
            self.proj = nn.Linear(d, e, bias=False)
            nn.init.orthogonal_(self.proj.weight)              # a random rotation: cosines are roughly kept at step 0
            self.drop = nn.Dropout(J["dropout"])
            self.pool_type = J["doc_pool"]
            if self.pool_type == "attn":
                self.att = nn.Linear(e, 1)
                nn.init.zeros_(self.att.weight); nn.init.zeros_(self.att.bias)     # = mean pooling at step 0
            self._amp, self._ckpt, self._ckpt_micro, self._n_overflow = True, bool(J.get("grad_ckpt")), J.get("ckpt_micro", 4), 0

        def chunk_vecs(self, ids, am):
            h = self.enc.hidden(ids, am)
            m = am.unsqueeze(-1).to(h.dtype)
            return (h * m).sum(1) / m.sum(1).clamp(min=1.0)

        def proj_chunks(self, vecs):
            with torch.autocast(device_type=vecs.device.type, enabled=False):
                return self.proj(self.drop(vecs.float() - self.center.float()))

        def chunk_reps(self, vecs):
            return F.normalize(self.proj_chunks(vecs), dim=-1)

        def doc_reps(self, vecs, lengths):
            """vecs [N, d] = the chunks of B documents, document after document -> [B, e] unit vectors."""
            z = self.proj_chunks(vecs)
            with torch.autocast(device_type=vecs.device.type, enabled=False):
                B = len(lengths)
                L = torch.as_tensor(lengths, device=z.device)
                if self.pool_type == "mean":
                    idx = torch.repeat_interleave(torch.arange(B, device=z.device), L)
                    pooled = torch.zeros(B, z.size(1), device=z.device).index_add_(0, idx, z) / L[:, None].float()
                else:
                    Z = nn.utils.rnn.pad_sequence(list(torch.split(z, list(lengths))), batch_first=True)
                    M = torch.arange(Z.size(1), device=z.device)[None] < L[:, None]
                    if self.pool_type == "max":
                        pooled = Z.masked_fill(~M.unsqueeze(-1), -1e4).max(1).values
                    else:
                        a = torch.softmax(self.att(Z).squeeze(-1).masked_fill(~M, -1e4), 1)
                        pooled = (a.unsqueeze(-1) * Z).sum(1)
                return F.normalize(pooled, dim=-1)

    def apply_method(model, method, J):
        if method == "ia3":
            for p in model.enc.parameters():
                p.requires_grad_(False)
            new = []
            for where, names in (("out", J["ia3_out"]), ("in", J["ia3_in"])):
                for holder, nm in P5._slots(model.enc, names):
                    m = IA3Linear(getattr(holder, nm), where)
                    setattr(holder, nm, m)
                    new.append(m.l)
            assert new, "no IA3 target found - check ia3_out / ia3_in"
            return new, []
        return P5.apply_method(model, method, J)

    def build_model(J, method):
        layers = J["kd_student_layers"] if method == "kd" else None
        enc = P5.build_encoder(J, layers)
        p1_ids = {id(p) for p in enc.parameters()}
        model = PCRModel(enc, J)
        new, later = apply_method(model, method, J)
        model._p1_ids = p1_ids
        return model, new, later

    def count_params(model):
        return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)

    # ---------------------------------------------------------- encoding
    def batch_tensors(docs, sels, device):
        ids, am = P5.collate_chunks([d["chunks"][k] for d, sel in zip(docs, sels) for k in sel])
        return ids.to(device), am.to(device)

    def embed(model, docs, sels, device, log=None):
        """Document embeddings [B, e] (with or without gradient, depending on the caller's context)."""
        ids, am = batch_tensors(docs, sels, device)
        v = P5.encode_chunks(model, ids, am, log)
        return model.doc_reps(v, [len(s) for s in sels]), int(am.sum())

    @torch.no_grad()
    def encode_vecs(model, docs, view, device, tok_budget=32768, log=None):
        """Chunk vectors (fp16, CPU) of the chunks a view selects -> list of [n_sel, d] per document."""
        model.eval()
        fn = VIEWS[view]
        sels = [fn(d) for d in docs]
        items = sorted(((i, j, len(docs[i]["chunks"][k])) for i, sel in enumerate(sels) for j, k in enumerate(sel)), key=lambda x: x[2])
        out = [torch.zeros(len(s), model.enc.d, dtype=torch.float16) for s in sels]
        p = 0
        while p < len(items):
            b = [items[p]]; p += 1
            while p < len(items) and (len(b) + 1) * items[p][2] <= tok_budget:
                b.append(items[p]); p += 1
            ids, am = P5.collate_chunks([docs[i]["chunks"][sels[i][j]] for i, j, _ in b])
            v = P5.encode_chunks(model, ids.to(device), am.to(device), log).half().cpu()
            for (i, j, _), vec in zip(b, v):
                out[i][j] = vec
        return out

    @torch.no_grad()
    def reps_from_vecs(model, vecs, device, bs=512, sub=None):
        """-> (document embeddings [N, e] float32 CPU, chunk embeddings: list of [n, e] fp16 CPU).
        sub(i) -> list of positions inside vecs[i] to use (a view applied on cached vectors)."""
        model.eval()
        D, C = [], []
        for s in range(0, len(vecs), bs):
            part = [vecs[i] if sub is None else vecs[i][sub(i)] for i in range(s, min(s + bs, len(vecs)))]
            v = torch.cat(part).to(device).float()
            D.append(model.doc_reps(v, [len(x) for x in part]).cpu())
            C += [c.half() for c in torch.split(model.chunk_reps(v).cpu(), [len(x) for x in part])]
        return torch.cat(D), C

    def _pad(chs):
        n = max(len(c) for c in chs)
        X = torch.zeros(len(chs), n, chs[0].size(1), dtype=torch.float16)
        M = torch.zeros(len(chs), n, dtype=torch.bool)
        for i, c in enumerate(chs):
            X[i, :len(c)] = c
            M[i, :len(c)] = True
        return X, M

    @torch.no_grad()
    def maxsim(Cq, Cp, device, sym, bq=8):
        """Late interaction over chunk embeddings: mean over query chunks of the best-matching candidate chunk
        (sym: averaged with the candidate -> query direction)."""
        Xq, Mq = _pad(Cq)
        Xp, Mp = _pad(Cp)
        Xp, Mp = Xp.to(device).float(), Mp.to(device)
        out = np.zeros((len(Cq), len(Cp)), np.float32)
        for s in range(0, len(Cq), bq):
            q, mq = Xq[s:s + bq].to(device).float(), Mq[s:s + bq].to(device)
            sim = torch.einsum("aie,pje->apij", q, Xp)
            s1 = sim.masked_fill(~Mp[None, :, None, :], -4).max(-1).values                 # [a, p, i]
            s1 = (s1 * mq[:, None, :]).sum(-1) / mq.sum(-1, keepdim=True)
            if sym:
                s2 = sim.masked_fill(~mq[:, None, :, None], -4).max(-2).values             # [a, p, j]
                s2 = (s2 * Mp[None]).sum(-1) / Mp.sum(-1)[None]
                s1 = 0.5 * (s1 + s2)
            out[s:s + len(q)] = s1.cpu().numpy()
        return out

    def score_fns(R, C, rows_q, rows_p, device, fns=("mean", "maxsim", "maxsim_sym", "mean+maxsim")):
        """Raw model score matrices [n_queries, n_pool] for several similarity functions."""
        out = {}
        Rq, Rp = R[rows_q], R[rows_p]
        if "mean" in fns or "mean+maxsim" in fns:
            out["mean"] = (Rq @ Rp.T).numpy()
        if any(f.startswith("maxsim") for f in fns) or "mean+maxsim" in fns:
            Cq, Cp = [C[i] for i in rows_q], [C[i] for i in rows_p]
            ms = maxsim(Cq, Cp, device, sym=False)
            if "maxsim" in fns:
                out["maxsim"] = ms
            if "maxsim_sym" in fns:
                out["maxsim_sym"] = maxsim(Cq, Cp, device, sym=True)
            if "mean+maxsim" in fns:
                out["mean+maxsim"] = 0.5 * (out["mean"] + ms)
        return {f: out[f] for f in fns if f in out}

    # ---------------------------------------------------------- training (GradCache bi-encoder)
    class Batcher:
        """q_per_step queries; for each: one random relevant case + `hard_per_q` lexical hard negatives.
        Returns unique documents + the indices / masks the loss needs."""

        def __init__(self, S, hard, J, rng):
            self.S, self.hard, self.J, self.rng = S, hard, J, rng
            self.qs = [i for i, g in enumerate(S["gold"]) if g]
            self.order = []
            self.relkeys = [{S["ckey"][S["pool"][j]] for j in g} for g in S["gold"]]

        def next(self):
            S, J, rng = self.S, self.J, self.rng
            picked = []
            while len(picked) < min(J["q_per_step"], len(self.qs)):
                if not self.order:
                    self.order = rng.permutation(self.qs).tolist()
                i = self.order.pop()
                if i not in picked:
                    picked.append(i)
            keys, pos = [], {}

            def add(k):
                if k not in pos:
                    pos[k] = len(keys); keys.append(k)
                return pos[k]
            qi, pi = [], []
            for i in picked:
                qi.append(add(S["qkey"][S["qids"][i]]))
                pi.append(add(S["ckey"][S["pool"][int(rng.choice(S["gold"][i]))]]))
            for i in picked:
                h = self.hard[i]
                if h and J["hard_per_q"] > 0:
                    for j in rng.choice(h, size=min(J["hard_per_q"], len(h)), replace=False):
                        add(S["ckey"][S["pool"][int(j)]])
            M = np.zeros((len(picked), len(keys)), bool)            # True = not a negative for this query
            for a, i in enumerate(picked):
                qk = S["qkey"][S["qids"][i]]
                for b, k in enumerate(keys):
                    if k == qk or (k in self.relkeys[i] and b != pi[a]):
                        M[a, b] = True
            return keys, qi, pi, M

    def contrastive_loss(R, qi, pi, M, tau):
        Sm = (R[qi] @ R.T) / tau
        Sm = Sm.masked_fill(torch.as_tensor(M, device=R.device), -1e4)
        return F.cross_entropy(Sm, torch.as_tensor(pi, device=R.device)), Sm

    def kd_loss(Ss, Rt, qi, M, tau, T):
        St = ((Rt[qi] @ Rt.T) / tau).masked_fill(torch.as_tensor(M, device=Rt.device), -1e4)
        pt = F.softmax(St / T, -1)
        return F.kl_div(F.log_softmax(Ss / T, -1), pt, reduction="batchmean") * T * T

    def micro_batches(docs, sels, tok_budget):
        """Consecutive documents (all chunks of a document together) up to tok_budget padded tokens."""
        out, cur, n, mx = [], [], 0, 0
        for i, (d, sel) in enumerate(zip(docs, sels)):
            ln = max(len(d["chunks"][k]) for k in sel)
            if cur and (n + len(sel)) * max(mx, ln) > tok_budget:
                out.append(cur); cur, n, mx = [], 0, 0
            cur.append(i); n += len(sel); mx = max(mx, ln)
        if cur:
            out.append(cur)
        return out

    def _rng_state(device):
        return (torch.get_rng_state(), torch.cuda.get_rng_state(device) if device.type == "cuda" else None)

    def _set_rng(st, device):
        torch.set_rng_state(st[0])
        if st[1] is not None:
            torch.cuda.set_rng_state(st[1], device)

    @torch.no_grad()
    def compute_center(model, docs, device, J, log=None):
        vecs = encode_vecs(model, docs, "first1" if J.get("quick") else "eval6", device, J["infer_tok_budget"], log)
        c = torch.cat(vecs).float().mean(0)
        model.center.copy_(c.to(model.center.device))
        return c

    def eval_split(model, docs_by_key, S, Sl, E, T, device, view, J, log, fns=("mean",)):
        keys = split_keys(S)
        idx = {k: i for i, k in enumerate(keys)}
        vecs = encode_vecs(model, [docs_by_key[k] for k in keys], view, device, J["infer_tok_budget"], log)
        R, C = reps_from_vecs(model, vecs, device)
        rows_q = [idx[S["qkey"][q]] for q in S["qids"]]
        rows_p = [idx[S["ckey"][c]] for c in S["pool"]]
        return score_fns(R, C, rows_q, rows_p, device, fns), vecs, (rows_q, rows_p, keys)

    def train_one(J, model, new_params, later_params, docs_by_key, St, hard, Sd, Sl_dev, Ed, Td, device, log, teacher=None):
        torch.manual_seed(J["seed"])
        rng = np.random.default_rng(J["seed"])
        model.to(device)
        new_ids = {id(p) for p in new_params}
        head = [p for n, p in model.named_parameters() if not n.startswith("enc.")]
        peft_p = [p for p in model.enc.parameters() if id(p) in new_ids]
        body = [p for p in model.enc.parameters() if id(p) not in new_ids and (p.requires_grad or any(p is q for q in later_params))]
        groups = [dict(params=head, lr=J["head_lr"], weight_decay=J["weight_decay"])]
        if peft_p:
            groups.append(dict(params=peft_p, lr=J["peft_lr"], weight_decay=0.0 if J["method"] in ("prompt", "prefix", "ia3") else J["weight_decay"]))
        if body:
            groups.append(dict(params=body, lr=J["lr"], weight_decay=J["weight_decay"]))
        opt = torch.optim.AdamW(groups)
        for g in opt.param_groups:
            g["base_lr"] = g["lr"]
        opt_ids = {id(p) for g in opt.param_groups for p in g["params"]}
        frozen_ids = {i for i in model._p1_ids if i not in opt_ids}
        keep_keys = ({n for n, p in model.named_parameters(remove_duplicate=False) if id(p) not in frozen_ids}
                     | {n for n, _ in model.named_buffers()})
        model._keep_keys = keep_keys
        batcher = Batcher(St, hard, J, rng)
        steps_per_epoch = math.ceil(len(batcher.qs) / J["q_per_step"])
        total = max(10, int(J["epochs"] * steps_per_epoch))
        planned = total
        warm = max(1, int(J["warmup"] * total))
        use_amp = device.type == "cuda"
        scaler = torch.amp.GradScaler("cuda", init_scale=2.0 ** 10, enabled=use_amp)
        if use_amp:
            torch.cuda.reset_peak_memory_stats(device)
        best = dict(key=-1.0, step=0, state=None)
        hist, evals = [], []
        step, n_nan, n_skip, n_oom, tok_seen = 0, 0, 0, 0, 0
        budget = J["time_budget_h"] * 3600
        t0 = time.time()
        eval_secs = [None]
        next_eval = [int(total * (i + 1) / J["n_evals"]) for i in range(J["n_evals"])]
        gstage, early_done, early_abort = -1, False, False
        log(f"{len(batcher.qs)} training queries with >= 1 relevant case | {J['q_per_step']} queries per step "
            f"(+1 positive, +{J['hard_per_q']} hard negatives each) -> {steps_per_epoch} steps per epoch | "
            f"{J['epochs']} epochs = {total} steps planned")

        def lr_factor(s):
            return (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm))

        def do_eval(tag):
            te = time.time()
            sc, _, _ = eval_split(model, docs_by_key, Sd, Sl_dev, Ed, Td, device, J["eval_view"], J, log)
            Fm = final_scores(sc["mean"], None, 1.0, Ed, Td)
            mm = prf(topk(Fm), Sd["gold"])
            mf = prf(topk(final_scores(sc["mean"], Sl_dev, 0.5, Ed, Td)), Sd["gold"])
            rk = rank_metrics(Fm, Sd["gold"], ks=(20, 100))
            ok = bool(np.isfinite(sc["mean"]).all())
            key = (mm["f1"] + mf["f1"]) if ok else -1.0
            rec = dict(step=step, epoch=round(step / steps_per_epoch, 2), dev_f1_model=round(mm["f1"], 4), dev_f1_fused05=round(mf["f1"], 4),
                       recall20=rk["recall@20"], recall100=rk["recall@100"], mrr=rk["mrr"], finite=ok, eval_secs=round(time.time() - te, 1), tag=tag)
            evals.append(rec)
            eval_secs[0] = time.time() - te
            log("[EVAL] " + json.dumps(rec))
            # the untrained start is only a reference point: every job submits its own TRAINED weights (else identical zips)
            if key > best["key"] and tag != "init":
                best.update(key=key, step=step, f1_model=mm["f1"], f1_fused=mf["f1"],
                            state={k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep_keys})
            model.train()

        do_eval("init")                                       # untrained head: the starting point (+ times one eval)
        model.train()
        tl, tn = 0.0, 0
        t_cal = None
        while step < total:
            if J["method"] == "gradual":
                st = max(i for i, (f, _) in enumerate(J["gradual_schedule"]) if step >= f * total)
                if st != gstage:
                    gstage = st
                    P5.set_gradual_stage(model, J["gradual_schedule"][st][1])
                    log(f"gradual unfreezing: step {step} -> top {J['gradual_schedule'][st][1]} blocks trainable")
            f = lr_factor(step)
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * f
            keys, qi, pi, M = batcher.next()
            docs = [docs_by_key[k] for k in keys]
            sels = [P5.pick_chunks(len(d["chunks"]), J["train_chunks"], rng) for d in docs]
            mbs = micro_batches(docs, sels, J["tok_budget"])
            while True:
                try:
                    states, parts, ntok = [], [], 0
                    with torch.no_grad():                     # pass 1: embeddings, no graph
                        for mb in mbs:
                            states.append(_rng_state(device))
                            r, nt = embed(model, [docs[i] for i in mb], [sels[i] for i in mb], device, log)
                            parts.append(r); ntok += nt
                    R = torch.cat(parts).float().detach().requires_grad_(True)
                    loss, Ss = contrastive_loss(R, qi, pi, M, J["tau"])
                    if teacher is not None:
                        with torch.no_grad():
                            Rt = torch.cat([embed(teacher, [docs[i] for i in mb], [sels[i] for i in mb], device)[0] for mb in mbs]).float()
                        loss = J["kd_alpha"] * loss + (1 - J["kd_alpha"]) * kd_loss(Ss, Rt, qi, M, J["tau"], J["kd_temperature"])
                    if not torch.isfinite(loss):
                        n_nan += 1
                        if n_nan <= 5 or n_nan % 50 == 0:
                            log(f"non-finite loss at step {step} (#{n_nan}) - batch skipped")
                        opt.zero_grad(set_to_none=True)
                        break
                    loss.backward()
                    G = R.grad.detach()
                    opt.zero_grad(set_to_none=True)
                    off = 0
                    for mb, st in zip(mbs, states):           # pass 2: same micro-batches, same dropout masks
                        _set_rng(st, device)
                        r, _ = embed(model, [docs[i] for i in mb], [sels[i] for i in mb], device, log)
                        g = G[off:off + len(mb)]
                        off += len(mb)
                        scaler.scale((r.float() * g).sum()).backward()
                    scaler.unscale_(opt)
                    for gr in opt.param_groups:
                        torch.nn.utils.clip_grad_norm_([p for p in gr["params"] if p.grad is not None], 1.0)
                    sc0 = scaler.get_scale() if use_amp else 1.0
                    scaler.step(opt)
                    scaler.update()
                    if use_amp and scaler.get_scale() < sc0:
                        n_skip += 1
                    tl += loss.item(); tn += 1; tok_seen += ntok
                    break
                except torch.cuda.OutOfMemoryError:
                    n_oom += 1
                    opt.zero_grad(set_to_none=True)
                    loss = R = parts = None
                    torch.cuda.empty_cache()
                    if not model._ckpt:
                        model._ckpt = True
                        log(f"CUDA OOM at step {step} -> gradient checkpointing ON (every {model._ckpt_micro} chunks)")
                    elif J["tok_budget"] > 2 * 512 * J["train_chunks"]:
                        J["tok_budget"] = int(J["tok_budget"] * 0.75)
                        log(f"CUDA OOM again -> micro-batch token budget reduced to {J['tok_budget']}")
                    else:
                        raise
                    mbs = micro_batches(docs, sels, J["tok_budget"])
            step += 1
            if step == 2:
                t_cal = time.time()
            if step == J["calib_steps"] and t_cal is not None:
                sps = (time.time() - t_cal) / (step - 2)
                ev = eval_secs[0] or 60.0
                final_s = ev * J["_final_vs_eval_tokens"] * 1.15
                reserve = J["n_evals"] * ev + final_s + 240
                fit = step + int((budget - (time.time() - t0) - reserve) / sps)
                total = max(step + 10, min(planned, fit))
                warm = min(warm, max(1, int(J["warmup"] * total)))
                next_eval = [max(step + 1, int(total * (i + 1) / J["n_evals"])) for i in range(J["n_evals"])]
                log(f"calibration: {sps:.2f} s/step | {ev / 60:.1f} min per dev eval, ~{final_s / 60:.1f} min for the final dev + test "
                    f"encoding | steps {planned} planned -> {total} fit the {J['time_budget_h']:.2f} h budget (= {total / steps_per_epoch:.1f} "
                    f"passes over the training queries)")
            if step % J["log_every"] == 0:
                el = time.time() - t0
                log(json.dumps(dict(step=step, total=total, loss=round(tl / max(tn, 1), 4), lr_f=round(f, 4), tok_s=int(tok_seen / max(el, 1e-6)),
                                    min=round(el / 60, 1), nan=n_nan, skip=n_skip, fp32=not model._amp, ckpt=model._ckpt, micro=len(mbs))))
                hist.append(dict(step=step, loss=tl / max(tn, 1), min=el / 60))
                tl, tn = 0.0, 0
            if J.get("early_check") and not early_done and step >= J["early_check"] * total and step >= J["calib_steps"]:
                early_done = True
                do_eval("early-check")
                if evals[-1]["dev_f1_model"] < J.get("early_min_f1", 0.015):
                    log(f"!!! dev F1@7 {evals[-1]['dev_f1_model']:.4f} after {step} steps ({100 * step / total:.0f}% of training): "
                        "the model learnt nothing -> stopping this job early")
                    early_abort = True
                    break
            if step in next_eval or step >= total:
                do_eval("scheduled")
                next_eval = [s for s in next_eval if s > step]
            elif time.time() - t0 > 1.08 * budget:
                log(f"time budget exceeded at step {step}/{total} -> final eval and stop")
                do_eval("time-stop")
                break
        if best["state"] is None:
            best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep_keys}
        info = dict(train_secs=time.time() - t0, steps=step, planned_steps=planned, epochs=step / steps_per_epoch,
                    tok_per_s=tok_seen / max(time.time() - t0, 1e-6), nan_batches=n_nan, skipped_steps=n_skip, ooms=n_oom,
                    fp32=not model._amp, grad_ckpt=model._ckpt, early_abort=early_abort, history=hist, evals=evals,
                    peak_mem_gb=(torch.cuda.max_memory_allocated(device) / 2 ** 30) if use_amp else None)
        return best, info

    # ---------------------------------------------------------- frozen probe: heads on cached chunk vectors
    def probe_train(model, feats, St, hard, J, device, log, loss_kind="infonce", neg="bm25"):
        """Train the head (projection, + attention pooling) on cached chunk vectors. feats: {key: [n, d] fp16}.
        loss_kind: 'infonce' | 'pairwise';  neg: 'bm25' (lexical hard negatives) | 'random' | 'none' (in-batch only)."""
        torch.manual_seed(J["seed"])
        rng = np.random.default_rng(J["seed"])
        model.to(device).train()
        params = [p for n, p in model.named_parameters() if not n.startswith("enc.")]
        opt = torch.optim.AdamW(params, lr=J["head_lr"], weight_decay=J["weight_decay"])
        Jp = dict(J, q_per_step=J["probe_q_per_step"], hard_per_q=0 if neg == "none" else J["probe_hard_per_q"])
        hard_used = hard
        if neg == "random":
            hard_used = [rng.choice(len(St["pool"]), size=30, replace=False).tolist() for _ in St["qids"]]
        batcher = Batcher(St, hard_used, Jp, rng)
        spe = math.ceil(len(batcher.qs) / Jp["q_per_step"])
        total = max(20, int(J["probe_epochs"] * spe))
        Mx = J.get("probe_chunks", 8)

        def emb(keys, grad=True):
            parts = []
            for k in keys:
                v = feats[k]
                if len(v) > Mx:
                    v = v[sorted(rng.choice(len(v), Mx, replace=False).tolist())]
                parts.append(v)
            v = torch.cat(parts).to(device).float()
            return model.doc_reps(v, [len(p) for p in parts])
        for step in range(total):
            for g in opt.param_groups:
                g["lr"] = J["head_lr"] * min(1.0, (step + 1) / max(1, 0.05 * total)) * max(0.0, (total - step) / total)
            keys, qi, pi, M = batcher.next()
            R = emb(keys)
            if loss_kind == "infonce":
                loss, _ = contrastive_loss(R, qi, pi, M, J["tau"])
            else:                                         # pairwise: softplus((s_neg - s_pos) / tau) over the batch negatives
                Sm = R[qi] @ R.T
                pt = torch.as_tensor(pi, device=device)
                sp = Sm.gather(1, pt[:, None])
                excl = torch.as_tensor(M, device=device).clone()
                excl[torch.arange(len(pi), device=device), pt] = True
                keep = (~excl).float()
                loss = (F.softplus((Sm - sp) / J["tau"]) * keep).sum() / keep.sum().clamp(min=1.0)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
        model.eval()
        return dict(steps=total)

    def cached_scores(model, vecs_by_key, S, device, fns=("mean",), sub=None):
        keys = split_keys(S)
        idx = {k: i for i, k in enumerate(keys)}
        vecs = [vecs_by_key[k] for k in keys]
        R, C = reps_from_vecs(model, vecs, device, sub=(None if sub is None else (lambda i: sub(keys[i]))))
        return score_fns(R, C, [idx[S["qkey"][q]] for q in S["qids"]], [idx[S["ckey"][c]] for c in S["pool"]], device, fns)

    @torch.no_grad()
    def zero_shot_scores(vecs_by_key, S, device, center, mode):
        """Phase-1 transfer WITHOUT any training: mean-pooled hidden states, cosine (optionally centred)."""
        keys = split_keys(S)
        idx = {k: i for i, k in enumerate(keys)}
        c = center.float() if center is not None else 0.0
        if mode == "first1":
            V = torch.stack([vecs_by_key[k][0].float() for k in keys]) - c
        else:
            V = torch.stack([vecs_by_key[k].float().mean(0) for k in keys]) - c
        V = F.normalize(V, dim=-1)
        rq = [idx[S["qkey"][q]] for q in S["qids"]]
        rp = [idx[S["ckey"][c]] for c in S["pool"]]
        if mode == "maxsim":
            C = [F.normalize(vecs_by_key[k].float() - c, dim=-1).half() for k in keys]
            return maxsim([C[i] for i in rq], [C[i] for i in rp], device, sym=False)
        return (V[rq] @ V[rp].T).numpy()


# ============================================================== analysis
def analyze(top, F_model, F_lex, F_final, S, words, T=None, n_show=8):
    gold = S["gold"]
    f1q = []
    for i, g in enumerate(gold):
        tp = len(set(top[i].tolist()) & set(g))
        p, r = tp / K, tp / max(1, len(g))
        f1q.append(2 * p * r / max(1e-12, p + r))
    f1q = np.asarray(f1q)
    ng = np.asarray([len(g) for g in gold])
    qw = np.asarray([words[S["qkey"][q]] for q in S["qids"]])
    out = {}
    out["by_n_relevant"] = {nm: dict(queries=int(m.sum()), micro_f1=round(prf(top, gold, np.where(m)[0].tolist())["f1"], 4))
                            for nm, m in [("1", ng == 1), ("2-4", (ng >= 2) & (ng <= 4)), ("5-9", (ng >= 5) & (ng <= 9)), (">=10", ng >= 10)] if m.any()}
    out["by_query_words"] = {nm: dict(queries=int(m.sum()), micro_f1=round(prf(top, gold, np.where(m)[0].tolist())["f1"], 4))
                             for nm, m in [("<2k", qw < 2000), ("2k-5k", (qw >= 2000) & (qw < 5000)), ("5k-10k", (qw >= 5000) & (qw < 10000)),
                                           (">=10k", qw >= 10000)] if m.any()}
    out["queries_with_zero_hits"] = int((f1q == 0).sum())
    out["rank_model_only"] = rank_metrics(F_model, gold) if F_model is not None else None
    out["rank_lexical"] = rank_metrics(F_lex, gold)
    out["rank_final"] = rank_metrics(F_final, gold)
    if F_model is not None:
        tm, tl = topk(F_model), topk(F_lex)
        out["top7_overlap_model_vs_lexical"] = round(float(np.mean([len(set(a) & set(b)) for a, b in zip(tm.tolist(), tl.tolist())]) / K), 3)
        hm = sum(len(set(tm[i].tolist()) & set(g) - set(tl[i].tolist())) for i, g in enumerate(gold))
        hl = sum(len(set(tl[i].tolist()) & set(g) - set(tm[i].tolist())) for i, g in enumerate(gold))
        out["hits_only_model"], out["hits_only_lexical"] = int(hm), int(hl)
    pool_w = np.asarray([words[S["ckey"][c]] for c in S["pool"]])
    tpw, fpw = [], []
    for i, g in enumerate(gold):
        for j in top[i].tolist():
            (tpw if j in g else fpw).append(pool_w[j])
    gw = [pool_w[j] for g in gold for j in g]
    out["words_of_returned_relevant_p50"] = int(np.median(tpw)) if tpw else None
    out["words_of_returned_irrelevant_p50"] = int(np.median(fpw)) if fpw else None
    out["words_of_all_relevant_p50"] = int(np.median(gw)) if gw else None
    out["words_of_pool_p50"] = int(np.median(pool_w))
    if T is not None:
        out["returned_temporal_violations"] = int(sum(T[i, j] for i in range(len(gold)) for j in top[i]))
    worst = np.argsort(f1q, kind="stable")[:n_show]
    out["hardest_queries"] = [dict(id=S["qids"][i], n_relevant=int(ng[i]), words=int(qw[i]), f1=round(float(f1q[i]), 3)) for i in worst]
    return out


# ============================================================== quick-test subsampling (identical in notebook and jobs)
def subsample_split(S, nq, n_extra, seed):
    """Fewer queries + a smaller pool (their own docs, their relevant cases, random others). Returns (S', q_sel, p_sel)."""
    rng = np.random.default_rng(seed)
    cand = [i for i in range(len(S["qids"])) if S["gold"] is None or S["gold"][i]]
    qsel = sorted(rng.choice(cand, size=min(nq, len(cand)), replace=False).tolist())
    keep = set()
    for i in qsel:
        keep |= set(S["excl"][i])
        if S["gold"] is not None:
            keep |= set(S["gold"][i])
    rest = [j for j in range(len(S["pool"])) if j not in keep]
    keep |= set(rng.choice(rest, size=min(n_extra, len(rest)), replace=False).tolist()) if rest else set()
    psel = sorted(keep)
    qids = [S["qids"][i] for i in qsel]
    pool = [S["pool"][j] for j in psel]
    rel = None
    if S["gold"] is not None:
        rel = {S["qids"][i]: [S["pool"][j] for j in S["gold"][i]] for i in qsel}
    S2 = make_split(S["name"], qids, pool, {q: S["qkey"][q] for q in qids}, {c: S["ckey"][c] for c in pool}, rel)
    return S2, qsel, psel


QUICK = dict(train=(48, 150, 11), dev=(30, 150, 12), test=(20, 150, 13))


def quick_splits(splits, lex):
    out_s, out_l = {}, {}
    for sp, S in splits.items():
        nq, ne, seed = QUICK[sp]
        S2, qs, ps = subsample_split(S, nq, ne, seed)
        out_s[sp] = S2
        out_l[sp] = lex[sp][np.ix_(qs, ps)]
    return out_s, out_l


# ============================================================== one experiment (script entry point)
def run_job(job_path):
    J = json.load(open(job_path))
    out = Path(J["out_dir"]); out.mkdir(parents=True, exist_ok=True)
    logf = open(out / "log.txt", "a")

    def log(msg):
        line = f"[{time.strftime('%H:%M:%S')}][{J['name']}] {msg}"
        print(line, flush=True); logf.write(line + "\n"); logf.flush()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    D = pickle.load(open(J["cache"], "rb"))
    docs, years = D["docs"], D["years"]
    LX = pickle.load(open(J["lex_cache"], "rb"))
    splits, lex = LX["splits"], LX["scores"]
    if J.get("quick"):
        splits, lex = quick_splits(splits, lex)
    St, Sd, Ste = splits["train"], splits["dev"], splits["test"]
    Et, Ed, Ete = excl_mask(St), excl_mask(Sd), excl_mask(Ste)
    tslack = J.get("temporal_slack")
    Td = temporal_violations(Sd, years, tslack) if tslack is not None else None
    Tte = temporal_violations(Ste, years, tslack) if tslack is not None else None
    hard = hard_negatives(lex["train"], St, Et, J["hard_depth"])
    J["_bos"] = D["spec"].get("bos")
    ntok = lambda keys, view: sum(len(docs[k]["chunks"][j]) for k in keys for j in VIEWS[view](docs[k]))
    dev_keys, test_keys = split_keys(Sd), split_keys(Ste)
    J["_final_vs_eval_tokens"] = (ntok(dev_keys, "all") + ntok([k for k in test_keys if k not in set(dev_keys)], "all")) / max(1, ntok(dev_keys, J["eval_view"]))
    method = J["method"]
    log(f"arch={J['arch']} method={method} attn={J['attn']} device={device} | train {len(St['qids'])} queries / {len(St['pool'])} pool, "
        f"dev {len(Sd['qids'])} / {len(Sd['pool'])}, test {len(Ste['qids'])} / {len(Ste['pool'])} | temporal slack {tslack} | budget {J['time_budget_h']:.2f} h")
    R = dict(name=J["name"], method=method, config={k: v for k, v in J.items() if not k.startswith("_")})

    teacher = None
    if method == "kd":
        cands = J["kd_teacher_dir"] if isinstance(J["kd_teacher_dir"], list) else [J["kd_teacher_dir"]]
        ok = []
        for c in map(Path, cands):
            if (c / "val_model.pt").exists() and (c / "results.json").exists():
                rc = json.load(open(c / "results.json"))
                f1 = rc.get("dev_model_only", {}).get("f1", 0.0)
                log(f"teacher candidate {c.name}: dev F1@7 (model only) {f1:.4f}, collapsed={rc.get('collapsed')}")
                if J.get("quick") or not rc.get("collapsed"):
                    ok.append((f1, str(c)))
        assert ok, f"KD needs a healthy teacher checkpoint in one of {cands} (teacher job with save_model=True)"
        tdir = Path(max(ok)[1])
        tj = json.load(open(tdir / "job.json"))
        tj["_bos"] = J["_bos"]
        teacher, _, _ = build_model(tj, tj["method"])
        missing, unexpected = teacher.load_state_dict(torch.load(tdir / "val_model.pt", map_location="cpu"), strict=False)
        assert not unexpected, f"teacher checkpoint has unexpected keys {unexpected[:5]}"
        teacher.to(device).eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
        tres = json.load(open(tdir / "results.json"))
        R["teacher"] = dict(name=tj["name"], method=tj["method"], params=count_params(teacher)[0],
                            dev_f1_model=tres.get("dev_model_only", {}).get("f1"), ms_per_doc=tres.get("ms_per_doc"))
        log(f"teacher {tj['name']} ({tj['method']}) {R['teacher']['params'] / 1e6:.2f}M params, dev F1@7 (model only) {R['teacher']['dev_f1_model']}")

    model, new, later = build_model(J, method)
    R["params_total"] = count_params(model)[0]
    R["params_trainable"] = sum(p.numel() for p in model.parameters() if p.requires_grad or any(p is q for q in later))
    R["params_encoder"] = sum(p.numel() for p in model.enc.parameters())
    if method == "qlora":
        nf = [m for m in model.modules() if isinstance(m, P5.NF4Linear)]
        R["qlora"] = dict(quantised_layers=len(nf), weights=sum(m.n for m in nf), nf4_bytes=sum(m.nbytes() for m in nf),
                          fp16_bytes=2 * sum(m.n for m in nf))
        log(f"QLoRA: {len(nf)} projections in NF4: {R['qlora']['nf4_bytes'] / 2 ** 20:.1f} MB instead of {R['qlora']['fp16_bytes'] / 2 ** 20:.1f} MB (fp16)")
        R["params_total"] += R["qlora"]["weights"]; R["params_encoder"] += R["qlora"]["weights"]
    log(f"params total={R['params_total'] / 1e6:.2f}M (encoder {R['params_encoder'] / 1e6:.2f}M) trainable={R['params_trainable'] / 1e6:.3f}M | PEFT tensors {len(new)}")
    model.to(device)
    train_keys = split_keys(St)
    rng0 = np.random.default_rng(0)
    ck = [train_keys[i] for i in rng0.choice(len(train_keys), size=min(len(train_keys), 32 if J.get("quick") else 256), replace=False)]
    compute_center(model, [docs[k] for k in ck], device, J, log)

    words = {k: d["n_words"] for k, d in docs.items()}
    fns = tuple(J["score_fns"])
    if method == "frozen":
        t0 = time.time()
        need_train = list(dict.fromkeys(train_keys))
        tv = encode_vecs(model, [docs[k] for k in need_train], "eval8", device, J["infer_tok_budget"], log)
        feats = dict(zip(need_train, tv))
        dt_keys = list(dict.fromkeys(dev_keys + test_keys))
        dv = encode_vecs(model, [docs[k] for k in dt_keys], "all", device, J["infer_tok_budget"], log)
        vfeats = dict(zip(dt_keys, dv))
        R["feature_secs"] = time.time() - t0
        enc_test_secs = R["feature_secs"] * ntok(test_keys, "all") / max(1, ntok(need_train, "eval8") + ntok(dt_keys, "all"))
        log(f"frozen chunk vectors cached in {R['feature_secs'] / 60:.1f} min")
        center = torch.cat(list(feats.values())).float().mean(0)
        model.center.copy_(center.to(model.center.device))
        # (1) zero-shot Phase-1 transfer: no training at all
        zs = {}
        for nm, c, mode in [("zero-shot mean, raw cosine", None, "mean"), ("zero-shot mean, centred", center, "mean"),
                            ("zero-shot first chunk, centred", center, "first1"), ("zero-shot MaxSim, centred", center, "maxsim")]:
            M = zero_shot_scores(vfeats, Sd, device, c, mode)
            zs[nm] = dict(model_only=round(prf(topk(final_scores(M, None, 1.0, Ed, Td)), Sd["gold"])["f1"], 4),
                          fused_best=round(max(prf(topk(final_scores(M, lex["dev"], w, Ed, Td)), Sd["gold"])["f1"] for w in (0.1, 0.2, 0.3, 0.5)), 4))
            log(f"[zero-shot] {nm}: {json.dumps(zs[nm])}")
        R["zero_shot"] = zs
        # (2) the probe: train the projection head on cached vectors
        t1 = time.time()
        info = probe_train(model, feats, St, hard, J, device, log)
        info.update(train_secs=time.time() - t1, epochs=J["probe_epochs"], evals=[], peak_mem_gb=None, steps=info["steps"])
        best = dict(state={k: v.detach().cpu().clone() for k, v in model.state_dict().items() if not k.startswith("enc.")}, step=info["steps"])
        dev_sc = cached_scores(model, vfeats, Sd, device, fns)
        # (3) ablations, one change at a time (heads retrained on the cached vectors)
        ab = {}
        base_state = {k: v.clone() for k, v in model.state_dict().items() if not k.startswith("enc.")}

        def ab_eval(m, sub=None, fn="mean"):
            sc = cached_scores(m, vfeats, Sd, device, (fn,), sub)
            mo = prf(topk(final_scores(sc[fn], None, 1.0, Ed, Td)), Sd["gold"])["f1"]
            fu = max(prf(topk(final_scores(sc[fn], lex["dev"], w, Ed, Td)), Sd["gold"])["f1"] for w in (0.2, 0.3, 0.5, 0.7))
            return dict(model_only=round(mo, 4), fused_best=round(fu, 4))
        for fn in fns:
            ab[f"similarity={fn}"] = ab_eval(model, fn=fn)
        for v in ("first1", "head4", "uniform4", "marker4"):
            ab[f"chunks={v} (at test time)"] = ab_eval(model, sub=lambda k, v=v: VIEWS[v](docs[k]))
        for nm, kw, Jo in [("objective=in-batch negatives only (no BM25 hard negatives)", dict(neg="none"), {}),
                           ("objective=random instead of BM25 hard negatives", dict(neg="random"), {}),
                           ("objective=pairwise softplus instead of InfoNCE", dict(loss_kind="pairwise"), {}),
                           ("pooling=attention", {}, dict(doc_pool="attn")), ("pooling=max", {}, dict(doc_pool="max")),
                           ("temperature=0.1", {}, dict(tau=0.1)), ("temperature=0.02", {}, dict(tau=0.02))]:
            J2 = dict(J, **Jo)
            m2 = PCRModel(model.enc, J2).to(device)
            m2.center.copy_(model.center)
            probe_train(m2, feats, St, hard, J2, device, log, **kw)
            ab[nm] = ab_eval(m2)
            log(f"[probe ablation] {nm}: {json.dumps(ab[nm])}")
            del m2
        R["probe_ablation"] = ab
        model.load_state_dict(base_state, strict=False)
        test_sc = cached_scores(model, vfeats, Ste, device, fns)
        test_infer_secs = enc_test_secs
    else:
        best, info = train_one(J, model, new, later, docs, St, hard, Sd, lex["dev"], Ed, Td, device, log, teacher=teacher)
        del teacher
        model.load_state_dict(best["state"], strict=False)
        model.to(device).eval()
        t0 = time.time()
        dev_sc, _, _ = eval_split(model, docs, Sd, lex["dev"], Ed, Td, device, "all", J, log, fns)
        t1 = time.time()
        test_sc, _, _ = eval_split(model, docs, Ste, lex["test"], Ete, Tte, device, "all", J, log, fns)
        test_infer_secs = time.time() - t1
        log(f"final encoding: dev {(t1 - t0) / 60:.1f} min, test {test_infer_secs / 60:.1f} min")
    R["train"] = info
    R["best_step"] = best.get("step")
    if J["save_model"]:
        torch.save(best["state"], out / "val_model.pt")

    # ---- dev: which similarity function, candidate set and fusion weight (honest estimate by cross-fitting)
    sim = {}
    for fn, M in dev_sc.items():
        sim[fn] = dict(model_only=round(prf(topk(final_scores(M, None, 1.0, Ed, Td)), Sd["gold"])["f1"], 4))
    R["similarity_dev"] = sim
    fn_mo = max(sim, key=lambda f: (sim[f]["model_only"], f == "mean"))
    best_r, table = choose_ranker(dev_sc, lex["dev"], Sd, Ed, Td)
    R["fusion"] = dict(best=best_r, crossfit_f1=crossfit_ranker(dev_sc, lex["dev"], Sd, Ed, Td),
                       by_w={str(r["w"]): round(r["f1"], 4) for r in table if r["fn"] == best_r["fn"] and r["cand"] == best_r["cand"]})
    Fm_dev = final_scores(dev_sc[fn_mo], None, 1.0, Ed, Td)
    R["dev_model_only"] = dict(fn=fn_mo, **{k: round(v, 4) for k, v in prf(topk(Fm_dev), Sd["gold"]).items() if k != "tp"})
    Cd = None if best_r["cand"] is None else lexical_cands(lex["dev"], Ed, best_r["cand"])
    Ff_dev = final_scores(dev_sc[best_r["fn"]], lex["dev"], best_r["w"], Ed, Td, Cd)
    top_dev = topk(Ff_dev)
    R["dev"] = {k: round(v, 4) for k, v in prf(top_dev, Sd["gold"]).items() if k != "tp"}
    R["dev"]["f1_crossfit"] = round(R["fusion"]["crossfit_f1"], 4)
    Fl_dev = final_scores(None, lex["dev"], 0.0, Ed, Td)
    R["dev_lexical_only"] = {k: round(v, 4) for k, v in prf(topk(Fl_dev), Sd["gold"]).items() if k != "tp"}
    R["analysis"] = analyze(top_dev, Fm_dev, Fl_dev, Ff_dev, Sd, words, Td)
    np.savez_compressed(out / "dev_scores.npz", qids=np.asarray(Sd["qids"]), pool=np.asarray(Sd["pool"]),
                        **{f"s_{fn}": M.astype(np.float16) for fn, M in dev_sc.items()})
    log(f"DEV F1@7 fused {R['dev']['f1']:.4f} (cross-fitted {R['dev']['f1_crossfit']:.4f}; {best_r['fn']}, w={best_r['w']}, "
        f"candidates={'all' if best_r['cand'] is None else 'lexical top-' + str(best_r['cand'])}) | model only {R['dev_model_only']['f1']:.4f} "
        f"({fn_mo}) | lexical only {R['dev_lexical_only']['f1']:.4f} | similarity {json.dumps(sim)} | "
        f"recall@100 model {R['analysis']['rank_model_only']['recall@100']} / lexical {R['analysis']['rank_lexical']['recall@100']}")

    # ---- test: the fused submission and a model-only submission
    Ct = None if best_r["cand"] is None else lexical_cands(lex["test"], Ete, best_r["cand"])
    Ff_te = final_scores(test_sc[best_r["fn"]], lex["test"], best_r["w"], Ete, Tte, Ct)
    pred = to_prediction(topk(Ff_te), Ste)
    check_prediction(pred, Ste)
    write_zip(pred, out / f"submission_{J['name']}.zip")
    pred_mo = to_prediction(topk(final_scores(test_sc[fn_mo], None, 1.0, Ete, Tte)), Ste)
    check_prediction(pred_mo, Ste)
    write_zip(pred_mo, out / f"submission_{J['name']}_model_only.zip")
    pred_lex = to_prediction(topk(final_scores(None, lex["test"], 0.0, Ete, Tte)), Ste)
    R["test_overlap_with_lexical_top7"] = round(float(np.mean([len(set(pred[q]) & set(pred_lex[q])) for q in pred])) / K, 3)
    np.savez_compressed(out / "test_scores.npz", qids=np.asarray(Ste["qids"]), pool=np.asarray(Ste["pool"]),
                        **{f"s_{fn}": M.astype(np.float16) for fn, M in test_sc.items()})
    R["test_infer_secs"] = test_infer_secs
    R["ms_per_doc"] = 1000 * test_infer_secs / max(1, len(test_keys))
    R["collapsed"] = bool(info.get("early_abort") or not all(np.isfinite(M).all() for M in test_sc.values())
                          or R["dev_model_only"]["f1"] < J.get("collapse_f1", 0.02))
    json.dump(R, open(out / "results.json", "w"), indent=1, default=str)
    log(f"DONE -> submission_{J['name']}.zip (+ _model_only) | test top-7 overlap with the lexical top-7: {R['test_overlap_with_lexical_top7']}"
        + (" | !!! COLLAPSED - do NOT upload" if R["collapsed"] else ""))


if __name__ == "__main__":
    run_job(sys.argv[1])


Writing /kaggle/working/p7/p7_pcr.py


In [5]:
# ============================================================ 2. DATA + LEXICAL RETRIEVAL + TEMPORAL CHECK + TOKENISATION CHECKS + CHUNK CACHE
sys.path.insert(0, str(WORK))
import importlib, p2_ner as P2, p5_lsi as P5, p7_pcr as P7; importlib.reload(P2); importlib.reload(P5); importlib.reload(P7)
import torch, numpy as np, pandas as pd
pd.set_option("display.width", 300); pd.set_option("display.max_columns", 40)

# (a) data: queries + candidate pools per split (test relevance never read)
data = P7.load_pcr(DATA)
TEXTS, SPLITS = data["texts"], data["splits"]
display(pd.DataFrame(data["stats"]).T)
ks = {sp: set(P7.split_keys(S)) for sp, S in SPLITS.items()}
print(f"unique document texts: {len(TEXTS)} | shared texts train/dev {len(ks['train'] & ks['dev'])}, train/test {len(ks['train'] & ks['test'])}, "
      f"dev/test {len(ks['dev'] & ks['test'])} (pools overlap; each split is ranked only against its own pool)")
for sp in ("train", "dev"):
    assert data["stats"][sp]["relevant_missing_from_pool"] == 0, "relevant ids outside the pool"

# (b) years: a PRIOR case cannot mention a year after the query's judgment date (decided below on dev)
YEARS = {k: P7.doc_year(t) for k, t in TEXTS.items()}
for sp in ("train", "dev", "test"):
    print(f"temporal check {sp}:", json.dumps(P7.temporal_report(SPLITS[sp], YEARS)))

# (c) lexical retrieval: TF-IDF / BM25 grid on dev, then the best setting on train (hard negatives) and test
t0 = time.time()
Sd = SPLITS["dev"]; Ed = P7.excl_mask(Sd)
dev_texts, dev_qrows = P7.lex_doc_texts(Sd, TEXTS)
XD = P7.vectorize_many([("dev", ng, st) for ng, st in P7.LEX_VEC], {"dev": dev_texts})
rows, MATS = [], {}
for (nm, ng, st), X in XD.items():
    for cfg in P7.LEX_SCORE:
        M = P7.lex_matrix(X, dev_qrows, len(Sd["pool"]), cfg)
        Fx = P7.final_scores(None, M, 0.0, Ed)
        m = P7.prf(P7.topk(Fx), Sd["gold"])
        name = P7.lex_name(ng, st, *cfg)
        MATS[name] = M
        rows.append(dict(setting=name, ngram=ng, stop=st, kind=cfg[0], k1=cfg[1], b=cfg[2], q=cfg[3], dev_f1=round(m["f1"], 4),
                         dev_p=round(m["p"], 4), dev_r=round(m["r"], 4), **P7.rank_metrics(Fx, Sd["gold"], ks=(7, 20, 50, 100, 200))))
LEXDF = pd.DataFrame(rows).sort_values("dev_f1", ascending=False).reset_index(drop=True)
LEXDF.to_csv(WORK / "lexical.csv", index=False)
print(f"lexical grid: {len(LEXDF)} settings on dev in {time.time() - t0:.0f}s"); display(LEXDF.head(12))
B = LEXDF.iloc[0]
LEX_BEST = dict(ngram=int(B["ngram"]), stop=bool(B["stop"]), cfg=(B["kind"], None if pd.isna(B["k1"]) else float(B["k1"]),
                None if pd.isna(B["b"]) else float(B["b"]), None if pd.isna(B["q"]) else B["q"]), name=B["setting"])
print("best lexical setting:", LEX_BEST["name"])
for kind in ("tfidf", "bm25"):
    r = LEXDF[LEXDF.kind == kind].iloc[0]; print(f"  best {kind}: {r['setting']} -> dev F1@7 {r['dev_f1']}")
M_DEV = MATS[LEX_BEST["name"]]; del MATS, XD

tem = {}
for slack in (None, 0, 1, 2):
    T = None if slack is None else P7.temporal_violations(Sd, YEARS, slack)
    tem[slack] = P7.prf(P7.topk(P7.final_scores(None, M_DEV, 0.0, Ed, T)), Sd["gold"])["f1"]
trep = P7.temporal_report(SPLITS["train"], YEARS)
ok_slacks = [s for s in (0, 1, 2) if trep[f"slack{s}"]["gold_pairs_violating"] <= 0.03]   # the rule must hold for >= 97% of train pairs
cand = max(ok_slacks, key=lambda s: tem[s]) if ok_slacks else None
TEMPORAL_SLACK = cand if cand is not None and tem[cand] >= tem[None] + 0.005 else None
print("dev F1@7 of the best lexical setting with the temporal constraint (slack in years):", {str(k): round(v, 4) for k, v in tem.items()},
      "-> temporal constraint", "OFF" if TEMPORAL_SLACK is None else f"ON (slack {TEMPORAL_SLACK})")

t0 = time.time()
tt = {sp: P7.lex_doc_texts(SPLITS[sp], TEXTS) for sp in ("train", "test")}
XT = P7.vectorize_many([(sp, LEX_BEST["ngram"], LEX_BEST["stop"]) for sp in tt], {sp: tt[sp][0] for sp in tt})
LEX = {"dev": M_DEV}
for (sp, _, _), X in XT.items():
    LEX[sp] = P7.lex_matrix(X, tt[sp][1], len(SPLITS[sp]["pool"]), LEX_BEST["cfg"])
del XT
St = SPLITS["train"]
Tt = None if TEMPORAL_SLACK is None else P7.temporal_violations(St, YEARS, TEMPORAL_SLACK)
mt = P7.prf(P7.topk(P7.final_scores(None, LEX["train"], 0.0, P7.excl_mask(St), Tt)), St["gold"])
print(f"train + test lexical scores in {time.time() - t0:.0f}s | the dev-chosen setting on TRAIN (not tuned there): F1@7 {mt['f1']:.4f} "
      f"(P {mt['p']:.4f} R {mt['r']:.4f})")
LEX_CACHE = WORK / "lex.pkl"
pickle.dump(dict(splits=SPLITS, scores=LEX, best=LEX_BEST, temporal_slack=TEMPORAL_SLACK, table=rows), open(LEX_CACHE, "wb"), protocol=4)
Ste = SPLITS["test"]
Tte = None if TEMPORAL_SLACK is None else P7.temporal_violations(Ste, YEARS, TEMPORAL_SLACK)
pl = P7.to_prediction(P7.topk(P7.final_scores(None, LEX["test"], 0.0, P7.excl_mask(Ste), Tte)), Ste); P7.check_prediction(pl, Ste)
SUB = Path("/kaggle/working/submissions" + ("_quick" if QUICK_TEST else "")); SUB.mkdir(exist_ok=True)
P7.write_zip(pl, SUB / "lexical_baseline.zip"); print("lexical baseline zip (reference only, not a Phase-1 experiment):", SUB / "lexical_baseline.zip")

# (d) token ids must be identical to the Phase-1 pipeline
tok = P2.load_bpe(TOK_PY, TOK_PATH, norm=NORM)
print("tokenizer kind:", tok.kind, "| normalisation:", NORM, "| vocab", tok.vocab_size); assert tok.vocab_size == 32000
NORM_FN, ENC_FN = P5.text_fns(tok, ARCH)
sample = [TEXTS[k][:6000] for k in list(TEXTS)[:8]]
if ARCH == "ret":                          # Phase-1 RetNet: ret_data.clean_text -> ORIGINAL a1_tokenizer.encode
    if r'?[^\s\w]+|\s+")' in open(TOK_PY).read():
        ref = lambda s: tok.tok.encode(P2.clean_text_ret(s)); print("checking against your original a1_tokenizer.encode")
    else:
        ref = lambda s: [i for pc in P2.PRE_RE_ORIG.findall(P2.clean_text_ret(s)) for i in tok.tok._encode_pretoken(pc)]
        print("attached a1_tokenizer.py has the underscore fix; Phase-1 RetNet used the ORIGINAL regex, which p2_ner applies")
else:
    if tok.kind == "new":
        assert tok.tok.id_to_token[:5] == ["<pad>", "<s>", "</s>", "<unk>", "Ā"], "this is not the new from-scratch tokenizer"
        if ARCH == "txl":
            bad0 = [s for s in sample if list(ENC_FN(NORM_FN(s))) != list(tok.tok.encode(P5.txl_normalize(s)))]
            assert not bad0, "differs from the Phase-1 TXL call tok.encode(normalize_text(text))"
            print("identical to the Phase-1 call tok.encode(normalize_text(text))")
    elif tok.kind == "old":
        assert tok.tok.itos[:4] == ["<pad>", "<unk>", "<s>", "</s>"], "this is not your old a1_tokenizer vocabulary"
        assert tok.tok.decode(ENC_FN(NORM_FN("Section_144 of the CrPC ____ blank"))) == "Section_144 of the CrPC ____ blank"
        if r"(?:[^\s\w]|_)+" in open(TOK_PY).read():
            bad0 = [s for s in sample if list(ENC_FN(NORM_FN(s))) != list(tok.tok.encode(P5.txl_normalize(s)))]
            assert not bad0, "differs from the patched a1_tokenizer.encode of Phase 1"
            print("old tokenizer with the Phase-1 underscore fix: identical to your patched a1_tokenizer.encode")
    ref = tok.reference_encode
bad = [s for s in sample if list(ENC_FN(NORM_FN(s))) != list(ref(s))]
print(f"token-id check on {len(sample)} document texts: {len(sample) - len(bad)} identical"); assert not bad

# (e) piece-wise encoding (whitespace-aligned pieces) == encoding of the whole document
longs = [t[:60000] for t in list(TEXTS.values())[:300] if len(t.split()) > 2000][:6] + [t[:20000] for t in list(TEXTS.values())[-4:]]
for s in longs:
    full = list(ENC_FN(NORM_FN(s)))
    for pw in (128, 7):
        assert list(P7.encode_pieces(s, NORM_FN, ENC_FN, pw)[0]) == full, "piece-wise encoding differs from the full encoding"
print(f"piece-wise check on {len(longs)} documents: identical to the full-document encoding")

# (f) tokenise + chunk every unique document once (all CPU cores) -> cache used by every job
CHUNK_LEN, MAX_STORE, SELECT = 350, 20, "marker"   # long documents: first + last + most-<CITATION> chunks + spread
ADD_BOS = False                       # RetNet pretraining saw [<s>] + text -> every chunk gets <s>
BOS_ID = 2 if tok.kind == "old" else 1
SPEC = dict(arch=ARCH, tok_py=TOK_PY, tok_path=TOK_PATH, norm=NORM, chunk_len=CHUNK_LEN, max_store=MAX_STORE, select=SELECT,
            bos=BOS_ID if ADD_BOS else None, piece_words=128, n_docs=len(TEXTS))
CACHE = WORK / "cache.pkl"
t0 = time.time()
if CACHE.exists() and pickle.load(open(CACHE, "rb"))["spec"] == SPEC:
    DOCS = pickle.load(open(CACHE, "rb"))["docs"]; print("re-using", CACHE)
else:
    DOCS = P7.prepare(TEXTS, list(TEXTS), SPEC, CACHE, extra=dict(years=YEARS))
print(f"tokenised + chunked in {time.time() - t0:.0f}s")
display(pd.DataFrame(P7.chunk_stats(DOCS, SPLITS, SPEC)).T)
del TEXTS, data

# (g) the encoder in CAUSAL mode must reproduce the Phase-1 forward pass exactly
J0 = dict(arch=ARCH, model_py=MODEL_PY, phase1_ckpt=PHASE1_CKPT)
base = P2.load_backbone(J0).eval()
print(f"Phase-1 model: {sum(p.numel() for p in base.parameters()) / 1e6:.2f}M params")
ids = torch.as_tensor(DOCS[P7.split_keys(Sd)[0]]["chunks"][0][:64].astype(np.int64))[None]
am = torch.ones_like(ids, dtype=torch.bool)
with torch.no_grad():
    if ARCH == "ret":
        tg = P2.RetTagger(base, 2, attn="causal", dropout=0.0).eval()
        S0, pd0 = base.init_state(1, ids.device)
        ref_h = base.hidden(ids, torch.zeros_like(ids), torch.arange(ids.size(1))[None], S0, pd0)[0]; mine = tg.hidden(ids, am)
    elif ARCH == "txl":
        tg = P2.TXLTagger(base, 2, attn="causal", dropout=0.0).eval()
        ref_h = base(ids)["logits"]; mine = torch.nn.functional.linear(tg.hidden(ids, am), base.emb.weight, base.out_bias)
    elif ARCH == "encdec":
        tg = P5.EncTaggerBOS(base, 2, attn="causal", dropout=0.0, use_decoder=True, bos_id=1).eval()
        mem0, msk0 = base.encode(ids)
        ref_h = base.decode_hidden(torch.cat([torch.ones_like(ids[:, :1]), ids], 1), mem0, msk0)[:, 1:]; mine = tg.hidden(ids, am)
        assert CHUNK_LEN + 1 <= base.c.dec_max_len, "decoder input [<s>] + chunk exceeds the decoder positions"
    elif ARCH == "dec":
        tg = P2.DecTagger(base, 2, attn="causal", dropout=0.0).eval()
        ref_h = base.hidden_states(ids); mine = tg.hidden(ids, am)
    elif ARCH == "encdec2":
        tg = P2.ED2Tagger(base, 2, attn="bidir", dropout=0.0).eval()
        ref_h = base.encode(ids, am); mine = tg.hidden(ids, am)
        assert CHUNK_LEN + 32 <= base.encoder.embedding.position_embedding.num_embeddings
diff = (ref_h - mine).abs().max().item(); print("encoder vs Phase-1 forward, max |diff| =", diff); assert diff < 1e-3
del base, tg


[data] train: {"queries": 827, "pool": 4320, "duplicate_query_ids": 0, "duplicate_candidate_ids": 0, "queries_in_own_pool": 827, "pool_texts_unique": 4318, "queries_with_pool_copies_of_their_text": 2, "relevant_mean": 6.42, "relevant_p50": 4, "relevant_max": 52, "queries_without_relevant": 10, "relevant_missing_from_pool": 0, "relevant_with_the_query_text": 0, "f1_at_7_upper_bound": 0.6386}
[data] dev: {"queries": 118, "pool": 1023, "duplicate_query_ids": 0, "duplicate_candidate_ids": 0, "queries_in_own_pool": 118, "pool_texts_unique": 1023, "queries_with_pool_copies_of_their_text": 0, "relevant_mean": 8.25, "relevant_p50": 5, "relevant_max": 92, "queries_without_relevant": 0, "relevant_missing_from_pool": 0, "relevant_with_the_query_text": 0, "f1_at_7_upper_bound": 0.6181}
[data] test: {"queries": 237, "pool": 1727, "duplicate_query_ids": 0, "duplicate_candidate_ids": 0, "queries_in_own_pool": 237, "pool_texts_unique": 1727, "queries_with_pool_copies_of_their_text": 0}


,queries,pool,duplicate_query_ids,duplicate_candidate_ids,queries_in_own_pool,pool_texts_unique,queries_with_pool_copies_of_their_text,relevant_mean,relevant_p50,relevant_max,queries_without_relevant,relevant_missing_from_pool,relevant_with_the_query_text,f1_at_7_upper_bound
train,827.0,4320.0,0.0,0.0,827.0,4318.0,2.0,6.42,4.0,52.0,10.0,0.0,0.0,0.6386
dev,118.0,1023.0,0.0,0.0,118.0,1023.0,0.0,8.25,5.0,92.0,0.0,0.0,0.0,0.6181
test,237.0,1727.0,0.0,0.0,237.0,1727.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


unique document texts: 5819 | shared texts train/dev 437, train/test 742, dev/test 222 (pools overlap; each split is ranked only against its own pool)
temporal check train: {"queries_with_date_header": 0.427, "candidates_with_a_year": 0.957, "slack0": {"gold_pairs_violating": 0.0893, "all_pairs_violating": 0.4212}, "slack1": {"gold_pairs_violating": 0.0816, "all_pairs_violating": 0.4115}, "slack2": {"gold_pairs_violating": 0.0759, "all_pairs_violating": 0.402}}
temporal check dev: {"queries_with_date_header": 0.475, "candidates_with_a_year": 0.978, "slack0": {"gold_pairs_violating": 0.0781, "all_pairs_violating": 0.4285}, "slack1": {"gold_pairs_violating": 0.075, "all_pairs_violating": 0.4184}, "slack2": {"gold_pairs_violating": 0.0689, "all_pairs_violating": 0.4065}}
temporal check test: {"queries_with_date_header": 0.426, "candidates_with_a_year": 0.962}
lexical grid: 52 settings on dev in 33s


,setting,ngram,stop,kind,k1,b,q,dev_f1,dev_p,dev_r,recall@7,recall@20,recall@50,recall@100,recall@200,mrr,median_gold_rank
0,tfidf uni+bi -stop,2,True,tfidf,NaN,NaN,None,0.4347,0.4734,0.4018,0.4018,0.5694,0.6783,0.7503,0.8171,0.8954,13
1,tfidf uni+bi,2,False,tfidf,NaN,NaN,None,0.3991,0.4346,0.3690,0.3690,0.5283,0.6290,0.7133,0.7831,0.8476,16
2,bm25 uni+bi -stop k1=2.5 b=0.9 q=binary,2,True,bm25,2.5,0.9,binary,0.3891,0.4237,0.3597,0.3597,0.5077,0.6053,0.6680,0.7585,0.8140,18
3,bm25 uni+bi -stop k1=2.5 b=0.9 q=log,2,True,bm25,2.5,0.9,log,0.3780,0.4116,0.3494,0.3494,0.5108,0.6187,0.6845,0.7739,0.8160,18
4,bm25 uni+bi -stop k1=1.5 b=0.9 q=log,2,True,bm25,1.5,0.9,log,0.3635,0.3959,0.3361,0.3361,0.4738,0.5848,0.6598,0.7513,0.7998,26
5,bm25 uni+bi -stop k1=1.5 b=0.9 q=binary,2,True,bm25,1.5,0.9,binary,0.3624,0.3947,0.3350,0.3350,0.4625,0.5632,0.6444,0.7369,0.7846,29
6,tfidf uni,1,False,tfidf,NaN,NaN,None,0.3558,0.3874,0.3289,0.3289,0.4656,0.5817,0.6608,0.7595,0.8373,26
7,tfidf uni -stop,1,True,tfidf,NaN,NaN,None,0.3558,0.3874,0.3289,0.3289,0.4738,0.5889,0.6660,0.7718,0.8315,25
8,bm25 uni+bi -stop k1=0.9 b=0.9 q=log,2,True,bm25,0.9,0.9,log,0.3369,0.3668,0.3114,0.3114,0.4306,0.5375,0.6331,0.7287,0.7535,37
9,bm25 uni+bi k1=2.5 b=0.9 q=log,2,False,bm25,2.5,0.9,log,0.3324,0.3620,0.3073,0.3073,0.4317,0.5457,0.6228,0.7102,0.7484,36


best lexical setting: tfidf uni+bi -stop
  best tfidf: tfidf uni+bi -stop -> dev F1@7 0.4347
  best bm25: bm25 uni+bi -stop k1=2.5 b=0.9 q=binary -> dev F1@7 0.3891
dev F1@7 of the best lexical setting with the temporal constraint (slack in years): {'None': 0.4347, '0': 0.4558, '1': 0.4569, '2': 0.4536} -> temporal constraint OFF
train + test lexical scores in 37s | the dev-chosen setting on TRAIN (not tuned there): F1@7 0.3430 (P 0.3287 R 0.3585)
lexical baseline zip (reference only, not a Phase-1 experiment): /kaggle/working/submissions/lexical_baseline.zip
tokenizer kind: new | normalisation: nfkc | vocab 32000
token-id check on 8 document texts: 8 identical
piece-wise check on 10 documents: identical to the full-document encoding
[prepare] 5819 unique documents tokenised + chunked in 27s
tokenised + chunked in 27s


,docs,words_p50,words_p90,words_max,tok_per_word,chunks_total_mean,chunks_kept_mean,docs_fully_covered_pct,tokens_kept_pct,citations_per_doc
train,4318.0,3635.0,12424.0,410597.0,1.260,23.17,13.07,68.4,55.4,7.7
dev,1023.0,4036.0,16598.0,410597.0,1.256,31.39,13.64,64.4,42.6,10.1
test,1727.0,3947.0,13490.0,410597.0,1.261,25.65,13.57,65.1,52.0,9.3


Phase-1 model: 90.30M params
encoder vs Phase-1 forward, max |diff| = 0.0


In [6]:
# ============================================================ 3. EXPERIMENTS (2 GPUs -> 2 queues)
COMMON = dict(
    arch=ARCH, cache=str(CACHE), lex_cache=str(LEX_CACHE), model_py=MODEL_PY, phase1_ckpt=PHASE1_CKPT,
    attn="causal",             # 'bidir' (tokens see both sides) | 'causal' (= Phase-1 computation; enc-dec decoder)
    emb_dim=256, doc_pool="mean", dropout=0.1, tau=0.05,   # document embedding, pooling over chunks, InfoNCE temperature
    q_per_step=16, hard_per_q=2, hard_depth=30,   # per step: 16 queries + 16 positives + 32 BM25 hard negatives (GradCache)
    train_chunks=4,           # random stored chunks per document while training (evaluation uses all stored chunks)
    tok_budget=8400,           # padded tokens per GradCache micro-batch
    infer_tok_budget=32768, eval_view="eval6",    # evals during training look at <= 6 chunks per document (faster)
    score_fns=["mean", "maxsim", "maxsim_sym", "mean+maxsim"],
    temporal_slack=TEMPORAL_SLACK,
    encdec_use_decoder=True,                         # BART-style: decoder states (encoder alone was weak in P2/P3)
    dec_bos=True, bos_id=1,                          # decoder input = [<s>] + chunk, as in Phase-1 pretraining
    lowrank_targets=["q", "k", "v", "o", "fc1", "fc2"],   # LoRMA-all / AdaLoRA targets (self- and cross-attention, FFN)
    adapter_targets=["o", "fc2"],                    # Houlsby adapters after the attention outputs and the FFN output
    kd_student_layers=[0, 2, 5],                     # KD student: 3 of 6 encoder + 3 of 6 decoder layers
    ia3_out=["k", "v"], ia3_in=["fc2"],   # IA3: rescale keys + values, and the FFN inner activation
    head_lr=1e-3, peft_lr=1e-3, lr=2e-5, weight_decay=0.01, warmup=0.05,
    epochs=40,                # passes over the training queries: an upper bound - the time budget ends training first
    time_budget_h=None,       # set per job below from GROUP
    calib_steps=8, n_evals=4, log_every=10,
    seed=781, rank=16, alpha=32, lowrank_dropout=0.05, qlora_alpha=32, nf4_block=64, n_prompt=20,
    partial_top_layers=4, gradual_schedule=[[0.0, 0], [0.15, 2], [0.35, 4], [0.55, 6]],
    kd_temperature=2.0, kd_alpha=0.5,
    probe_epochs=30, probe_q_per_step=64, probe_hard_per_q=4, probe_chunks=8,
    grad_ckpt=False, ckpt_micro=4,       # gradient checkpointing switches itself on after a CUDA OOM
    early_check=0.2, early_min_f1=0.015, collapse_f1=0.02, save_model=False, quick=QUICK_TEST,
)
# LoRMA multiplies a projection's OUTPUT: y = (I + s*B*A) W x. On residual-stream projections this overflowed fp16 in
# Phase 4, so LoRMA adapts the attention-side projections with s = alpha/r = 1 (16/16).
#               name           method     overrides
TEACHERS = [str(OUT / "lorma"), str(OUT / "full")]
JOBS = {
  "lorma":    ("lorma",   dict(lowrank_targets=["q", "k", "v"], alpha=16, peft_lr=3e-4, save_model=True)),   # compulsory + teacher
  "full":     ("full",    dict(lr=2e-5, save_model=True)),                                                     # reference + teacher
  "kd":       ("kd",      dict(kd_teacher_dir=TEACHERS, lr=5e-5)),                                             # compulsory
  "kd_small": ("kd",      dict(kd_teacher_dir=TEACHERS, lr=5e-5, kd_student_layers=[0, 5])),
  "partial":  ("partial", dict(lr=5e-5)),
  "qlora":    ("qlora",   dict(peft_lr=5e-4)),
  "ia3":      ("ia3",     dict(peft_lr=3e-3)),
  "prompt":   ("prompt",  dict(peft_lr=3e-3, n_prompt=20)),
  "prefix":   ("prefix",  dict(peft_lr=1e-3, n_prompt=16)),
  "frozen":   ("frozen",  dict(hours_scale=0.5)),                  # probe + ablations on cached features
  "gradual":  ("gradual", dict(lr=5e-5)),                          # optional extra (not in a group)
}
GROUPS = {   # queues (one per GPU; the teachers lorma + full run first) , hours per job
  "ALL": ([["lorma", "kd", "partial", "ia3", "frozen"], ["full", "kd_small", "qlora", "prompt", "prefix"]], 1.7),
  "A":   ([["lorma", "kd", "ia3"], ["full", "kd_small", "frozen"]], 3.1),          # account 1
  "B":   ([["partial", "prompt"], ["qlora", "prefix"]], 4.7),                       # account 2 (same time)
}
QUEUES, JOB_HOURS = GROUPS[GROUP]
RUN_ONLY = None           # e.g. ["prompt", "prefix"] to (re)run only those jobs
if RUN_ONLY:
    QUEUES = [[n for n in q if n in RUN_ONLY] for q in QUEUES]
def teachers_of(n):
    t = JOBS[n][1].get("kd_teacher_dir")
    return [Path(x).name for x in (t if isinstance(t, list) else [t] if t else [])]

n_gpu = torch.cuda.device_count(); print("GPUs:", n_gpu)
if n_gpu < 2:                              # one queue: every teacher before every KD job
    QUEUES = [[n for q in QUEUES for n in q if not teachers_of(n)] + [n for q in QUEUES for n in q if teachers_of(n)]]
print("group", GROUP, "| queues:", QUEUES, "| hours per job:", JOB_HOURS, "| temporal slack:", TEMPORAL_SLACK)
SCHEDULED, DONE = set(sum(QUEUES, [])), set()

def write_job(name, hours):
    meth, over = JOBS[name]
    over = dict(over); scale = over.pop("hours_scale", 1.0)
    J = dict(COMMON, name=name, method=meth, out_dir=str(OUT / name), **over)
    J["time_budget_h"] = hours * scale
    if QUICK_TEST: J.update(epochs=3, n_evals=2, calib_steps=10 ** 9, log_every=2, time_budget_h=0.5, early_check=0, q_per_step=6,
                            probe_epochs=2, probe_q_per_step=16, tok_budget=min(J["tok_budget"], 4096))
    Path(J["out_dir"]).mkdir(parents=True, exist_ok=True)
    p = Path(J["out_dir"]) / "job.json"; json.dump(J, open(p, "w"), indent=1)
    return p

def run_queue(names, gpu):
    for n in names:
        if (OUT / n / "results.json").exists():
            print(f"[skip] {n} already done"); DONE.add(n); continue
        wait = [t for t in teachers_of(n) if t in SCHEDULED]             # KD waits until its teacher jobs are finished
        if wait and not all(t in DONE for t in wait):
            print(f"[gpu{gpu}] {n} waits for its teacher job(s) {wait}")
            while not all(t in DONE for t in wait): time.sleep(30)
        try:
            run_one(n, gpu)
        finally:
            DONE.add(n)

def run_one(n, gpu):
    left = SESSION_H - (time.time() - T_START) / 3600 - 0.5          # keep 30 min for the summary + saving the output
    scale = JOBS[n][1].get("hours_scale", 1.0)
    hours = min(JOB_HOURS, (left - 0.2) / scale)                    # ~10 min per job for loading + the final dev/test encoding
    if not QUICK_TEST and hours * scale < 0.4:
        print(f"[gpu{gpu}] !!! not enough session time left for {n} ({left:.2f} h) - skipped; rerun it with RUN_ONLY"); return
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu) if n_gpu else "", PYTHONUNBUFFERED="1")
    p = subprocess.Popen([sys.executable, str(WORK / "p7_pcr.py"), str(write_job(n, hours))], env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "Warning" not in line and "warn(" not in line: print(f"[gpu{gpu}] {line}", end="")
    p.wait()
    if p.returncode != 0:
        print(f"[gpu{gpu}] !!! job {n} FAILED (see {OUT / n / 'log.txt'} and the lines above)")

T0 = time.time()
threads = [threading.Thread(target=run_queue, args=(q, g)) for g, q in enumerate(QUEUES)]
for t in threads: t.start()
for t in threads: t.join()
print(f"all jobs finished in {(time.time() - T0) / 60:.1f} min")


GPUs: 2
group ALL | queues: [['lorma', 'kd', 'partial', 'ia3', 'frozen'], ['full', 'kd_small', 'qlora', 'prompt', 'prefix']] | hours per job: 1.7 | temporal slack: None
[gpu0] [22:37:19][lorma] arch=encdec method=lorma attn=causal device=cuda | train 827 queries / 4320 pool, dev 118 / 1023, test 237 / 1727 | temporal slack None | budget 1.70 h
[gpu1] [22:37:19][full] arch=encdec method=full attn=causal device=cuda | train 827 queries / 4320 pool, dev 118 / 1023, test 237 / 1727 | temporal slack None | budget 1.70 h
[gpu1] [22:37:21][full] params total=90.46M (encoder 90.30M) trainable=90.463M | PEFT tensors 0
[gpu0] [22:37:21][lorma] params total=91.57M (encoder 91.40M) trainable=1.270M | PEFT tensors 108
[gpu1] [22:37:35][full] 817 training queries with >= 1 relevant case | 16 queries per step (+1 positive, +2 hard negatives each) -> 52 steps per epoch | 40 epochs = 2080 steps planned
[gpu0] [22:37:35][lorma] 817 training queries with >= 1 relevant case | 16 queries per step (+1 posit

In [7]:
# ============================================================ 4. SUMMARY + SUBMISSIONS (+ an ensemble) + REPORT TABLES
import zipfile
LX = pickle.load(open(LEX_CACHE, "rb"))
SPL, LXS = (P7.quick_splits(LX["splits"], LX["scores"]) if QUICK_TEST else (LX["splits"], LX["scores"]))
Sd, Ste = SPL["dev"], SPL["test"]
Ed, Ete = P7.excl_mask(Sd), P7.excl_mask(Ste)
Td = None if TEMPORAL_SLACK is None else P7.temporal_violations(Sd, YEARS, TEMPORAL_SLACK)
Tte = None if TEMPORAL_SLACK is None else P7.temporal_violations(Ste, YEARS, TEMPORAL_SLACK)
lex_dev = P7.prf(P7.topk(P7.final_scores(None, LXS["dev"], 0.0, Ed, Td)), Sd["gold"])
rows, res = [dict(exp="(lexical only)", method=LX["best"]["name"], dev_F1=round(lex_dev["f1"], 4), dev_P=round(lex_dev["p"], 4),
                  dev_R=round(lex_dev["r"], 4), UPLOAD="reference only")], {}
for n in JOBS:
    rp = OUT / n / "results.json"
    if not rp.exists(): continue
    r = json.load(open(rp)); tr = r["train"]; res[n] = r
    for z in (OUT / n / f"submission_{n}.zip", OUT / n / f"submission_{n}_model_only.zip"):
        with zipfile.ZipFile(z) as zz:
            assert zz.namelist() == ["prediction.json"]
            P7.check_prediction(json.loads(zz.read("prediction.json")), Ste)
        shutil.copy(z, SUB / z.name)
    fb = r["fusion"]["best"]; an = r["analysis"]
    rows.append(dict(exp=n, method=r["method"], dev_F1=r["dev"]["f1"], dev_F1_crossfit=r["dev"]["f1_crossfit"], dev_P=r["dev"]["p"], dev_R=r["dev"]["r"],
                     model_only_F1=r["dev_model_only"]["f1"], sim=fb["fn"], w=fb["w"], cands="all" if fb["cand"] is None else f"lex top-{fb['cand']}",
                     model_recall100=an["rank_model_only"]["recall@100"], model_mrr=an["rank_model_only"]["mrr"],
                     overlap_lex_top7=an.get("top7_overlap_model_vs_lexical"), epochs=round(tr.get("epochs") or 0, 2), best_step=r.get("best_step"),
                     train_h=round(tr["train_secs"] / 3600, 2), params_M=round(r["params_total"] / 1e6, 2), trainable_M=round(r["params_trainable"] / 1e6, 3),
                     peak_GB=round(tr.get("peak_mem_gb") or 0, 2), ms_per_doc=round(r["ms_per_doc"], 1),
                     UPLOAD="NO - collapsed" if r["collapsed"] else ("optional (probe)" if n == "frozen" else "yes")))
df = pd.DataFrame(rows)
if len(df) > 1: df = pd.concat([df.iloc[:1], df.iloc[1:].sort_values("dev_F1_crossfit", ascending=False)])
display(df)
df.to_csv(WORK / ("summary_quick.csv" if QUICK_TEST else "summary.csv"), index=False)
if "frozen" in res:
    print("PHASE-1 TRANSFER WITHOUT TRAINING (dev F1@7; frozen mean-pooled hidden states):"); display(pd.DataFrame(res["frozen"]["zero_shot"]).T)
    print("ABLATIONS on the frozen probe (one change at a time; default = mean pooling, InfoNCE + BM25 hard negatives, tau 0.05, all chunks):")
    display(pd.DataFrame(res["frozen"]["probe_ablation"]).T)
ok = [n for n in df["exp"][1:] if n in res and res[n]["collapsed"] is False]
if ok:
    print("SIMILARITY FUNCTIONS (dev F1@7, model only):"); display(pd.DataFrame({n: {k: v["model_only"] for k, v in res[n]["similarity_dev"].items()} for n in ok}).T)
    b = res[ok[0]]; an = b["analysis"]
    print(f"\nBEST RUN {ok[0]}: fusion weight curve (dev F1@7 by w):", b["fusion"]["by_w"])
    print("by number of relevant cases:", an["by_n_relevant"]); print("by query length:", an["by_query_words"])
    print("ranking (model only / lexical / final):", an["rank_model_only"], an["rank_lexical"], an["rank_final"], sep="\n  ")
    print(f"hits only the model finds: {an.get('hits_only_model')} | only lexical: {an.get('hits_only_lexical')} | queries with 0 hits: "
          f"{an['queries_with_zero_hits']} | words p50 returned-relevant {an['words_of_returned_relevant_p50']} / returned-irrelevant "
          f"{an['words_of_returned_irrelevant_p50']} / all relevant {an['words_of_all_relevant_p50']} / pool {an['words_of_pool_p50']}")
    print("hardest queries:", an["hardest_queries"])

# extra submission: average of the z-normalised model scores of the best 3 healthy runs, fused with lexical (an ensemble - NOT a new technique)
ENS = [n for n in ok if n != "frozen"][:3]
if len(ENS) >= 2:
    def zs(n, split, S, E):
        z = np.load(OUT / n / f"{split}_scores.npz"); fn = res[n]["dev_model_only"]["fn"]
        assert list(z["qids"]) == S["qids"] and list(z["pool"]) == S["pool"]
        return P7.zrows(z[f"s_{fn}"].astype(np.float32), E)
    Md = np.mean([zs(n, "dev", Sd, Ed) for n in ENS], 0); Mt = np.mean([zs(n, "test", Ste, Ete) for n in ENS], 0)
    best, _ = P7.choose_ranker({"ens": Md}, LXS["dev"], Sd, Ed, Td, cand_depths=(None,))
    cf = P7.crossfit_ranker({"ens": Md}, LXS["dev"], Sd, Ed, Td, cand_depths=(None,))
    pe = P7.to_prediction(P7.topk(P7.final_scores(Mt, LXS["test"], best["w"], Ete, Tte)), Ste); P7.check_prediction(pe, Ste)
    P7.write_zip(pe, SUB / ("submission_ensemble_" + "_".join(ENS) + ".zip"))
    print(f"\nensemble of {ENS}: dev F1@7 {best['f1']:.4f} (cross-fitted {cf:.4f}, w={best['w']})")
print("submission zips:", sorted(p.name for p in SUB.glob("*.zip")), "->", SUB)
if QUICK_TEST: print("\nQUICK TEST ONLY - do NOT submit these (they cover only a few test queries). Set QUICK_TEST=False and Save & Run All (Commit).")


,exp,method,dev_F1,dev_P,dev_R,UPLOAD,dev_F1_crossfit,model_only_F1,sim,w,cands,model_recall100,model_mrr,overlap_lex_top7,epochs,best_step,train_h,params_M,trainable_M,peak_GB,ms_per_doc
0,(lexical only),tfidf uni+bi -stop,0.4347,0.4734,0.4018,reference only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,partial,partial,0.4436,0.4831,0.4101,yes,0.4391,0.3713,maxsim_sym,0.5,all,0.7636,0.7930,0.515,30.04,1171.0,1.61,90.46,26.428,2.14,63.2
6,qlora,qlora,0.4436,0.4831,0.4101,yes,0.4369,0.3713,maxsim_sym,0.5,all,0.7646,0.8140,0.512,15.38,800.0,1.59,93.17,2.867,4.68,89.0
8,prompt,prompt,0.4380,0.4770,0.4049,yes,0.4369,0.3624,mean+maxsim,0.5,all,0.7431,0.7586,0.492,24.65,1282.0,1.60,90.48,0.177,2.89,64.9
7,ia3,ia3,0.4402,0.4794,0.4070,yes,0.4369,0.3702,mean,0.5,all,0.7492,0.7928,0.513,21.27,1106.0,1.60,90.52,0.218,3.76,73.4
3,kd,kd,0.4380,0.4770,0.4049,yes,0.4358,0.3535,maxsim_sym,0.5,all,0.7688,0.7410,0.483,31.56,1230.0,1.75,55.99,55.995,3.13,32.2
2,full,full,0.4369,0.4758,0.4039,yes,0.4358,0.3691,mean,0.5,all,0.7657,0.7881,0.510,22.77,1184.0,1.73,90.46,90.463,5.13,62.9
9,prefix,prefix,0.4391,0.4782,0.4060,yes,0.4358,0.3502,mean+maxsim,0.5,all,0.7503,0.7477,0.481,24.35,1266.0,1.60,90.53,0.236,2.88,64.4
4,kd_small,kd,0.4391,0.4782,0.4060,yes,0.4358,0.3558,maxsim_sym,0.5,all,0.7698,0.7574,0.498,36.10,1407.0,1.56,44.51,44.506,2.34,21.7
1,lorma,lorma,0.4358,0.4746,0.4029,yes,0.4314,0.3524,maxsim_sym,0.4,all,0.7369,0.7743,0.504,20.46,213.0,1.68,91.57,1.270,3.69,82.0


PHASE-1 TRANSFER WITHOUT TRAINING (dev F1@7; frozen mean-pooled hidden states):


,model_only,fused_best
"zero-shot mean, raw cosine",0.2279,0.4380
"zero-shot mean, centred",0.2357,0.4314
"zero-shot first chunk, centred",0.1145,0.4325
"zero-shot MaxSim, centred",0.2513,0.4336


ABLATIONS on the frozen probe (one change at a time; default = mean pooling, InfoNCE + BM25 hard negatives, tau 0.05, all chunks):


,model_only,fused_best
similarity=mean,0.3213,0.4325
similarity=maxsim,0.3191,0.4325
similarity=maxsim_sym,0.3624,0.4347
similarity=mean+maxsim,0.3346,0.4336
chunks=first1 (at test time),0.1267,0.4347
chunks=head4 (at test time),0.2279,0.4314
chunks=uniform4 (at test time),0.2123,0.4380
chunks=marker4 (at test time),0.2212,0.4347
objective=in-batch negatives only (no BM25 hard negatives),0.3235,0.4325
objective=random instead of BM25 hard negatives,0.3268,0.4325


SIMILARITY FUNCTIONS (dev F1@7, model only):


,mean,maxsim,maxsim_sym,mean+maxsim
partial,0.3424,0.3480,0.3713,0.3569
qlora,0.3424,0.3469,0.3713,0.3558
prompt,0.3168,0.3291,0.3624,0.3346
ia3,0.3380,0.3435,0.3702,0.3535
kd,0.3280,0.3424,0.3535,0.3346
full,0.3446,0.3502,0.3691,0.3635
prefix,0.3180,0.3235,0.3502,0.3291
kd_small,0.3335,0.3235,0.3558,0.3413
lorma,0.3157,0.3357,0.3524,0.3346
frozen,0.3213,0.3191,0.3624,0.3346



BEST RUN partial: fusion weight curve (dev F1@7 by w): {'0.1': 0.4336, '0.2': 0.4358, '0.3': 0.4414, '0.4': 0.4391, '0.5': 0.4436, '0.6': 0.4402, '0.7': 0.438, '0.8': 0.4358, '0.9': 0.4213, '1.0': 0.3713}
by number of relevant cases: {'1': {'queries': 15, 'micro_f1': 0.2}, '2-4': {'queries': 34, 'micro_f1': 0.4048}, '5-9': {'queries': 39, 'micro_f1': 0.5905}, '>=10': {'queries': 30, 'micro_f1': 0.401}}
by query length: {'<2k': {'queries': 18, 'micro_f1': 0.2822}, '2k-5k': {'queries': 47, 'micro_f1': 0.541}, '5k-10k': {'queries': 33, 'micro_f1': 0.5209}, '>=10k': {'queries': 20, 'micro_f1': 0.3128}}
ranking (model only / lexical / final):
  {'recall@7': 0.3433, 'recall@20': 0.5242, 'recall@50': 0.667, 'recall@100': 0.7636, 'mrr': 0.793, 'median_gold_rank': 17}
  {'recall@7': 0.4018, 'recall@20': 0.5694, 'recall@50': 0.6783, 'recall@100': 0.7503, 'mrr': 0.8954, 'median_gold_rank': 13}
  {'recall@7': 0.4101, 'recall@20': 0.5879, 'recall@50': 0.7112, 'recall@100': 0.776, 'mrr': 0.9032, 'm